# Cambrian-P, ReVSI and VSI-Bench: a validity audit (Colab)

**Question.** Do annotation errors, missing evidence in the sampled frames, or non-visual answer patterns explain some of Cambrian-P-7B's apparent successes and failures on VSI-Bench, and what changes under ReVSI? This notebook reports evidence and uncertainty; it does not presuppose the answer.

**Three hypotheses kept apart:** (1) incorrect or ambiguous reference answers, (2) insufficient model-visible evidence, (3) non-visual shortcuts. A score difference alone does not establish contamination, memorization or Cambrian-P-specific overfitting. The ReVSI paper's Cambrian experiments concern **Cambrian-S**, not Cambrian-P; Cambrian-P is tested independently here. ReVSI is an alternative curated reference, not unquestionable truth: disagreement stays disagreement until a final human review.

**What runs.** A frozen 10% sample of ReVSI's unique (dataset, scene) videos in the `32_frame` config (all released questions of those videos), evaluated with Cambrian-P on the **released 32-frame inputs, every frame once** (a controlled frame-budget experiment, not a reproduction of Cambrian-P's native 128-frame results), scored with the **pinned official ReVSI scorer**; a paired **black-frame control**; a **scene-disjoint frequency baseline**; scene-cluster bootstrap uncertainty; the earlier VSI-Bench run imported as a separate, failure-selected historical cohort with a scene/QA overlap table; and a **human review** (in-notebook form or external CSV) through one validator.

**Run modes** (cell A1): `full` = all questions of the frozen sample (not the whole benchmark); `smoke` = at most two videos and a few questions, in a separate run folder; `review_only` = reload saved predictions and exports, no installs and no model.

**Review workload:** you must final-review only the first `REVIEW_VIDEOS` (default 10) of the exported candidate videos, one question each, in the fixed queue order; everything else in the queue is optional.

**Order:** A (setup) → B (data, sample, overlap) → C (media, frames) → D (inference; only missing jobs) → E (scores) → F (controls) → G1 (queues) → H1 (export candidates) → G2/G3 (review: form or CSV) → H2 (final exports and report) → V (self-tests). Everything is saved to Drive as it is produced, and all printed output goes to `logs/notebook-*.log` in the run folder.

## A. Mount Drive and preflight

In [ ]:
# @title A1. Settings (the one authoritative settings cell)
RUN_MODE = "full"                  # full / smoke / review_only
SEED = 42
VIDEO_FRACTION = 0.10
REVSI_CONFIG = "32_frame"
REVSI_SPLIT = "test"
MODEL_REPO = "nyu-visionx/Cambrian-P-7B"
EXISTING_VSI_RUN = "/content/drive/MyDrive/Cambrian-P/vsibench_runs/vsibench-video10pct-seed42-cambrianp7b-v1"
RUN_ID = "revsi32-video10pct-seed42-cambrianp7b-v1"
OUTPUT_ROOT = "/content/drive/MyDrive/Cambrian-P/revsi_runs/" + RUN_ID
CACHE_ROOT = "/content/drive/MyDrive/Cambrian-P/cache"
LOCAL_ROOT = "/content/revsi_work"
MIN_REVIEW_VIDEOS = 10
TARGET_REVIEW_VIDEOS = 20
REVIEW_VIDEOS = 10                 # candidate videos you must review: the first 10 of the exported 20, in the fixed
                                   # queue order (decided before any review). Changing it never re-runs inference.
AUDIT_REQUIRED = False             # the seeded audit questions are optional unless True
RUN_VISUAL_CONTROLS = True
RUN_ALL_FRAME_SECONDARY = False    # not implemented as a separate protocol here; stays False (reported)
ALLOW_SAMPLE_EXPANSION = False     # the sample stays frozen; expansion would need a separately labelled cohort

# Further settings (defaults are fine)
DATASET_REPO = "3dlg-hcvc/ReVSI"
DATASET_REVISION = ""              # resolved once and saved in pins.json; reused on resume
MODEL_REVISION = ""
CODE_COMMIT = ""                   # Cambrian-P code
SCORER_REPO = "EvolvingLMMs-Lab/lmms-eval"
SCORER_COMMIT = ""                 # the commit of lmms_eval/tasks/revsi (official ReVSI scorer); resolved once
VSI_DATASET_REPO, VSI_DATASET_REVISION = "nyu-visionx/VSI-Bench", "3a6a1cf9ac54c403509b15260897cc6ddc41b408"
REPO_URL = "https://github.com/cambrian-mllm/cambrian-p.git"
NUM_FRAMES = int(REVSI_CONFIG.split("_")[0])      # 32: the released 32-frame video, every frame once
FRAME_PROTOCOL = f"ReVSI released {REVSI_CONFIG} video (official nested indices); all {NUM_FRAMES} frames, once, in order"
PROMPT_POLICY = "official revsi_doc_to_text with the task YAML's default kwargs"
PREPROCESSING = ("decord decode of the fixed-budget video (must hold exactly NUM_FRAMES frames); expand2square with the "
                 "image-processor mean colour; SigLIP image processor; fp16; camera tokens from the model's own head")
AUDIT_QA = 30                      # seeded audit queue (independent of correctness; optional, see AUDIT_REQUIRED)
CONTROL_QA = 100                   # predeclared black-frame control questions
MAX_RETRIES = 2
MEDIA_BATCH_GB = 1
VERIFY_FRAMES = True               # check each input against the all-frame video at the official indices
VERIFY_MAX_GB = 1.0                # cap on all-frame bytes fetched for that check
ARCHIVE_DOWNLOAD_MAX_GB = 0.0      # 0 = never download the whole 4.9 GB video.zip (only if range reads are refused)
CACHE_ENV, CACHE_MODEL, CACHE_VIDEOS = True, True, True
SMOKE_VIDEOS, SMOKE_QA = 2, 6

import os, sys, json, shutil, subprocess, time as _t
assert RUN_MODE in ("full", "smoke", "review_only"), "RUN_MODE must be full, smoke or review_only"
assert not ALLOW_SAMPLE_EXPANSION, "sample expansion needs a separately labelled supplemental cohort (not implemented)"
RUN_KIND = "smoke" if RUN_MODE == "smoke" else "full"
if RUN_MODE == "smoke":                             # separately fingerprinted run folder
    RUN_ID, OUTPUT_ROOT = RUN_ID + "-smoke", OUTPUT_ROOT + "-smoke"
from google.colab import drive
drive.mount("/content/drive")
for d in (OUTPUT_ROOT, LOCAL_ROOT, CACHE_ROOT, "/content/revsi", os.path.join(OUTPUT_ROOT, "logs")):
    os.makedirs(d, exist_ok=True)

class _DriveTee:                                    # every printed line also goes to a log on Drive
    def __init__(self, stream, path):
        self._stream, self._f, self._revsi_tee = stream, open(path, "a", buffering=1), True
    def write(self, s):
        self._stream.write(s)
        try:
            self._f.write(s)
        except Exception:
            pass
        return len(s)
    def flush(self):
        self._stream.flush()
    def __getattr__(self, name):
        return getattr(self._stream, name)
NOTEBOOK_LOG = os.path.join(OUTPUT_ROOT, "logs", f"notebook-{_t.strftime('%Y%m%d-%H%M%S')}.log")
sys.stdout = _DriveTee(sys.stdout._stream if getattr(sys.stdout, "_revsi_tee", False) else sys.stdout, NOTEBOOK_LOG)

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN") or None   # optional; never printed
except Exception:
    pass
try:                                                # CPU runtimes have no nvidia-smi: that just means no GPU
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    gpu = ""
du = shutil.disk_usage("/content")
try:
    ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
except (ValueError, OSError):
    ram = float("nan")
print(f"Run mode:        {RUN_MODE}   (full = all questions of the frozen sample, not the whole benchmark)")
print(f"Output folder:   {OUTPUT_ROOT}")
print(f"Intended sample: {VIDEO_FRACTION:.0%} of unique ReVSI {REVSI_CONFIG} videos, seed {SEED}")
print(f"Model / frames:  {MODEL_REPO}; {FRAME_PROTOCOL}")
print(f"GPU:             {gpu or 'none (review/export/analysis work on CPU; inference needs a large GPU)'}")
print(f"RAM / disk:      {ram:.0f} GB RAM; {du.free / 1e9:.0f} GB free local disk")
print(f"Notebook log:    {NOTEBOOK_LOG}")

In [ ]:
# @title A2. Lightweight analysis dependencies (kernel only; separate from the model environment)
import importlib.util
if importlib.util.find_spec("av") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "av"], check=True)
import av, numpy, pandas, pyarrow, yaml, PIL, ipywidgets   # noqa: F401
ANALYSIS_VERSIONS = {m.__name__: getattr(m, "__version__", "?") for m in (av, numpy, pandas, pyarrow, yaml, PIL, ipywidgets)}
print("Analysis packages:", ANALYSIS_VERSIONS)

### Helper code: `revsi_lib.py`

In [ ]:
%%writefile /content/revsi/revsi_lib.py
"""Shared logic for the Cambrian-P / ReVSI / VSI-Bench validity audit (no GPU, no model code).

Imported by the notebook kernel, the inference worker and the review module. The ReVSI prompt formatter, scorer and
aggregator are NOT re-implemented: they are executed from the pinned lmms-eval file (load_official), unedited.
"""
import ast
import hashlib
import html
import io
import json
import math
import os
import random
import re
import time
import urllib.request
import zipfile

LIB_VERSION = "revsi-lib-1"
BENCHMARK = "ReVSI"
REVSI_REPO = "3dlg-hcvc/ReVSI"
REVSI_FIELDS = ("id", "dataset", "scene_id", "question_type", "question", "ground_truth", "options", "num_frames",
                "queried_object_ids")
CONDITIONS = ("normal", "black_frames")
OPTION_RE = re.compile(r"^([A-Z])\. ")
def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def stable_hash(obj, n=None):
    s = hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
    return s[:n] if n else s



def safe_key(video_key):
    """Filesystem-safe, collision-proof name for a media path; the mapping is kept in the manifests."""
    base = re.sub(r"[^A-Za-z0-9._-]+", "__", os.path.splitext(video_key)[0]).strip("_")[:80]
    return f"{base}-{hashlib.sha1(video_key.encode()).hexdigest()[:8]}"





# ----------------------------------------------------------------------------------------------- files
def write_json_atomic(path, obj):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    tmp = f"{path}.tmp-{os.getpid()}"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=1, ensure_ascii=False, default=str)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def write_jsonl_atomic(path, rows):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    tmp = f"{path}.tmp-{os.getpid()}"
    with open(tmp, "w") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False, default=str) + "\n")
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def read_jsonl(path, tolerate_truncated_tail=True):
    """Rows of a JSONL file. In tolerant mode a torn line (from an interrupted append) is skipped and counted."""
    rows, bad = [], 0
    if not os.path.exists(path):
        return rows, bad
    with open(path) as f:
        lines = f.read().split("\n")
    for k, line in enumerate(lines):
        if not line.strip():
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            # A torn line from an interrupted append (any position, since appends after a crash start a new line).
            if tolerate_truncated_tail:
                bad += 1
            else:
                raise
    return rows, bad


class Journal:
    """Append-only JSONL with fsync after every row: an interrupt loses at most the row being written."""

    def __init__(self, path):
        self.path = path
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

    def append(self, row):
        with open(self.path, "a+b") as f:
            # If an earlier append was interrupted mid-line, start on a fresh line so this row stays readable.
            f.seek(0, os.SEEK_END)
            if f.tell():
                f.seek(-1, os.SEEK_END)
                if f.read(1) != b"\n":
                    f.write(b"\n")
            f.write((json.dumps(row, ensure_ascii=False, default=str) + "\n").encode())
            f.flush()
            os.fsync(f.fileno())




def safe_extract_path(dest_root, member_name):
    """Destination for a tar member, refusing absolute paths, '..' and anything outside dest_root."""
    if not member_name or member_name.startswith(("/", "\\")) or re.match(r"^[A-Za-z]:", member_name):
        raise ValueError(f"unsafe member path {member_name!r}")
    norm = os.path.normpath(member_name)
    if norm.startswith("..") or os.path.isabs(norm) or "/../" in f"/{norm}/":
        raise ValueError(f"unsafe member path {member_name!r}")
    dest = os.path.realpath(os.path.join(dest_root, norm))
    root = os.path.realpath(dest_root)
    if not dest.startswith(root + os.sep):
        raise ValueError(f"member escapes the destination: {member_name!r}")
    return dest




# ----------------------------------------------------------------------------------------------- shards
def write_shard(shard_dir, name, rows, config_hash):
    """Durable shard: rows file plus a completion marker holding its hash. Written atomically, marker last."""
    os.makedirs(shard_dir, exist_ok=True)
    data = os.path.join(shard_dir, f"{name}.jsonl")
    write_jsonl_atomic(data, rows)
    marker = {"name": name, "rows": len(rows), "sha256": sha256_file(data), "config_hash": config_hash,
              "finished": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "lib": LIB_VERSION}
    write_json_atomic(os.path.join(shard_dir, f"{name}.done.json"), marker)
    return data


def load_shards(shard_dir, config_hash):
    """Rows of every complete, verified shard; incomplete, corrupted or foreign shards are reported, not used."""
    rows, report = [], {"complete": 0, "incomplete": [], "corrupted": [], "other_config": []}
    if not os.path.isdir(shard_dir):
        return rows, report
    for fn in sorted(os.listdir(shard_dir)):
        if not fn.endswith(".jsonl") or ".tmp-" in fn:
            continue
        name = fn[:-len(".jsonl")]
        marker_path = os.path.join(shard_dir, f"{name}.done.json")
        if not os.path.exists(marker_path):
            report["incomplete"].append(name)
            continue
        marker = json.load(open(marker_path))
        if marker.get("config_hash") != config_hash:
            report["other_config"].append(name)
            continue
        if sha256_file(os.path.join(shard_dir, fn)) != marker.get("sha256"):
            report["corrupted"].append(name)
            continue
        part, _ = read_jsonl(os.path.join(shard_dir, fn), tolerate_truncated_tail=False)
        rows += part
        report["complete"] += 1
    return rows, report


def merge_results(rows, config_hash):
    """One outcome per qa_id: the first successful answer wins; failed attempts are kept separately."""
    ok, failed, video_meta = {}, {}, {}
    for r in rows:
        if r.get("config_hash") != config_hash:
            continue
        if r.get("kind") == "video_meta":
            video_meta.setdefault(r["video_key"], r)
            continue
        if r.get("status") == "ok":
            # Deterministic whatever the file order: the earliest recorded answer wins.
            prev = ok.get(r["qa_id"])
            if prev is None or (r.get("answered_at") or "") < (prev.get("answered_at") or ""):
                ok[r["qa_id"]] = r
        else:
            failed.setdefault(r["qa_id"], []).append(r)
    for q in list(failed):
        if q in ok:
            del failed[q]
    return ok, failed, video_meta




# ----------------------------------------------------------------------------------------------- frames
def sample_indices(total_frames, num_frames):
    """The model wrapper's sampling (lmms-eval cambrian_p.load_video): np.linspace(0, total-1, k, dtype=int)."""
    import numpy as np
    if total_frames <= 0:
        raise ValueError("video has no frames")
    return [int(i) for i in np.linspace(0, total_frames - 1, num_frames, dtype=int)]


def frame_name(position, frame_index, seconds):
    """Input-frame file name keeping position, frame index and timestamp as separate fields."""
    return f"in{position:03d}_frame{frame_index:06d}_t{seconds:09.3f}s.jpg"


def parse_frame_name(name):
    m = re.fullmatch(r"(in|x)(\d{3})_frame(\d{6})_t(\d+\.\d{3})s\.jpg", name)
    if not m:
        return None
    return {"set": "input" if m.group(1) == "in" else "extra", "position": int(m.group(2)),
            "frame_index": int(m.group(3)), "seconds": float(m.group(4))}




# ----------------------------------------------------------------------------------------------- report
def esc(s):
    return html.escape("" if s is None else str(s), quote=True)


def rel_link(target_path, from_dir):
    return os.path.relpath(target_path, from_dir).replace(os.sep, "/")





class RangeNotSupported(IOError):
    """The server answered a Range request with the whole file (so random access is not available)."""


class HttpRangeFile(io.RawIOBase):
    """A read-only, seekable view of a remote file through HTTP Range requests, so zipfile can read the central
    directory and single members without downloading the archive. Raises if the server ignores Range."""

    def __init__(self, url, size, headers=None, timeout=300):
        self.url, self.size, self.pos, self.headers, self.timeout = url, size, 0, dict(headers or {}), timeout
        self.requests = self.bytes_read = 0
        req = urllib.request.Request(url, headers=dict(self.headers, Range="bytes=0-0"))
        with urllib.request.urlopen(req, timeout=timeout) as r:      # probe once, before zipfile hides the cause
            if r.status != 206:
                raise RangeNotSupported(f"server ignored the Range header (HTTP {r.status})")

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=0):
        self.pos = offset if whence == 0 else self.pos + offset if whence == 1 else self.size + offset
        return self.pos

    def readinto(self, b):
        n = min(len(b), self.size - self.pos)
        if n <= 0:
            return 0
        req = urllib.request.Request(self.url, headers=dict(self.headers, Range=f"bytes={self.pos}-{self.pos + n - 1}"))
        for attempt in range(4):
            try:
                with urllib.request.urlopen(req, timeout=self.timeout) as r:
                    if r.status != 206:
                        raise RangeNotSupported(f"server ignored the Range header (HTTP {r.status})")
                    data = r.read()
                break
            except RangeNotSupported:
                raise
            except OSError:
                if attempt == 3:
                    raise
                time.sleep(2 ** attempt)
        b[:len(data)] = data
        self.pos += len(data)
        self.requests += 1
        self.bytes_read += len(data)
        return len(data)


def open_zip(fileobj):
    return zipfile.ZipFile(io.BufferedReader(fileobj, 1 << 20) if isinstance(fileobj, HttpRangeFile) else fileobj)


def zip_listing(zf):
    return {i.filename: {"bytes": i.file_size, "compressed": i.compress_size, "crc32": i.CRC}
            for i in zf.infolist() if not i.is_dir()}


def extract_zip_member(zf, member, dest_root, expected_bytes=None, expected_crc=None):
    """Write one member under dest_root (safe path), verifying size and CRC (zipfile raises on a CRC mismatch)."""
    info = zf.getinfo(member)
    if expected_bytes is not None and info.file_size != expected_bytes:
        raise IOError(f"{member}: archive says {info.file_size} bytes, manifest says {expected_bytes}")
    if expected_crc is not None and info.CRC != expected_crc:
        raise IOError(f"{member}: archive CRC {info.CRC} differs from the manifest's {expected_crc}")
    dest = safe_extract_path(dest_root, member)
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    tmp = dest + ".part"
    with zf.open(info) as src, open(tmp, "wb") as out:
        while True:
            block = src.read(1 << 20)
            if not block:
                break
            out.write(block)
    if os.path.getsize(tmp) != info.file_size:
        os.remove(tmp)
        raise IOError(f"{member}: wrote {os.path.getsize(tmp)} bytes, expected {info.file_size}")
    os.replace(tmp, dest)
    return dest


def file_crc32(path):
    crc = 0
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            crc = zipfile.crc32(block, crc)
    return crc & 0xFFFFFFFF




# ----------------------------------------------------------------------------------------------- identities
def canonical_key(dataset, scene_id):
    """Canonical scene/video key: (source dataset, scene id)."""
    return f"{dataset}/{scene_id}"


def qa_uid(config, native_id):
    """A question id that cannot collide across configs or benchmarks (the same native id names different
    questions/answers in different ReVSI configs)."""
    return f"revsi:{config}:{int(native_id)}"


def job_id(uid, condition):
    return f"{uid}::{condition}"


def content_hash(row):
    """Hash of everything about a question that the model or the scorer sees."""
    keep = {k: row.get(k) for k in ("qa_uid", "dataset", "scene_id", "question_type", "question", "options",
                                     "reference_answer", "prompt")}
    return stable_hash(keep, 16)


def cache_key(identity_hash, qa_content_hash, frame_manifest_hash, condition):
    """benchmark + revision + config + model revision + prompt/decoding/preprocessing (all in identity_hash)
    + question content + exact input frames + experimental condition."""
    return stable_hash([identity_hash, qa_content_hash, frame_manifest_hash, condition], 20)


IDENTITY_KEYS = ("BENCHMARK", "DATASET_REPO", "DATASET_REVISION", "REVSI_CONFIG", "REVSI_SPLIT", "SAMPLING_UNIT",
                 "VIDEO_FRACTION", "SEED", "MODEL_REPO", "MODEL_REVISION", "CODE_COMMIT", "NUM_FRAMES", "FRAME_PROTOCOL",
                 "PROMPT_POLICY", "GENERATION", "PREPROCESSING", "SCORER", "RUN_KIND")


def config_identity(settings):
    missing = [k for k in IDENTITY_KEYS if k not in settings]
    if missing:
        raise ValueError(f"config_identity: missing {missing}")
    return stable_hash({k: settings[k] for k in IDENTITY_KEYS}, 16)


def check_run_config(path, identity_settings, extra=None):
    """Create run_config.json, or verify the saved one; a changed identity is refused (no mixing of runs)."""
    current = {"identity": identity_settings, "config_hash": config_identity(identity_settings)}
    if os.path.exists(path):
        saved = json.load(open(path))
        if saved.get("config_hash") != current["config_hash"]:
            diff = {k: (saved["identity"].get(k), v) for k, v in identity_settings.items() if saved["identity"].get(k) != v}
            raise RuntimeError("This run folder was made with different settings (saved -> now): "
                               f"{json.dumps(diff, default=str)[:800]}. Restore them, or set a new RUN_ID.")
        if extra:
            saved.update(extra)
            write_json_atomic(path, saved)
        return saved
    current.update(extra or {})
    write_json_atomic(path, current)
    return current


# ----------------------------------------------------------------------------------------------- official code
_OFFICIAL_NAMES = {"MCQ_QUESTION_TYPES", "NQ_QUESTION_TYPES", "REVSI_METRICS", "COMPOSITE_METRICS"}
_OFFICIAL_FUNCS = {"revsi_doc_to_text", "_mean_relative_accuracy", "revsi_process_results",
                   "_collapse_question_types", "_compute_all_subscores", "_aggregate_metric"}


def load_official(utils_path, yaml_path):
    """ReVSI's prompt formatter, scorer and aggregator, executed from the pinned lmms-eval files themselves.
    Only constants and functions run (module-level cache-path lines need lmms-eval and are skipped)."""
    import numpy as np
    import pandas as pd
    import yaml
    src = open(utils_path).read()
    tree = ast.parse(src)
    keep = [n for n in tree.body
            if (isinstance(n, ast.FunctionDef) and n.name in _OFFICIAL_FUNCS)
            or (isinstance(n, ast.Assign) and all(isinstance(t, ast.Name) and t.id in _OFFICIAL_NAMES for t in n.targets))]
    found = {n.name for n in keep if isinstance(n, ast.FunctionDef)} | \
        {t.id for n in keep if isinstance(n, ast.Assign) for t in n.targets}
    missing = (_OFFICIAL_FUNCS | _OFFICIAL_NAMES) - found
    if missing:
        raise RuntimeError(f"{utils_path}: official definitions not found: {sorted(missing)}")
    ns = {"np": np, "pd": pd, "os": os}
    exec(compile(ast.Module(body=keep, type_ignores=[]), utils_path, "exec"), ns)
    cfg = yaml.safe_load("".join(l for l in open(yaml_path) if "!function" not in l))
    return {"ns": ns, "sha256": sha256_file(utils_path), "yaml_sha256": sha256_file(yaml_path),
            "prompt_kwargs": cfg["lmms_eval_specific_kwargs"]["default"], "generation_kwargs": cfg["generation_kwargs"],
            "dataset_path": cfg["dataset_path"], "test_split": cfg["test_split"]}


def official_prompt(official, row):
    doc = {"question": row["question"], "question_type": row["question_type"], "options": row.get("options")}
    return official["ns"]["revsi_doc_to_text"](doc, dict(official["prompt_kwargs"]))


def option_letters(options):
    return [OPTION_RE.match(o).group(1) for o in options or [] if OPTION_RE.match(o)]


def official_parse(raw):
    """The official parse, copied from revsi_process_results for reporting only (the score itself comes from the
    official function): first whitespace token, trailing period removed."""
    return str(raw).strip().split(" ")[0].rstrip(".").strip()


def _float(x):
    try:
        v = float(x)
    except (TypeError, ValueError):
        return None
    return v if math.isfinite(v) else None


def score_answer(row, raw_answer, official, status="ok"):
    """Official score of one answer (exactly revsi_process_results' value) plus an automatic status. Missing
    responses and inference errors are never scored as wrong."""
    ns = official["ns"]
    out = {"qa_uid": row["qa_uid"], "native_id": row["native_id"], "video_key": row["video_key"],
           "question_type": row["question_type"], "task_family": row["task_family"], "raw_answer": raw_answer,
           "parsed_answer": None, "parsed_value": None, "reference_answer": row["reference_answer"],
           "official_score": None, "automatic_status": None, "candidate_reason": None, "error": None,
           "zero_target_exact_match": None, "scorer_sha256": official["sha256"]}
    qt = row["question_type"]
    if status == "missing":
        out["automatic_status"] = "missing"
        return out
    if status != "ok":
        out["automatic_status"] = "execution_error"
        return out
    if qt not in ns["MCQ_QUESTION_TYPES"] and qt not in ns["NQ_QUESTION_TYPES"]:
        out.update(automatic_status="unscorable", error=f"unsupported question type {qt!r}")
        return out
    doc = {"question_type": qt, "ground_truth": row["reference_answer"]}
    try:
        score = float(ns["revsi_process_results"](dict(doc), [raw_answer])["overall_acc"]["acc"])
    except Exception as exc:
        out.update(automatic_status="unscorable", error=f"official scorer raised {type(exc).__name__}: {exc}")
        return out
    parsed = official_parse(raw_answer)
    out.update(official_score=score, parsed_answer=parsed)
    if qt in ns["MCQ_QUESTION_TYPES"]:
        letters = option_letters(row.get("options"))
        if row["reference_answer"] not in letters:
            out.update(automatic_status="unscorable", error="reference letter is not one of the options")
        elif score == 1.0:
            out["automatic_status"] = "correct"
        elif parsed.upper() not in letters:
            out.update(automatic_status="format_error", candidate_reason=f"official parse {parsed!r} is not an option letter")
        else:
            out.update(automatic_status="wrong_candidate",
                       candidate_reason=f"official exact match 0: chose {parsed.upper()}, reference {row['reference_answer']}")
        return out
    ref, val = _float(row["reference_answer"]), _float(parsed)
    if ref is None:
        out.update(automatic_status="unscorable", error="numeric reference is not a number")
    elif val is None:
        out.update(automatic_status="format_error", candidate_reason=f"official parse {parsed!r} is not a number")
    else:
        out["parsed_value"] = val
        if ref == 0:
            # Relative error divides by the reference; a zero target gets a separate exact-match diagnostic.
            out["zero_target_exact_match"] = float(val == 0)
        if score == 1.0:
            out["automatic_status"] = "correct"
        elif score == 0.0:
            out.update(automatic_status="wrong_candidate",
                       candidate_reason=f"official MRA 0: {val:g} vs reference {ref:g}")
        else:
            out.update(automatic_status="imperfect", candidate_reason=f"official MRA {score:.2f} (partial credit)")
    return out


def official_aggregate(scores, official):
    """The official per-subtype means, composite collapse and macro overall over the given scored rows only
    (subset-aware: unselected or unanswered questions are never turned into zeros)."""
    rows = [{"question_type": s["question_type"], "acc": s["official_score"]} for s in scores
            if s.get("official_score") is not None]
    if not rows:
        return None
    return {k: float(v) for k, v in official["ns"]["_compute_all_subscores"](rows).items()}


def scene_bootstrap(scores, official, n_boot=1000, seed=0):
    """Scene-cluster bootstrap of every official aggregate: resample scenes (not questions) with replacement."""
    by_scene = {}
    for s in scores:
        if s.get("official_score") is not None:
            by_scene.setdefault(s["video_key"], []).append(s)
    keys = sorted(by_scene)
    if len(keys) < 2:
        return {}
    rng = random.Random(seed)
    draws = {}
    for _ in range(n_boot):
        sample = [s for k in (rng.choice(keys) for _ in keys) for s in by_scene[k]]
        for m, v in (official_aggregate(sample, official) or {}).items():
            draws.setdefault(m, []).append(v)
    out = {}
    for m, vs in draws.items():
        vs = sorted(vs)
        out[m] = {"ci95": [vs[int(0.025 * (len(vs) - 1))], vs[int(0.975 * (len(vs) - 1))]], "n_draws": len(vs),
                  "n_scenes": len(keys)}
    return out


# ----------------------------------------------------------------------------------------------- selection
def select_videos(video_keys, fraction, seed):
    population = sorted(set(video_keys))
    k = math.ceil(fraction * len(population))
    return sorted(random.Random(seed).sample(population, k))


def selection_hash(keys):
    return hashlib.sha256("\n".join(sorted(keys)).encode()).hexdigest()


def group_by_video(rows):
    out = {}
    for r in rows:
        out.setdefault(r["video_key"], []).append(r)
    return out

### Helper code: `revsi_pipeline.py`

In [ ]:
%%writefile /content/revsi/revsi_pipeline.py
"""Orchestration for the Cambrian-P / ReVSI audit: discovery and schema checks, the frozen video sample, the
historical VSI-Bench cohort and scene/QA overlap, media and frame manifests, inference jobs (normal and black-frame
control), merging with cache keys, official scoring, metrics with scene-cluster bootstrap, controls and the
scene-disjoint frequency baseline. Network access and the GPU worker are passed in, so the flow can be tested offline."""
import csv
import fnmatch
import json
import os
import re
import time
import zipfile

import revsi_lib as L

ALIAS_TO_VSI = {"ARKitScenes": "arkitscenes", "ScanNetv2": "scannet", "ScanNetPPv2": "scannetpp"}
TASK_OF = {  # ReVSI subtype -> official composite task (from COMPOSITE_METRICS; others are their own task)
    "object_rel_direction_forward_easy": "object_rel_direction", "object_rel_direction_backward_easy": "object_rel_direction",
    "object_rel_direction_forward_hard": "object_rel_direction", "object_rel_direction_backward_hard": "object_rel_direction",
    "object_rel_distance_closest": "object_rel_distance", "object_rel_distance_farthest": "object_rel_distance",
    "object_counting_single": "object_counting", "object_counting_multiple": "object_counting",
    "room_size_estimation_single": "room_size_estimation", "room_size_estimation_multiple": "room_size_estimation"}
VSI_FAMILY = {  # VSI-Bench question type -> the ReVSI task it corresponds to (documented common-task mapping)
    "object_counting": "object_counting", "object_abs_distance": "object_abs_distance",
    "object_size_estimation": "object_size_estimation", "room_size_estimation": "room_size_estimation",
    "object_rel_distance": "object_rel_distance", "object_rel_direction_easy": "object_rel_direction",
    "object_rel_direction_medium": "object_rel_direction", "object_rel_direction_hard": "object_rel_direction",
    "route_planning": "route_planning", "obj_appearance_order": None}


def task_of(question_type):
    return TASK_OF.get(question_type, question_type)


# ----------------------------------------------------------------------------------------------- discovery
def card_config_files(card_text, config, split, repo_files):
    import yaml
    meta = yaml.safe_load(card_text.split("---", 2)[1])
    cfg = next((c for c in meta.get("configs", []) if c.get("config_name") == config), None)
    if cfg is None:
        raise RuntimeError(f"config {config!r} not on the card; available: {[c.get('config_name') for c in meta.get('configs', [])]}")
    pats = [d["path"] for d in cfg["data_files"] if d.get("split") == split]
    if not pats:
        raise RuntimeError(f"config {config!r} has no split {split!r}")
    return sorted(f for f in repo_files for p in pats if fnmatch.fnmatch(f, p))


def read_revsi_parquet(paths):
    import pandas as pd
    rows = []
    for p in paths:
        for r in pd.read_parquet(p).to_dict("records"):
            opts = r.get("options")
            r["options"] = None if opts is None or (isinstance(opts, float) and opts != opts) else [str(o) for o in opts]
            q = r.get("queried_object_ids")
            r["queried_object_ids"] = None if q is None or (isinstance(q, float) and q != q) else [int(x) for x in q]
            r["id"] = int(r["id"])
            rows.append(r)
    return rows


def schema_problems(raw, official, config, frame_idx, visibility):
    """Actionable schema errors for the selected config (empty list = valid)."""
    p = []
    budget = config.split("_")[0]
    for r in raw:
        miss = [k for k in L.REVSI_FIELDS if k not in r]
        if miss:
            p.append(f"id {r.get('id')}: missing fields {miss}")
            continue
        qt = r["question_type"]
        mc, nq = qt in official["ns"]["MCQ_QUESTION_TYPES"], qt in official["ns"]["NQ_QUESTION_TYPES"]
        if not (mc or nq):
            p.append(f"id {r['id']}: question type {qt!r} is not in the official scorer")
        if mc and (not r["options"] or str(r["ground_truth"]) not in L.option_letters(r["options"])):
            p.append(f"id {r['id']}: multiple-choice row without valid options/reference letter")
        if nq and L._float(r["ground_truth"]) is None:
            p.append(f"id {r['id']}: numeric reference {r['ground_truth']!r} is not a number")
        if str(r["num_frames"]) != budget:
            p.append(f"id {r['id']}: num_frames {r['num_frames']!r} but config {config}")
        if budget != "all" and r["scene_id"] not in frame_idx:
            p.append(f"id {r['id']}: scene {r['scene_id']} has no entry in sampled_video_frame_idx.json")
        if r["scene_id"] not in visibility:
            p.append(f"id {r['id']}: scene {r['scene_id']} has no entry in obj_visibility.json")
    if len({r["id"] for r in raw}) != len(raw):
        p.append("native ids are not unique within the config")
    by_scene = {}
    for r in raw:
        by_scene.setdefault(r["scene_id"], set()).add(r["dataset"])
    clash = [s for s, d in by_scene.items() if len(d) > 1]
    if clash:
        p.append(f"scene ids shared by several datasets (scene-only metadata keys would be ambiguous): {clash[:3]}")
    if budget != "all":
        key = f"{budget}-frame"
        bad = [s for s in by_scene if s in frame_idx and (len(frame_idx[s].get(key, [])) != int(budget)
                                                          or frame_idx[s][key] != sorted(frame_idx[s][key]))]
        if bad:
            p.append(f"{len(bad)} scenes do not have {budget} increasing official frame indices, e.g. {bad[:2]}")
        nest = [s for s in by_scene if s in frame_idx and not (
            set(frame_idx[s].get("16-frame", [])) <= set(frame_idx[s].get("32-frame", [])) <= set(frame_idx[s].get("64-frame", [])))]
        if nest:
            p.append(f"{len(nest)} scenes break the nested 16 < 32 < 64 sampling, e.g. {nest[:2]}")
    return p[:50]


def visibility_flag(row, visibility, budget):
    """Release metadata: are the queried objects visible in this budget's frames? An aggregate per-budget flag,
    not proof that a spatial relation or a metric size can be read from the frames."""
    ids = row.get("queried_object_ids")
    if budget == "all" or not ids:
        return "unknown"
    vis = {o["object_id"]: o.get(f"visibility_{budget}") for o in visibility.get(row["scene_id"], [])}
    flags = [vis.get(i) for i in ids]
    if any(f is None for f in flags):
        return "unknown"
    return "all_visible" if all(flags) else "none_visible" if not any(flags) else "some_visible"


def normalize_row(r, config, official, visibility):
    family = "multiple_choice" if r["question_type"] in official["ns"]["MCQ_QUESTION_TYPES"] else "numeric"
    q = {"qa_uid": L.qa_uid(config, r["id"]), "native_id": r["id"], "config": config,
         "video_key": L.canonical_key(r["dataset"], r["scene_id"]), "dataset": r["dataset"], "scene_id": r["scene_id"],
         "question_type": r["question_type"], "task": task_of(r["question_type"]), "task_family": family,
         "question": r["question"], "options": r["options"], "reference_answer": str(r["ground_truth"]),
         "queried_object_ids": r["queried_object_ids"], "num_frames": str(r["num_frames"])}
    q["prompt"] = L.official_prompt(official, q)
    q["visibility"] = visibility_flag(q, visibility, config.split("_")[0])
    q["content_hash"] = L.content_hash(q)
    return q


def prompt_independent_of_reference(q, official):
    probe = dict(q, reference_answer="@@REF@@", ground_truth="@@REF@@")
    return L.official_prompt(official, probe) == q["prompt"] and "@@REF@@" not in q["prompt"]


# ----------------------------------------------------------------------------------------------- sample
def freeze_sample(out_root, qa_rows, settings, file_hashes):
    """Sample ceil(fraction * N) unique (dataset, scene_id) videos once (no answers or predictions are read);
    keep every released question of those videos; verify instead of resampling on later runs."""
    path = os.path.join(out_root, "sample_manifest.json")
    population = sorted({q["video_key"] for q in qa_rows})
    keys = L.select_videos(population, settings["VIDEO_FRACTION"], settings["SEED"])
    chosen = sorted((q for q in qa_rows if q["video_key"] in set(keys)), key=lambda q: (q["video_key"], q["native_id"]))
    count = lambda rows, k: dict(sorted({v: sum(1 for r in rows if r[k] == v) for v in {r[k] for r in rows}}.items()))
    man = {"benchmark": L.BENCHMARK, "dataset_repo": settings["DATASET_REPO"], "dataset_revision": settings["DATASET_REVISION"],
           "config": settings["REVSI_CONFIG"], "split": settings["REVSI_SPLIT"], "annotation_sha256": file_hashes,
           "sampling_unit": "video = unique (source dataset, scene_id)", "seed": settings["SEED"],
           "fraction": settings["VIDEO_FRACTION"], "population_videos": len(population), "population_keys": population,
           "population_questions": len(qa_rows), "selected_videos": len(keys), "selected_keys": keys,
           "selection_hash": L.selection_hash(keys), "selected_questions": len(chosen),
           "selected_qa_uids_hash": L.stable_hash([q["qa_uid"] for q in chosen]),
           "per_task": count(chosen, "task"), "per_question_type": count(chosen, "question_type"),
           "per_source": count(chosen, "dataset"),
           "per_source_videos": {d: sum(1 for k in keys if k.split("/")[0] == d) for d in sorted({k.split("/")[0] for k in keys})},
           "algorithm": "random.Random(SEED).sample(sorted(unique (dataset, scene_id)), ceil(VIDEO_FRACTION * N)); "
                        "all released questions of the selected videos in the selected config; answers and predictions "
                        "are not read", "note": "10% of videos is not 10% of questions"}
    if os.path.exists(path):
        saved = json.load(open(path))
        diff = [k for k, v in man.items() if saved.get(k) != v]
        if diff:
            raise RuntimeError(f"sample_manifest.json differs from the sample recomputed from the pinned data: {diff}")
        qa, _ = L.read_jsonl(os.path.join(out_root, "qa_manifest.jsonl"), tolerate_truncated_tail=False)
        if qa != chosen:
            raise RuntimeError("qa_manifest.jsonl differs from the recomputed selection")
        return saved, qa, False
    L.write_jsonl_atomic(os.path.join(out_root, "qa_manifest.jsonl"), chosen)
    man["created"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    L.write_json_atomic(path, man)
    return man, chosen, True


def control_subset(qa, max_n=100):
    """Predeclared black-frame control questions: up to max_n, balanced round-robin over question types, ordered
    by a hash of the question id only (chosen before any prediction exists)."""
    by_type = {}
    for q in sorted(qa, key=lambda q: L.stable_hash(["control", q["qa_uid"]])):
        by_type.setdefault(q["question_type"], []).append(q)
    out, types = [], sorted(by_type)
    while len(out) < max_n and any(by_type[t] for t in types):
        for t in types:
            if by_type[t] and len(out) < max_n:
                out.append(by_type[t].pop(0))
    return sorted(out, key=lambda q: q["qa_uid"])


# ----------------------------------------------------------------------------------------------- historical cohort + overlap
def import_historical_vsi(run_dir):
    """The earlier VSI-Bench run as a separate, failure-selected historical cohort (never ReVSI predictions)."""
    out = {"run_dir": run_dir, "available": bool(run_dir) and os.path.isdir(run_dir or ""), "provenance_checks": {}}
    if not out["available"]:
        out["note"] = "no historical VSI-Bench run found; the primary ReVSI evaluation does not need it"
        return out
    j = lambda *p: os.path.join(run_dir, *p)
    rc = json.load(open(j("run_config.json"))) if os.path.exists(j("run_config.json")) else {}
    ident = rc.get("identity", {})
    sel = json.load(open(j("selection.json"))) if os.path.exists(j("selection.json")) else {}
    qa, _ = L.read_jsonl(j("qa_manifest.jsonl"))
    scores, _ = L.read_jsonl(j("scores.jsonl"))
    queue = json.load(open(j("review_queue.json"))) if os.path.exists(j("review_queue.json")) else {}
    shards = [f for f in os.listdir(j("predictions"))] if os.path.isdir(j("predictions")) else []
    metas = 0
    for f in shards:
        if f.endswith(".jsonl"):
            metas += sum(1 for r in L.read_jsonl(j("predictions", f))[0] if r.get("kind") == "video_meta" and r.get("frame_sha1"))
    out.update(config_hash=rc.get("config_hash"), identity={k: ident.get(k) for k in (
        "DATASET_REPO", "DATASET_CONFIG", "DATASET_REVISION", "MODEL_REPO", "MODEL_REVISION", "NUM_FRAMES", "CODE_COMMIT")},
        selected_videos=sel.get("selected_video_keys", []), qa=qa, scores=scores,
        candidate_videos=[e["video_key"] for e in queue.get("entries", [])])
    out["provenance_checks"] = {
        "run_config present": bool(rc), "model is Cambrian-P-7B": ident.get("MODEL_REPO") == "nyu-visionx/Cambrian-P-7B",
        "frames recorded": metas > 0, "NUM_FRAMES": ident.get("NUM_FRAMES"),
        "questions with scores": sum(1 for s in scores if s.get("official_score") is not None),
        "video_meta rows with frame hashes": metas}
    return out


def _norm_q(text):
    t = re.sub(r"[^\w\s()]", " ", text.lower())
    t = re.sub(r"\b(this|the) (room|scene|space|video)\b", "the scene", t)
    return re.sub(r"\s+", " ", t).strip()


def scene_overlap(revsi_all_rows, sample_keys, vsi_rows, hist):
    """One row per scene in ReVSI (config) or VSI-Bench. Same scene does not mean same video encoding,
    questions, object definitions or answers."""
    rv = {}
    for r in revsi_all_rows:
        rv.setdefault((r["dataset"], r["scene_id"]), 0)
        rv[(r["dataset"], r["scene_id"])] += 1
    vs = {}
    for r in vsi_rows or []:
        vs.setdefault((r["dataset"], r["scene_name"]), 0)
        vs[(r["dataset"], r["scene_name"])] += 1
    inv = {v: k for k, v in ALIAS_TO_VSI.items()}
    hist_sel = set(hist.get("selected_videos") or [])
    hist_cand = set(hist.get("candidate_videos") or [])
    rows = []
    for (d, s), n in sorted(rv.items()):
        vd = ALIAS_TO_VSI.get(d)
        vk = f"{vd}/{s}" if vd else None
        rows.append({"canonical_key": L.canonical_key(d, s), "revsi_dataset": d, "scene_id": s, "revsi_questions": n,
                     "in_revsi_sample": L.canonical_key(d, s) in set(sample_keys), "vsi_dataset_alias": vd or "",
                     "in_vsibench": (vd, s) in vs, "vsibench_questions": vs.get((vd, s), 0),
                     "in_historical_vsi_cohort": vk in hist_sel, "in_historical_vsi_candidates": vk in hist_cand})
    for (vd, s), n in sorted(vs.items()):
        if (inv.get(vd), s) not in rv:
            rows.append({"canonical_key": "", "revsi_dataset": inv.get(vd, ""), "scene_id": s, "revsi_questions": 0,
                         "in_revsi_sample": False, "vsi_dataset_alias": vd, "in_vsibench": True, "vsibench_questions": n,
                         "in_historical_vsi_cohort": f"{vd}/{s}" in hist_sel,
                         "in_historical_vsi_candidates": f"{vd}/{s}" in hist_cand})
    return rows


def qa_matches(sample_qa, vsi_rows):
    """Conservative cross-benchmark question matching on the same scene: same task family, same question after
    normalising scene wording, and the same option texts for multiple choice. Object identity across the two
    benchmarks' annotations is never verified here, so a match is labelled as such."""
    by_scene = {}
    for r in vsi_rows or []:
        by_scene.setdefault((r["dataset"], r["scene_name"]), []).append(r)
    out = []
    for q in sample_qa:
        vd = ALIAS_TO_VSI.get(q["dataset"])
        cands = by_scene.get((vd, q["scene_id"]), []) if vd else []
        best, why = None, ("no VSI-Bench video for this scene" if not cands else "no VSI-Bench question with the same "
                           "task, normalised wording and options")
        for v in cands:
            if VSI_FAMILY.get(v["question_type"]) != q["task"]:
                continue
            if _norm_q(v["question"]) != _norm_q(q["question"]):
                continue
            vo = sorted(o.split(". ", 1)[-1].strip().lower() for o in (v.get("options") or []))
            qo = sorted(o.split(". ", 1)[-1].strip().lower() for o in (q.get("options") or []))
            if vo != qo:
                why = "same wording but different options"
                continue
            best, why = v, ("same task, wording (after normalising scene words) and options; object identity and units "
                            "across the two annotations are unverified")
            break
        out.append({"qa_uid": q["qa_uid"], "video_key": q["video_key"], "question_type": q["question_type"],
                    "vsibench_id": best["id"] if best else "", "match_status": "template_match_identity_unverified" if best
                    else "unmatched", "revsi_reference": q["reference_answer"],
                    "vsibench_reference": best["ground_truth"] if best else "", "rationale": why})
    return out


def write_csv(path, rows, cols=None):
    cols = cols or (list(rows[0]) if rows else [])
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    tmp = path + ".tmp"
    with open(tmp, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
        w.writeheader()
        for r in rows:
            w.writerow({k: (json.dumps(v) if isinstance(v, (list, dict)) else v) for k, v in r.items()})
    os.replace(tmp, path)


def read_csv(path):
    with open(path, newline="") as f:
        return list(csv.DictReader(f))


# ----------------------------------------------------------------------------------------------- media + frames
def media_manifest(sample_keys, listing, config, verify=True):
    budget = config.split("_")[0]
    rows = []
    for k in sorted(sample_keys):
        sid = k.split("/", 1)[1]
        m_in, m_all = f"{budget}_frame/{sid}.mp4", f"all_frame/{sid}.mp4"
        rows.append({"video_key": k, "scene_id": sid, "safe_key": L.safe_key(k), "input_member": m_in,
                     "input_bytes": (listing.get(m_in) or {}).get("bytes"), "input_crc32": (listing.get(m_in) or {}).get("crc32"),
                     "verify_member": m_all if verify else None,
                     "verify_bytes": (listing.get(m_all) or {}).get("bytes") if verify else None,
                     "verify_crc32": (listing.get(m_all) or {}).get("crc32") if verify else None,
                     "input_missing": m_in not in listing})
    return rows


def frame_manifest_hash(media_row, official_indices, num_frames, protocol):
    """Identity of the exact model input, known before inference: the media file (by CRC), the official source
    indices and the protocol. Any change invalidates cached predictions."""
    return L.stable_hash([media_row["input_member"], media_row["input_crc32"], list(official_indices), num_frames, protocol], 16)


# ----------------------------------------------------------------------------------------------- jobs, merge, coverage
def build_jobs(qa, control_qa, frame_hashes, identity_hash, run_controls):
    """One job per (question, condition) with its full cache key."""
    jobs = []
    ctrl = {q["qa_uid"] for q in control_qa} if run_controls else set()
    for q in qa:
        for cond in ("normal", "black_frames"):
            if cond == "black_frames" and q["qa_uid"] not in ctrl:
                continue
            fh = frame_hashes.get(q["video_key"])
            jobs.append({"job_id": L.job_id(q["qa_uid"], cond), "qa_uid": q["qa_uid"], "condition": cond,
                         "video_key": q["video_key"], "frame_manifest_hash": fh,
                         "cache_key": L.cache_key(identity_hash, q["content_hash"], fh, cond) if fh else None})
    return jobs


def gather(shard_dir, journal_path, config_hash, jobs):
    """Merge durable shards + journal. A row counts only if its cache key equals the job's current cache key;
    others are reported as incompatible cached rows (kept on disk, never used)."""
    rows, report = L.load_shards(shard_dir, config_hash)
    jr, torn = L.read_jsonl(journal_path) if journal_path else ([], 0)
    report["journal_torn_lines"] = torn
    want = {j["job_id"]: j["cache_key"] for j in jobs}
    ok, failed, meta, incompatible = {}, {}, {}, 0
    for r in rows + jr:
        if r.get("config_hash") != config_hash:
            continue
        if r.get("kind") == "video_meta":
            meta.setdefault((r["video_key"], r.get("condition", "normal")), r)
            continue
        jid = r.get("job_id")
        if jid not in want:
            continue
        if r.get("cache_key") != want[jid]:
            incompatible += 1
            continue
        if r.get("status") == "ok":
            prev = ok.get(jid)
            if prev is None or (r.get("answered_at") or "") < (prev.get("answered_at") or ""):
                ok[jid] = r
        else:
            failed.setdefault(jid, []).append(r)
    for k in list(failed):
        if k in ok:
            del failed[k]
    report["incompatible_cached_rows"] = incompatible
    return ok, failed, meta, report


def coverage_counts(jobs, ok, failed, blocked_videos):
    c = {}
    for cond in ("normal", "black_frames"):
        js = [j for j in jobs if j["condition"] == cond]
        if not js:
            continue
        blocked = [j for j in js if j["video_key"] in blocked_videos and j["job_id"] not in ok]
        c[cond] = {"expected": len(js), "succeeded": sum(j["job_id"] in ok for j in js),
                   "failed": sum(j["job_id"] in failed for j in js),
                   "blocked_media": len(blocked),
                   "pending": sum(j["job_id"] not in ok and j["job_id"] not in failed and j["video_key"] not in blocked_videos
                                  for j in js)}
    return c


def run_jobs(jobs, qa_by_uid, media_by_video, frame_info, ctx, log=print):
    """Run every pending job, video by video (normal and control conditions share one decode)."""
    ok, failed, _, _ = gather(ctx["shard_dir"], ctx["journal"], ctx["config_hash"], jobs)
    pending = [j for j in jobs if j["job_id"] not in ok and j["video_key"] not in ctx.get("blocked", set())
               and j["cache_key"]]
    by_video = L.group_by_video(pending)
    summary = {"pending_jobs": len(pending), "videos": len(by_video), "stopped": None, "missing": []}
    if not pending:
        log("Nothing pending: every scheduled job has a saved result with the current cache key.")
        return summary
    batch, size = [], 0

    def flush():
        nonlocal batch, size
        if not batch:
            return 0
        bpath = os.path.join(ctx["work_dir"], f"batch-{int(time.time() * 1000)}.json")
        L.write_json_atomic(bpath, {"videos": batch})
        log(f"  running the model on {len(batch)} video(s), {sum(len(b['jobs']) for b in batch)} job(s) ...")
        code = ctx["run_worker"](bpath, [])
        for b in batch:
            ctx["release"](b["video_key"], b["local_path"])
        batch, size = [], 0
        return code

    for v in sorted(by_video):
        m = media_by_video[v]
        try:
            path = ctx["fetch_video"](v)
        except (KeyError, zipfile.BadZipFile) as exc:
            ctx["log_missing"](v, f"{type(exc).__name__}: {exc}")
            summary["missing"].append(v)
            continue
        fi = frame_info[v]
        batch.append({"video_key": v, "safe_key": m["safe_key"], "local_path": path,
                      "official_frame_indices": fi["official_frame_indices"], "timestamps_s": fi["timestamps_s"],
                      "jobs": [dict(j, prompt=qa_by_uid[j["qa_uid"]]["prompt"],      # never the reference answer
                                    native_id=qa_by_uid[j["qa_uid"]]["native_id"]) for j in by_video[v]]})
        size += os.path.getsize(path)
        if size >= ctx["batch_bytes"]:
            code = flush()
            if code:
                summary["stopped"] = code
                return summary
    code = flush()
    if code:
        summary["stopped"] = code
    return summary


# ----------------------------------------------------------------------------------------------- scoring + metrics
def score_jobs(qa, jobs, ok, failed, blocked, official):
    by_uid = {q["qa_uid"]: q for q in qa}
    scores = []
    for j in jobs:
        q = by_uid[j["qa_uid"]]
        if j["job_id"] in ok:
            s = L.score_answer(q, ok[j["job_id"]]["raw_answer"], official)
        elif j["job_id"] in failed or j["video_key"] in blocked:
            s = L.score_answer(q, None, official, status="error")
            s["error"] = (failed[j["job_id"]][-1].get("error") if j["job_id"] in failed else f"media: {blocked[j['video_key']]}")
        else:
            s = L.score_answer(q, None, official, status="missing")
        s.update(condition=j["condition"], job_id=j["job_id"], dataset=q["dataset"], task=q["task"],
                 visibility=q["visibility"], cache_key=j["cache_key"])
        scores.append(s)
    return scores


def metrics_block(scores, official, n_boot=1000):
    sc = [s for s in scores if s.get("official_score") is not None]
    agg = L.official_aggregate(sc, official)
    st = {}
    for s in scores:
        st[s["automatic_status"]] = st.get(s["automatic_status"], 0) + 1
    per_task_n = {}
    for s in sc:
        per_task_n[s["task"]] = per_task_n.get(s["task"], 0) + 1
    return {"official_subset_macro": agg, "bootstrap_scene_cluster": L.scene_bootstrap(sc, official, n_boot) if agg else {},
            "qa_weighted_mean": (sum(s["official_score"] for s in sc) / len(sc)) if sc else None,
            "scored_questions": len(sc), "scored_videos": len({s["video_key"] for s in sc}),
            "per_task_scored": dict(sorted(per_task_n.items())), "automatic_status": st,
            "excluded": {"missing": st.get("missing", 0), "execution_error": st.get("execution_error", 0),
                         "unscorable": st.get("unscorable", 0)}}


def control_results(scores, control_qa):
    """Paired normal vs black-frame results for the predeclared control questions."""
    byj = {(s["qa_uid"], s["condition"]): s for s in scores}
    rows = []
    for q in control_qa:
        a, b = byj.get((q["qa_uid"], "normal")), byj.get((q["qa_uid"], "black_frames"))
        pair = bool(a and b and a.get("official_score") is not None and b.get("official_score") is not None)
        rows.append({"qa_uid": q["qa_uid"], "video_key": q["video_key"], "question_type": q["question_type"],
                     "task": q["task"], "normal_answer": (a or {}).get("raw_answer"), "black_answer": (b or {}).get("raw_answer"),
                     "normal_score": (a or {}).get("official_score"), "black_score": (b or {}).get("official_score"),
                     "pair_available": pair,
                     "same_parsed_answer": (a["parsed_answer"] == b["parsed_answer"]) if pair else None,
                     "score_change": (b["official_score"] - a["official_score"]) if pair else None,
                     "status": "paired" if pair else "unavailable: " + ", ".join(
                         f"{c} {('missing' if not x else x.get('automatic_status'))}" for c, x in (("normal", a), ("black", b))
                         if not x or x.get("official_score") is None)})
    return rows


def frequency_baseline(qa, official, folds=5):
    """Local diagnostic baseline with scene-disjoint cross-validation: for each held-out scene fold, priors are fit
    on the other folds only (MC: most frequent valid letter for the question type; numeric: the constant that
    maximises the official MRA on the training folds). Never fit on the answers it is evaluated against."""
    ns = official["ns"]
    fold_of = lambda v: int(L.stable_hash(["fold", v], 8), 16) % folds
    preds = []
    for f in range(folds):
        train = [q for q in qa if fold_of(q["video_key"]) != f]
        test = [q for q in qa if fold_of(q["video_key"]) == f]
        for q in test:
            tr = [t for t in train if t["question_type"] == q["question_type"]]
            if q["task_family"] == "multiple_choice":
                counts = {}
                for t in tr:
                    counts[t["reference_answer"]] = counts.get(t["reference_answer"], 0) + 1
                valid = L.option_letters(q["options"])
                ranked = sorted(valid, key=lambda x: (-counts.get(x, 0), x))
                pred = ranked[0] if ranked else ""
            else:
                cands = sorted({L._float(t["reference_answer"]) for t in tr if L._float(t["reference_answer"]) is not None})
                best, best_v = None, -1
                for c in cands:
                    v = sum(ns["revsi_process_results"]({"question_type": t["question_type"], "ground_truth": t["reference_answer"]},
                                                        [repr(c)])["overall_acc"]["acc"] for t in tr)
                    if v > best_v:
                        best, best_v = c, v
                pred = repr(best) if best is not None else ""
            s = L.score_answer(q, pred, official)
            s.update(fold=f, train_questions=len(tr), task=q["task"], condition="frequency_baseline")
            preds.append(s)
    return preds

### Helper code: `revsi_frames.py`

In [ ]:
%%writefile /content/revsi/revsi_frames.py
"""Frame tools for the notebook kernel (PyAV + PIL; no model environment needed).

- verify_input_frames: compare every frame of the released fixed-budget video with the all-frame video at the official
  source indices (re-encoding makes them close, not identical; the per-frame differences are recorded) and derive
  timestamps from the all-frame video's real frame rate.
- extra_frames: diagnostic frames from the all-frame video (1 fps and/or time windows), kept apart from model inputs.
- contact_sheet: labelled grid of images.
"""
import os

import numpy as np
from PIL import Image, ImageDraw

MATCH_MEDIAN_MAX, MATCH_FRAME_MAX = 15.0, 45.0     # mean |pixel difference| (0-255) tolerated from re-encoding


def _open(path):
    import av
    c = av.open(path)
    s = c.streams.video[0]
    return c, s


def video_info(path):
    c, s = _open(path)
    n = sum(1 for _ in c.decode(s))
    fps = float(s.average_rate or 0) or None
    c.close()
    return n, fps


def frames_at(path, indices):
    """{index: RGB array} for the requested frame indices (one sequential decode)."""
    want, out = set(indices), {}
    c, s = _open(path)
    for i, f in enumerate(c.decode(s)):
        if i in want:
            out[i] = f.to_ndarray(format="rgb24")
        if len(out) == len(want):
            break
    c.close()
    return out


def verify_input_frames(input_path, all_path, official_indices):
    """Per-frame mean |difference| between input frame k and all-frame[official_indices[k]] (and its neighbours)."""
    n_in, _ = video_info(input_path)
    n_all, fps_all = video_info(all_path)
    if n_in != len(official_indices):
        return {"status": "mismatch", "reason": f"input video has {n_in} frames, {len(official_indices)} official indices",
                "input_frames": n_in, "all_frames": n_all, "fps_all": fps_all}
    if max(official_indices) >= n_all:
        return {"status": "mismatch", "reason": f"official index {max(official_indices)} beyond the all-frame video ({n_all})",
                "input_frames": n_in, "all_frames": n_all, "fps_all": fps_all}
    inp = frames_at(input_path, range(n_in))
    near = sorted({j for i in official_indices for j in (i - 1, i, i + 1) if 0 <= j < n_all})
    allf = frames_at(all_path, near)
    mad = lambda a, b: float(np.abs(a.astype(np.int16) - b.astype(np.int16)).mean()) if a.shape == b.shape else 255.0
    at, nb = [], []
    for k, i in enumerate(official_indices):
        at.append(round(mad(inp[k], allf[i]), 2))
        nb.append(round(min([mad(inp[k], allf[j]) for j in (i - 1, i + 1) if j in allf] or [255.0]), 2))
    med = float(np.median(at))
    ok = med <= MATCH_MEDIAN_MAX and max(at) <= MATCH_FRAME_MAX
    return {"status": "match" if ok else "mismatch", "input_frames": n_in, "all_frames": n_all, "fps_all": fps_all,
            "timestamps_s": [round(i / fps_all, 3) for i in official_indices] if fps_all else None,
            "mad_at_official_index": at, "mad_best_neighbour": nb, "median_mad": round(med, 2), "max_mad": max(at),
            "method": f"mean |RGB difference| (0-255) of input frame k vs all-frame frame at official index k (PyAV); "
                      f"match = median <= {MATCH_MEDIAN_MAX:g} and every frame <= {MATCH_FRAME_MAX:g} (re-encoding tolerance)",
            "reason": "" if ok else f"median {med:.1f} / max {max(at):.1f} above the re-encoding tolerance"}


def contact_sheet(images, captions, out_path, cols=8, thumb=200):
    """Labelled grid; returns out_path."""
    th = []
    for im in images:
        t = (im if isinstance(im, Image.Image) else Image.fromarray(im)).copy()
        t.thumbnail((thumb, thumb))
        th.append(t)
    if not th:
        return None
    w, h = max(t.width for t in th), max(t.height for t in th) + 16
    sheet = Image.new("RGB", (cols * w, -(-len(th) // cols) * h), "white")
    d = ImageDraw.Draw(sheet)
    for k, (t, cap) in enumerate(zip(th, captions)):
        x, y = (k % cols) * w, (k // cols) * h
        sheet.paste(t, (x, y + 16))
        d.text((x + 2, y + 2), cap, fill=(0, 0, 0))
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    sheet.save(out_path, quality=88)
    return out_path


def extra_frames(all_path, out_dir, fps=1.0, windows=(), max_frames=400):
    """Diagnostic frames from the all-frame video, named x<index>_t<seconds>.jpg, plus a contact sheet. These are
    NOT model inputs."""
    n, src_fps = video_info(all_path)
    src_fps = src_fps or 10.0
    want = set()
    if fps and fps > 0:
        want.update(range(0, n, max(1, int(round(src_fps / fps)))))
    for a, b, wf in windows:
        want.update(range(max(0, int(a * src_fps)), min(n, int(b * src_fps) + 1), max(1, int(round(src_fps / wf)))))
    idx = sorted(want)[:max_frames]
    got = frames_at(all_path, idx)
    os.makedirs(out_dir, exist_ok=True)
    files = []
    for i in idx:
        name = f"x{i:06d}_t{i / src_fps:09.3f}s.jpg"
        Image.fromarray(got[i]).save(os.path.join(out_dir, name), quality=88)
        files.append({"file": name, "frame_index": i, "timestamp_s": round(i / src_fps, 3)})
    sheet = contact_sheet([got[i] for i in idx], [f"x{i} {i / src_fps:.1f}s" for i in idx],
                          os.path.join(out_dir, "contact_sheet_extra.jpg"))
    return {"files": files, "sheet": sheet, "fps_source": src_fps, "requested": {"fps": fps, "windows": list(windows)}}

### Helper code: `revsi_review.py`

In [ ]:
%%writefile /content/revsi/revsi_review.py
"""Review queues, candidate export, review records (notebook form and external CSV share one validator),
verified-failure / benchmark-issue classification, deliverable manifests, counts and reports for the ReVSI audit.

Everything is recomputed from files in OUTPUT_ROOT, so review-only sessions need no model and no in-memory state."""
import csv
import html
import json
import os
import re
import shutil
import time

import revsi_lib as L

ANNOTATION = ("supported", "wrong", "ambiguous", "unreviewed")
EVIDENCE = ("sufficient", "partial", "absent", "uncertain", "unreviewed")
VERDICT = ("wrong", "correct", "uncertain", "unreviewed")
STAGE = ("draft", "final")
EVIDENCE_SOURCE = ("model_inputs", "extra_frames", "release_3d_metadata", "other")
CANDIDATE_STATUSES = ("wrong_candidate", "format_error")
TEMPLATE_COLS = ["required", "queue", "rank", "qa_uid", "video_key", "question_type", "question", "options",
                 "annotation_status", "input_evidence_status", "model_verdict", "review_stage", "reviewer", "notes",
                 "citations", "evidence_source", "proposed_answer", "reference_answer", "model_answer", "official_score",
                 "qa_hash", "prediction_hash", "input_manifest_hash"]


def now():
    return time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())


def video_dir_name(video_key):
    return L.safe_key(video_key)


# ----------------------------------------------------------------------------------------------- queues
def audit_queue(qa, seed, n=30, per_video=2):
    """Seeded audit chosen from question ids only (independent of correctness: includes apparent successes)."""
    order = sorted(qa, key=lambda q: L.stable_hash(["audit", seed, q["qa_uid"]]))
    by_task, used, out = {}, {}, []
    for q in order:
        by_task.setdefault(q["task"], []).append(q)
    tasks = sorted(by_task)
    while len(out) < n and any(by_task[t] for t in tasks):
        for t in tasks:
            while by_task[t] and len(out) < n:
                q = by_task[t].pop(0)
                if used.get(q["video_key"], 0) < per_video:
                    used[q["video_key"]] = used.get(q["video_key"], 0) + 1
                    out.append(q)
                    break
    return out


def diagnostic_queue(scores, qa_by_uid, target=20, supporting=2):
    """Automatic failure/format candidates (normal condition), one primary question per video in a deterministic,
    diversity-first order; every other candidate question is kept as supporting or reserve."""
    cands = [s for s in scores if s.get("condition") == "normal" and s.get("automatic_status") in CANDIDATE_STATUSES]
    by_video = {}
    for s in cands:
        by_video.setdefault(s["video_key"], []).append(s)
    tier = lambda s: CANDIDATE_STATUSES.index(s["automatic_status"])
    task_n, src_n, rows, remaining = {}, {}, [], dict(by_video)
    while remaining:
        def key(v):
            p = min(remaining[v], key=lambda s: (tier(s), task_n.get(s["task"], 0), L.stable_hash(s["qa_uid"])))
            return (tier(p), src_n.get(v.split("/")[0], 0), task_n.get(p["task"], 0), L.stable_hash(v))
        v = min(remaining, key=key)
        ss = sorted(remaining.pop(v), key=lambda s: (tier(s), task_n.get(s["task"], 0), L.stable_hash(s["qa_uid"])))
        p = ss[0]
        task_n[p["task"]] = task_n.get(p["task"], 0) + 1
        src_n[v.split("/")[0]] = src_n.get(v.split("/")[0], 0) + 1
        rank = len({r["video_key"] for r in rows}) + 1
        for k, s in enumerate(ss):
            role = "primary" if k == 0 else "supporting" if k <= supporting else "reserve"
            rows.append({"queue": "diagnostic", "rank": rank, "role": role, "export": rank <= target,
                         "qa_uid": s["qa_uid"], "video_key": v, "question_type": s["question_type"], "task": s["task"],
                         "automatic_status": s["automatic_status"], "official_score": s["official_score"],
                         "selection_reason": s.get("candidate_reason") or s["automatic_status"]})
    return rows


def build_queue(qa, scores, seed, audit_n, target, review_n=None, audit_required=False):
    """Audit + diagnostic queues. 'required' marks what must be final-reviewed for review_complete: the main
    question of the first review_n candidate videos in the fixed queue order (decided before any review, so the
    set cannot be cherry-picked), plus the audit questions only if audit_required."""
    by_uid = {q["qa_uid"]: q for q in qa}
    review_n = target if review_n is None else review_n
    rows = [{"queue": "audit", "rank": k + 1, "role": "audit", "export": False, "qa_uid": q["qa_uid"],
             "video_key": q["video_key"], "question_type": q["question_type"], "task": q["task"],
             "automatic_status": next((s["automatic_status"] for s in scores if s["qa_uid"] == q["qa_uid"]
                                       and s.get("condition") == "normal"), "missing"),
             "official_score": next((s["official_score"] for s in scores if s["qa_uid"] == q["qa_uid"]
                                     and s.get("condition") == "normal"), None),
             "selection_reason": f"seeded audit (seed {seed}); chosen from question ids only"}
            for k, q in enumerate(audit_queue(qa, seed, audit_n))]
    for r in rows:
        r["required"] = bool(audit_required)
    diag = diagnostic_queue(scores, by_uid, target)
    for r in diag:
        r["required"] = r["role"] == "primary" and r["rank"] <= review_n
    return rows + diag


# ----------------------------------------------------------------------------------------------- hashes + frames
def prediction_hash(pred_row):
    return L.stable_hash([pred_row.get("raw_answer"), pred_row.get("cache_key")], 16) if pred_row else None


def input_manifest_hash(meta_row):
    return L.stable_hash([meta_row.get("official_frame_indices"), meta_row.get("frame_sha1"),
                          meta_row.get("preprocessing")], 16) if meta_row else None


CITE_RE = re.compile(r"^(#|x|t=)?([0-9]+(?:\.[0-9]+)?)s?$", re.I)


def parse_citations(text, frame_meta, extra_index):
    """'#12' = model input position 12; 'x340' = extra (non-input) frame with source index 340; 't=41.2' = the model
    input frame at 41.2 s. Returns structured citations (input/extra, position, source index and seconds kept apart)."""
    out, errors = [], []
    idx = (frame_meta or {}).get("official_frame_indices") or []
    ts = (frame_meta or {}).get("timestamps_s") or []
    for tok in [t for t in re.split(r"[\s,;]+", text or "") if t]:
        m = CITE_RE.match(tok)
        if not m:
            errors.append(f"citation {tok!r}: use #<input position>, x<extra frame index> or t=<seconds>")
            continue
        kind, val = (m.group(1) or "#").lower(), m.group(2)
        if kind == "#":
            p = int(float(val))
            if not 0 <= p < len(idx):
                errors.append(f"#{p}: model inputs are #0-#{len(idx) - 1}")
                continue
            out.append({"frame_set": "input", "position": p, "source_frame_index": idx[p], "timestamp_s": ts[p] if ts else None})
        elif kind == "x":
            i = int(float(val))
            e = (extra_index or {}).get(i)
            if e is None:
                errors.append(f"x{i}: no extra frame with source index {i} has been extracted for this video")
                continue
            out.append({"frame_set": "extra", "position": None, "source_frame_index": i, "timestamp_s": e["timestamp_s"]})
        else:
            t = float(val)
            if not ts:
                errors.append("t=: input timestamps are unknown for this video")
                continue
            p = min(range(len(ts)), key=lambda k: abs(ts[k] - t))
            if abs(ts[p] - t) > 0.6:
                errors.append(f"t={t}: no model input frame within 0.6 s (nearest #{p} at {ts[p]} s)")
                continue
            out.append({"frame_set": "input", "position": p, "source_frame_index": idx[p], "timestamp_s": ts[p]})
    return out, errors


# ----------------------------------------------------------------------------------------------- review records
def current_refs(ctx, uid):
    """The current hashes a review of this question must carry."""
    q = ctx["qa_by_uid"].get(uid)
    pred = ctx["preds"].get(L.job_id(uid, "normal"))
    meta = ctx["metas"].get(q["video_key"]) if q else None
    return {"qa_hash": q["content_hash"] if q else None, "prediction_hash": prediction_hash(pred),
            "input_manifest_hash": input_manifest_hash(meta)}


def validate_review_row(row, ctx, source):
    """One validator for the notebook form and the CSV import. Returns (clean_row, errors)."""
    e = []
    uid = (row.get("qa_uid") or "").strip()
    q = ctx["qa_by_uid"].get(uid)
    if not q:
        return None, [f"qa_uid {uid!r} is not a question of this run's frozen sample"]
    if (row.get("video_key") or q["video_key"]) != q["video_key"]:
        e.append(f"video_key {row.get('video_key')!r} does not belong to {uid}")
    clean = {"qa_uid": uid, "video_key": q["video_key"], "source": source, "saved_at": now(),
             "reviewer": (row.get("reviewer") or "").strip(), "notes": (row.get("notes") or "").strip(),
             "proposed_answer": (row.get("proposed_answer") or "").strip(),
             "evidence_source": (row.get("evidence_source") or "").strip() or "model_inputs"}
    for f, allowed in (("annotation_status", ANNOTATION), ("input_evidence_status", EVIDENCE), ("model_verdict", VERDICT),
                       ("review_stage", STAGE)):
        v = (row.get(f) or ("unreviewed" if f != "review_stage" else "")).strip().lower()
        if v not in allowed:
            e.append(f"{f}: {v!r} is not one of {', '.join(allowed)}")
        clean[f] = v
    if clean["evidence_source"] not in EVIDENCE_SOURCE:
        e.append(f"evidence_source: {clean['evidence_source']!r} is not one of {', '.join(EVIDENCE_SOURCE)}")
    cites = row.get("citations")
    if isinstance(cites, str) or cites is None:
        cites, ce = parse_citations(cites or "", ctx["metas"].get(q["video_key"]), ctx["extra_index"].get(q["video_key"]))
        e += ce
    clean["citations"] = cites
    refs = current_refs(ctx, uid)
    for k, v in refs.items():
        given = (row.get(k) or v) if source == "notebook_form" else row.get(k)
        clean[k] = given
    if clean["review_stage"] == "final":
        if not clean["reviewer"]:
            e.append("reviewer: required for a final review")
        if not clean["notes"]:
            e.append("notes: required for a final review (what you saw, and why)")
        for f in ("annotation_status", "input_evidence_status", "model_verdict"):
            if clean.get(f) == "unreviewed":
                e.append(f"{f}: a final review cannot leave it 'unreviewed'")
        if clean.get("input_evidence_status") in ("sufficient", "partial") and not any(
                c["frame_set"] == "input" for c in clean["citations"]):
            e.append(f"input_evidence_status {clean.get('input_evidence_status')}: cite at least one model input frame (#n)")
        if clean.get("input_evidence_status") in ("absent", "uncertain") and not clean["notes"]:
            e.append("absent/uncertain evidence: describe the reviewed scope in notes")
        if clean.get("annotation_status") == "wrong" and not clean["proposed_answer"]:
            e.append("annotation_status wrong: give the proposed corrected answer (or 'unanswerable')")
    clean["review_id"] = L.stable_hash([clean["qa_uid"], clean["saved_at"], clean["source"], clean["notes"],
                                        clean["review_stage"]], 12)
    return clean, e


def append_review(out_root, clean):
    L.Journal(os.path.join(out_root, "reviews.jsonl")).append(clean)


def load_reviews(out_root):
    rows, _ = L.read_jsonl(os.path.join(out_root, "reviews.jsonl"))
    latest = {}
    for r in rows:                      # append-only history; the latest save per question is current
        if r.get("qa_uid") and (r["qa_uid"] not in latest or r["saved_at"] >= latest[r["qa_uid"]]["saved_at"]):
            latest[r["qa_uid"]] = r
    return rows, latest


def classify(uid, review, ctx):
    """Every flag for one question from its latest review (flags may coexist), plus verified_model_failure with
    reasons, from the one shared rule."""
    q = ctx["qa_by_uid"][uid]
    pred = ctx["preds"].get(L.job_id(uid, "normal"))
    score = ctx["scores_by_uid"].get(uid) or {}
    refs = current_refs(ctx, uid)
    r = review or {}
    stale = bool(review) and any(r.get(k) != v for k, v in refs.items())
    final = r.get("review_stage") == "final" and not stale
    case_dir = os.path.join(ctx["out_root"], "candidate_cases", video_dir_name(q["video_key"]))
    reasons = []
    if not review:
        reasons.append("not reviewed")
    elif stale:
        reasons.append("stale review: the question, prediction or model inputs changed after it was saved")
    elif r.get("review_stage") != "final":
        reasons.append("draft review")
    if not pred or pred.get("status") != "ok":
        reasons.append("no successful prediction")
    if final:
        if r.get("annotation_status") != "supported":
            reasons.append(f"reference not supported ({r.get('annotation_status')})")
        if r.get("input_evidence_status") != "sufficient":
            reasons.append(f"model-input evidence {r.get('input_evidence_status')}")
        if r.get("model_verdict") != "wrong":
            reasons.append(f"model verdict {r.get('model_verdict')}")
        cites = [c for c in r.get("citations") or [] if c.get("frame_set") == "input"]
        meta = ctx["metas"].get(q["video_key"]) or {}
        if not cites or any(not (0 <= c.get("position", -1) < len(meta.get("official_frame_indices") or [])) for c in cites):
            reasons.append("no valid model-input frame citation")
        if not (os.path.exists(os.path.join(case_dir, "video.mp4")) and os.path.isdir(os.path.join(case_dir, "input_frames"))):
            reasons.append("candidate media/frames not exported")
    return {"qa_uid": uid, "video_key": q["video_key"], "reviewed": bool(review), "final": final, "stale": stale,
            "draft": bool(review) and not stale and r.get("review_stage") != "final",
            "verified_model_failure": final and not reasons, "reasons": reasons,
            "benchmark_issue": final and r.get("annotation_status") in ("wrong", "ambiguous"),
            "insufficient_evidence": final and r.get("input_evidence_status") in ("partial", "absent", "uncertain"),
            "uncertain": final and r.get("model_verdict") == "uncertain",
            "automatic_status": score.get("automatic_status")}


def classify_all(queue_rows, latest, ctx):
    uids = sorted({r["qa_uid"] for r in queue_rows} | set(latest))
    return {u: classify(u, latest.get(u), ctx) for u in uids if u in ctx["qa_by_uid"]}


# ----------------------------------------------------------------------------------------------- export
def export_candidate(video_key, ctx, packaged_uids):
    """candidate_cases/<video>/: video.mp4 (the exact model input clip), source_video.mp4 (full processed video if
    available; NOT a model input), input_frames/ (the worker's saved inputs), contact_sheet.jpg, case.json."""
    import revsi_frames as F
    d = os.path.join(ctx["out_root"], "candidate_cases", video_dir_name(video_key))
    os.makedirs(os.path.join(d, "input_frames"), exist_ok=True)
    problems = []
    meta = ctx["metas"].get(video_key)
    if not meta:
        return ["no frame provenance (video_meta) for this video: inference has not run on it"]
    try:
        src = ctx["fetch_input"](video_key)
        if not os.path.exists(os.path.join(d, "video.mp4")):
            shutil.copyfile(src, os.path.join(d, "video.mp4.part"))
            os.replace(os.path.join(d, "video.mp4.part"), os.path.join(d, "video.mp4"))
    except Exception as exc:
        problems.append(f"input video unavailable: {type(exc).__name__}: {exc}")
    full = ctx.get("fetch_source") and ctx["fetch_source"](video_key)
    if full and not os.path.exists(os.path.join(d, "source_video.mp4")):
        shutil.copyfile(full, os.path.join(d, "source_video.mp4"))
    fsrc = os.path.join(ctx["frames_root"], meta["safe_key"])
    files = meta.get("evidence_files") or []
    for f in files:
        if os.path.exists(os.path.join(fsrc, f)) and not os.path.exists(os.path.join(d, "input_frames", f)):
            shutil.copyfile(os.path.join(fsrc, f), os.path.join(d, "input_frames", f))
    have = [f for f in files if os.path.exists(os.path.join(d, "input_frames", f))]
    if len(have) != len(meta.get("official_frame_indices") or []):
        problems.append(f"{len(have)} of {len(meta.get('official_frame_indices') or [])} input frames available")
    if have:
        from PIL import Image
        F.contact_sheet([Image.open(os.path.join(d, "input_frames", f)) for f in have],
                        [f"#{k} src{meta['official_frame_indices'][k]} {meta['timestamps_s'][k]:.1f}s" for k in range(len(have))],
                        os.path.join(d, "contact_sheet.jpg"))
    qa = []
    for uid in packaged_uids:
        q = ctx["qa_by_uid"][uid]
        s = ctx["scores_by_uid"].get(uid) or {}
        qa.append({"qa_uid": uid, "native_id": q["native_id"], "config": q["config"], "question_type": q["question_type"],
                   "question": q["question"], "options": q["options"], "reference_answer": q["reference_answer"],
                   "prediction": s.get("raw_answer"), "official_parse": s.get("parsed_answer"),
                   "official_score": s.get("official_score"), "automatic_status": s.get("automatic_status"),
                   "release_visibility_flag": q.get("visibility")})
    L.write_json_atomic(os.path.join(d, "case.json"), {
        "benchmark": L.BENCHMARK, "revisions": ctx["revisions"], "config": ctx["config"], "video_key": video_key,
        "source_dataset": video_key.split("/")[0], "scene_id": video_key.split("/", 1)[1], "packaged_qa": qa,
        "frame_provenance": {k: meta.get(k) for k in ("positions", "official_frame_indices", "timestamps_s", "frame_sha1",
                                                         "reader", "preprocessing", "input_frame_count")},
        "frame_verification": ctx["verification"].get(video_key), "exported_at": now(),
        "files": {"video.mp4": "exact model input clip", "source_video.mp4": "full processed video (not a model input)",
                  "input_frames/": "exact decoded model inputs (JPEG copies)", "contact_sheet.jpg": "grid of the model inputs"}})
    return problems


def export_final(ctx, cls, latest):
    """verified_failures/ and benchmark_issues/ (distinct videos), with manifests; recomputed from saved reviews."""
    out = {}
    for kind, flag in (("verified_failures", "verified_model_failure"), ("benchmark_issues", "benchmark_issue")):
        root = os.path.join(ctx["out_root"], kind)
        os.makedirs(root, exist_ok=True)
        vids = {}
        for u, c in cls.items():
            if c[flag]:
                vids.setdefault(c["video_key"], []).append(u)
        keep = {video_dir_name(v) for v in vids}
        for f in os.listdir(root):
            if f not in keep:
                shutil.rmtree(os.path.join(root, f), ignore_errors=True)
        cases = []
        for v, uids in sorted(vids.items()):
            src = os.path.join(ctx["out_root"], "candidate_cases", video_dir_name(v))
            dst = os.path.join(root, video_dir_name(v))
            if os.path.isdir(src):
                shutil.rmtree(dst, ignore_errors=True)
                shutil.copytree(src, dst)
            L.write_json_atomic(os.path.join(dst, "reviews.json"), {u: latest.get(u) for u in uids})
            paths = {k: os.path.join(kind, video_dir_name(v), k) for k in ("video.mp4", "case.json", "contact_sheet.jpg")}
            missing = [k for k, p in paths.items() if not os.path.exists(os.path.join(ctx["out_root"], p))]
            cases.append({"video_key": v, "qa_uids": sorted(uids), "paths": paths, "missing_files": missing})
        man = {"kind": kind, "distinct_videos": len(cases), "cases": cases, "generated": now(),
               "rule": "verified_model_failure: final, current review; reference supported; model-input evidence "
                       "sufficient; model wrong; valid input-frame citation; exported media" if flag == "verified_model_failure"
               else "benchmark_issue: final, current review with annotation_status wrong or ambiguous (not a model failure)"}
        L.write_json_atomic(os.path.join(ctx["out_root"], "verified_failure_manifest.json" if kind == "verified_failures"
                                         else "benchmark_issue_manifest.json"), man)
        out[kind] = man
    return out


# ----------------------------------------------------------------------------------------------- counts + messages
def counts(ctx, queue_rows, cls, coverage_normal, min_v, target_v):
    exported = [r for r in queue_rows if r["queue"] == "diagnostic" and r["export"] and r["role"] == "primary"]
    exp_dirs = [v for v in {r["video_key"] for r in exported}
                if os.path.exists(os.path.join(ctx["out_root"], "candidate_cases", video_dir_name(v), "case.json"))]
    vids = lambda flag: len({c["video_key"] for c in cls.values() if c[flag]})
    queued = {r["qa_uid"] for r in queue_rows if r.get("required", r["role"] in ("primary", "audit"))}
    c = {"sample_videos": ctx["sample_videos"], "sample_questions": ctx["sample_questions"],
         "candidate_videos": len({r["video_key"] for r in queue_rows if r["queue"] == "diagnostic"}),
         "candidate_exports": len(exp_dirs),
         "final_reviewed_videos": vids("final"), "verified_model_failure_videos": vids("verified_model_failure"),
         "annotation_issue_videos": vids("benchmark_issue"), "insufficient_evidence_videos": vids("insufficient_evidence"),
         "uncertain_videos": vids("uncertain"), "stale_reviews": sum(c_["stale"] for c_ in cls.values()),
         "draft_reviews": sum(c_["draft"] for c_ in cls.values()), "missing_media_videos": len(ctx.get("blocked", {})),
         "required_reviews": len(queued), "required_reviews_final": sum(1 for u in queued if cls.get(u, {}).get("final")),
         "optional_reviews_final": sum(1 for u, c_ in cls.items() if c_["final"] and u not in queued),
         "minimum": min_v, "target": target_v}
    # complete = no pending and no failed runnable job; media-blocked videos stay visible as their own count
    c["inference_complete"] = bool(coverage_normal) and coverage_normal["pending"] == 0 and coverage_normal["failed"] == 0
    c["media_blocked_questions"] = (coverage_normal or {}).get("blocked_media", 0)
    c["candidate_export_complete"] = c["candidate_exports"] >= min(target_v, len({r["video_key"] for r in exported}))
    c["review_complete"] = bool(queued) and c["required_reviews_final"] == len(queued)
    c["verified_failure_minimum_met"] = c["verified_model_failure_videos"] >= min_v
    c["verified_failure_target_met"] = c["verified_model_failure_videos"] >= target_v
    return c


def status_message(c, coverage_normal, empty_reasons=None):
    if not c["inference_complete"]:
        cn = coverage_normal or {}
        msg = (f"Inference is not complete: {cn.get('succeeded', 0)}/{cn.get('expected', 0)} answered, "
               f"{cn.get('pending', 0)} pending, {cn.get('failed', 0)} failed, {cn.get('blocked_media', 0)} blocked by media.")
        if not c["candidate_videos"]:
            return msg + " " + (empty_reasons or "")
        return msg + f" {c['candidate_exports']} candidate videos exported from the part answered so far."
    if c["verified_model_failure_videos"] == 0 and c["final_reviewed_videos"] == 0:
        if not c["candidate_videos"]:
            return "No candidate videos. " + (empty_reasons or "")
        return (f"{c['candidate_exports']} candidate videos exported; {c['final_reviewed_videos']} reviewed; verified "
                "failures remain unestablished. Import or complete reviews, then regenerate reports. Inference need not "
                "be rerun.")
    progress = (f" Required reviews: {c.get('required_reviews_final', 0)}/{c.get('required_reviews', 0)} final"
                + (" (complete)." if c.get("review_complete") else "."))
    tail = progress + (" Minimum met." if c["verified_failure_minimum_met"] else
            f" Minimum of {c['minimum']} not met: the frozen sample yields {c['verified_model_failure_videos']} so far; "
            "it is not expanded.")
    return (f"{c['verified_model_failure_videos']} verified model-failure video(s), {c['annotation_issue_videos']} with "
            f"benchmark issues, {c['insufficient_evidence_videos']} with insufficient input evidence, {c['uncertain_videos']} "
            f"uncertain; {c['final_reviewed_videos']} video(s) with final reviews ({c['candidate_videos']} candidate "
            f"videos).{tail}" + ("" if c["candidate_videos"] else " " + (empty_reasons or "")))


def empty_queue_reasons(scores_normal, coverage_normal, gather_report):
    st = {}
    for s in scores_normal:
        st[s["automatic_status"]] = st.get(s["automatic_status"], 0) + 1
    parts = []
    cn = coverage_normal or {}
    if cn.get("pending") or cn.get("failed"):
        parts.append(f"missing predictions: {cn.get('pending', 0)} pending, {cn.get('failed', 0)} failed (run section D)")
    if (gather_report or {}).get("incompatible_cached_rows"):
        parts.append(f"{gather_report['incompatible_cached_rows']} cached rows ignored because their cache key changed")
    if st.get("format_error"):
        parts.append(f"{st['format_error']} parse failures")
    if st.get("imperfect") and not st.get("wrong_candidate"):
        parts.append(f"only partial-credit numeric answers ({st['imperfect']}); they are graded scores, not candidates")
    if not parts:
        parts.append(f"genuinely no qualifying candidates among {sum(st.values())} scored answers ({st})")
    return "Why: " + "; ".join(parts) + "."


# ----------------------------------------------------------------------------------------------- tables + reports
def write_template(path, queue_rows, ctx):
    rows = []
    for r in queue_rows:
        if r["role"] not in ("primary", "audit", "supporting"):
            continue
        q = ctx["qa_by_uid"][r["qa_uid"]]
        s = ctx["scores_by_uid"].get(r["qa_uid"]) or {}
        refs = current_refs(ctx, r["qa_uid"])
        rows.append(dict({"required": "yes" if r.get("required") else "optional", "queue": r["queue"], "rank": r["rank"], "qa_uid": r["qa_uid"], "video_key": r["video_key"],
                          "question_type": q["question_type"], "question": q["question"],
                          "options": " | ".join(q["options"] or []), "annotation_status": "unreviewed",
                          "input_evidence_status": "unreviewed", "model_verdict": "unreviewed", "review_stage": "draft",
                          "reviewer": "", "notes": "", "citations": "", "evidence_source": "model_inputs",
                          "proposed_answer": "", "reference_answer": q["reference_answer"],
                          "model_answer": s.get("raw_answer"), "official_score": s.get("official_score")}, **refs))
    with open(path + ".tmp", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=TEMPLATE_COLS)
        w.writeheader()
        w.writerows(rows)
    os.replace(path + ".tmp", path)
    return len(rows)


def import_reviews(csv_path, ctx):
    """Validate an edited review CSV with the same validator as the form; append accepted rows, report rejections."""
    accepted, rejected = [], []
    with open(csv_path, newline="") as f:
        for n, row in enumerate(csv.DictReader(f), 2):
            if all((row.get(k) or "").strip() in ("", "unreviewed", "draft") for k in
                   ("annotation_status", "input_evidence_status", "model_verdict", "notes", "citations")) and \
                    (row.get("review_stage") or "draft") == "draft":
                continue                                # untouched template row
            clean, errs = validate_review_row(row, ctx, "csv_import")
            if errs:
                rejected.append({"line": n, "qa_uid": row.get("qa_uid"), "reasons": "; ".join(errs)})
            else:
                append_review(ctx["out_root"], clean)
                accepted.append({"line": n, "qa_uid": clean["qa_uid"], "review_stage": clean["review_stage"]})
    with open(os.path.join(ctx["out_root"], "review_import_errors.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["line", "qa_uid", "reasons"])
        w.writeheader()
        w.writerows(rejected)
    return accepted, rejected


def write_summary(path, queue_rows, cls, latest, ctx):
    cols = ["queue", "rank", "role", "qa_uid", "video_key", "question_type", "automatic_status", "official_score",
            "annotation_status", "input_evidence_status", "model_verdict", "review_stage", "verified_model_failure",
            "benchmark_issue", "insufficient_evidence", "uncertain", "stale", "reasons"]
    with open(path + ".tmp", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader()
        for r in queue_rows:
            c = cls.get(r["qa_uid"], {})
            rv = latest.get(r["qa_uid"]) or {}
            w.writerow({**{k: r.get(k) for k in ("queue", "rank", "role", "qa_uid", "video_key", "question_type",
                                                 "automatic_status", "official_score")},
                        **{k: rv.get(k, "unreviewed") for k in ("annotation_status", "input_evidence_status", "model_verdict")},
                        "review_stage": rv.get("review_stage", ""), "verified_model_failure": c.get("verified_model_failure"),
                        "benchmark_issue": c.get("benchmark_issue"), "insufficient_evidence": c.get("insufficient_evidence"),
                        "uncertain": c.get("uncertain"), "stale": c.get("stale"), "reasons": "; ".join(c.get("reasons", []))})
    os.replace(path + ".tmp", path)


def _pct(v):
    return "n/a" if v is None else f"{100 * v:.1f}"


def report(ctx, c, status, metrics, controls_summary, baseline, historical, overlap_summary, cls, latest, published):
    """report.md and report.html from one structure; links are relative to OUTPUT_ROOT."""
    md = [f"# Cambrian-P on ReVSI ({ctx['config']}): validity audit of a {ctx['fraction']:.0%} video sample", "",
          f"**Status:** {status}", "",
          "## 1. Published claims (reported by the ReVSI authors; not measured here)", ""]
    md += [f"- {p}" for p in published] + ["", "## 2. This sample's measurements", "",
           f"Sample: {c['sample_videos']} of {ctx['population_videos']} unique videos (seed {ctx['seed']}); "
           f"{c['sample_questions']} questions (all released questions of those videos; 10% of videos is not 10% of "
           "questions). Frame protocol: the released 32-frame videos, every frame once (Cambrian-P natively uses 128 "
           "frames; this is a controlled frame-budget experiment, not a reproduction of its 128-frame results).", ""]
    for name, m in metrics.items():
        agg = m.get("official_subset_macro") or {}
        ci = (m.get("bootstrap_scene_cluster") or {}).get("overall_acc", {}).get("ci95")
        md.append(f"### {name}")
        md.append(f"Scored questions: {m['scored_questions']} on {m['scored_videos']} videos; excluded {m['excluded']}.")
        md.append(f"Official macro overall (subset): {_pct(agg.get('overall_acc'))}"
                  + (f" (scene-cluster bootstrap 95% CI {_pct(ci[0])}–{_pct(ci[1])})" if ci else "")
                  + f"; QA-weighted mean (labelled, not official): {_pct(m.get('qa_weighted_mean'))}.")
        md += [f"- {k}: {_pct(v)} (n={m['per_task_scored'].get(k.replace('_acc', ''), '?')})" for k, v in agg.items() if k != "overall_acc"]
        md.append("")
    md += ["## 3. Dependence on visual evidence", "", controls_summary, "",
           f"Scene-disjoint frequency baseline (local diagnostic, fit on other scene folds only): official macro "
           f"{_pct((baseline or {}).get('overall_acc'))}.", "",
           "## 4. Historical VSI-Bench cohort and overlap (descriptive, confounded)", "", historical, "", overlap_summary, "",
           "## 5. Reviewed examples", "",
           "| video | question | flags | review |", "|---|---|---|---|"]
    for u, cl in sorted(cls.items()):
        if not cl["reviewed"]:
            continue
        r = latest.get(u, {})
        flags = ", ".join(k for k in ("verified_model_failure", "benchmark_issue", "insufficient_evidence", "uncertain",
                                      "stale", "draft") if cl.get(k)) or "none"
        md.append(f"| {cl['video_key']} | {u} | {flags} | {r.get('annotation_status')}/{r.get('input_evidence_status')}/"
                  f"{r.get('model_verdict')} ({r.get('review_stage')}) |")
    md += ["", "## 6. Counts and completion", ""] + [f"- {k}: {v}" for k, v in c.items()] + [
        "", "## 7. Remaining uncertainty", "",
        "- Disagreement between the model, ReVSI and VSI-Bench is disagreement until a final human review says otherwise; "
        "ReVSI is a curated alternative reference, not unquestionable truth.",
        "- Black-frame retention suggests a strong prior or shortcut; it does not prove memorization or contamination.",
        "- The ReVSI paper's Cambrian experiments concern Cambrian-S, not Cambrian-P.",
        "- Metric answers (sizes, distances, areas) cannot be verified by eye from RGB frames alone.",
        "- A score difference between benchmarks does not establish training contamination or overfitting.", "",
        f"Run folder: {ctx['out_root']}", "",
        "Recovery: mount Drive with 1a, run the setup and load cells (1b, 1c, B), then D (only missing jobs run), "
        "E, F, G and H; in review_only mode skip installs and the model."]
    text = "\n".join(md)
    open(os.path.join(ctx["out_root"], "report.md"), "w").write(text)
    body = []
    for line in md:
        if line.startswith("# "):
            body.append(f"<h1>{html.escape(line[2:])}</h1>")
        elif line.startswith("## "):
            body.append(f"<h2>{html.escape(line[3:])}</h2>")
        elif line.startswith("### "):
            body.append(f"<h3>{html.escape(line[4:])}</h3>")
        elif line.startswith("- "):
            body.append(f"<li>{html.escape(line[2:])}</li>")
        elif line.startswith("|"):
            if set(line) <= set("|-"):
                continue
            body.append("<tr>" + "".join(f"<td>{html.escape(x.strip())}</td>" for x in line.strip("|").split("|")) + "</tr>")
        elif line:
            body.append(f"<p>{html.escape(line)}</p>")
    cases = []
    for kind in ("verified_failures", "benchmark_issues"):
        man_path = os.path.join(ctx["out_root"], "verified_failure_manifest.json" if kind == "verified_failures"
                                else "benchmark_issue_manifest.json")
        for case in (json.load(open(man_path))["cases"] if os.path.exists(man_path) else []):
            p = case["paths"]
            cases.append(f"<h3>{html.escape(kind)}: {html.escape(case['video_key'])}</h3>"
                         f"<video controls width='480' src='{html.escape(p['video.mp4'])}'></video>"
                         f"<p><img width='720' src='{html.escape(p['contact_sheet.jpg'])}'></p>"
                         f"<p><a href='{html.escape(p['case.json'])}'>case.json</a></p>")
    page = ("<!doctype html><meta charset='utf-8'><title>ReVSI audit</title><style>body{font:15px system-ui;max-width:"
            "1000px;margin:24px auto;padding:0 16px}td{border:1px solid #ccc;padding:2px 6px}</style>"
            + "".join(body).replace("</td></tr><tr>", "</td></tr>\n<tr>") + "".join(cases)
            + "<p><i>Download the folder (or the zip) to view videos locally; Drive shows HTML as text.</i></p>")
    open(os.path.join(ctx["out_root"], "report.html"), "w").write(page)
    return text


# ----------------------------------------------------------------------------------------------- state from disk
def load_context(out_root, official, journal_path=None, fetch_input=None, fetch_source=None):
    """Everything the review, export and report steps need, rebuilt from files in OUTPUT_ROOT only (review-only
    sessions: no model, no in-memory state)."""
    import revsi_pipeline as P
    j = lambda *p: os.path.join(out_root, *p)
    rc = json.load(open(j("run_config.json")))
    sm = json.load(open(j("sample_manifest.json")))
    qa, _ = L.read_jsonl(j("qa_manifest.jsonl"), tolerate_truncated_tail=False)
    frames = {r["video_key"]: r for r in L.read_jsonl(j("frame_manifest.jsonl"))[0]}
    control_uids = set(json.load(open(j("control_subset.json")))) if os.path.exists(j("control_subset.json")) else set()
    run_controls = bool(rc.get("settings", {}).get("RUN_VISUAL_CONTROLS", True))
    jobs = P.build_jobs(qa, [q for q in qa if q["qa_uid"] in control_uids],
                        {v: f["frame_manifest_hash"] for v, f in frames.items()}, rc["config_hash"], run_controls)
    ok, failed, meta, report = P.gather(j("predictions"), journal_path, rc["config_hash"], jobs)
    blocked = {r["video_key"]: r.get("reason") for r in L.read_jsonl(j("logs", "media_missing.jsonl"))[0]}
    scores = P.score_jobs(qa, jobs, ok, failed, blocked, official)
    extra_index = {}
    for v in {q["video_key"] for q in qa}:
        p = os.path.join(out_root, "candidate_cases", video_dir_name(v), "extra_frames", "index.json")
        if os.path.exists(p):
            extra_index[v] = {e["frame_index"]: e for e in json.load(open(p))["files"]}
    metas = {}
    for (v, cond), mrow in meta.items():
        if cond == "normal":
            mrow = dict(mrow)
            if frames.get(v, {}).get("timestamps_s"):
                mrow["timestamps_s"] = frames[v]["timestamps_s"]
            metas[v] = mrow
    return {"out_root": out_root, "run_config": rc, "config": sm["config"], "revisions": rc.get("revisions", {}),
            "qa": qa, "qa_by_uid": {q["qa_uid"]: q for q in qa}, "jobs": jobs, "preds": ok, "failed": failed,
            "gather_report": report, "metas": metas, "meta_all": meta, "blocked": blocked, "scores": scores,
            "scores_by_uid": {s["qa_uid"]: s for s in scores if s["condition"] == "normal"},
            "coverage": P.coverage_counts(jobs, ok, failed, blocked), "extra_index": extra_index,
            "verification": {v: f.get("verification") for v, f in frames.items()},
            "frames_root": j("input_frames"), "sample_videos": sm["selected_videos"],
            "sample_questions": sm["selected_questions"], "population_videos": sm["population_videos"],
            "seed": sm["seed"], "fraction": sm["fraction"], "control_uids": control_uids,
            "fetch_input": fetch_input or (lambda v: os.path.join(out_root, "candidate_cases", video_dir_name(v), "video.mp4")),
            "fetch_source": fetch_source}

### Helper code: `revsi_worker.py`

In [ ]:
%%writefile /content/revsi/revsi_worker.py
"""Cambrian-P inference worker for the ReVSI audit. Runs inside the Cambrian-P venv.

Loads the model once, then for each video in the batch file decodes the sampled frames once and answers that
video's selected questions with the official VSI-Bench prompt (built in the notebook, stored in qa_manifest). Every answer is appended (fsync) to a local journal at once; after each video its rows
are written to a durable shard with a completion marker. Model loading and preprocessing follow the lmms-eval cambrian_p wrapper;
the question text is the official vsibench_doc_to_text output.
"""
import argparse
import json
import os
import sys
import time
import traceback

os.environ["MPLBACKEND"] = "Agg"   # Colab's inline backend leaks in through the environment; this script saves files only
import numpy as np
import torch
from PIL import Image

ap = argparse.ArgumentParser()
ap.add_argument("--batch", required=True, help="JSON: {videos: [{video_key, safe_key, local_path, official_frame_indices, timestamps_s, jobs: [...]}]}")
ap.add_argument("--frames_dir", required=True, help="where the exact model input frames are saved as evidence")
ap.add_argument("--model_path", required=True)
ap.add_argument("--num_frames", type=int, required=True)
ap.add_argument("--config_hash", required=True)
ap.add_argument("--identity", required=True, help="JSON of the run identity, copied into every row")
ap.add_argument("--journal", required=True)
ap.add_argument("--errors", required=True)
ap.add_argument("--shard_dir", required=True)
ap.add_argument("--status", required=True)
ap.add_argument("--lib_dir", required=True)
ap.add_argument("--max_retries", type=int, default=2)
ap.add_argument("--stop_after", type=int, default=0, help="test only: exit after this many new answers")
args = ap.parse_args()
sys.path.insert(0, args.lib_dir)
import revsi_lib as L  # noqa: E402
import hashlib  # noqa: E402

identity = json.loads(args.identity)
GEN = dict(identity["GENERATION"])
journal, errors = L.Journal(args.journal), L.Journal(args.errors)
status = {"started": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "answers": 0, "errors": 0, "videos": 0}


def save_status(**kw):
    status.update(kw)
    L.write_json_atomic(args.status, status)


# ---------------------------------------------------------------------------------------- model (from run_videos.py)
import cambrianp.utils as cambrianp_utils  # noqa: E402
from lmms_eval.models.simple.cambrian_p import CambrianP, expand2square  # noqa: E402
from cambrianp.constants import DEFAULT_IMAGE_TOKEN, IMAGE_TOKEN_INDEX  # noqa: E402
from cambrianp.conversation import conv_templates  # noqa: E402
from cambrianp.mm_utils import tokenizer_image_token  # noqa: E402


# Workaround kept from the source notebook, re-checked against the pinned code (b3c1552) and checkpoint:
# cambrianp/model/builder.py:230 still re-casts the config of any path containing "cambrian" to LLaVA-OneVision
# style, which overwrites vocab_size and deletes the camera-head keys; and the checkpoint's rec_model_path is
# JSON null, which VGGTReconstructor would try to torch.load. Keep the saved config, enable the camera head.
def _keep_checkpoint_config(config, model_path):
    config.load_rec_model = True
    if not os.path.isfile(str(getattr(config, "rec_model_path", None))):
        config.rec_model_path = "None"
    return config


cambrianp_utils.cast_cambrian_config_to_llava_ov_style = _keep_checkpoint_config
save_status(phase="loading model")
m = CambrianP(pretrained=args.model_path, model_name="llava_qwen", conv_template="qwen_1_5",
              max_frames_num=args.num_frames, use_camera_tokens=True, camera_tokens_mode="camera_tokens",
              camera_tokens_place="append_to_frame", query_mode="query_after_image")
pad_id = m.tokenizer.pad_token_id if m.tokenizer.pad_token_id is not None else m.tokenizer.eos_token_id

# Check the camera tokens and trained head really loaded (the failure mode the workaround exists for).
inner = m.model.get_model()
rec_head = getattr(inner, "rec_head", None)
cam = getattr(rec_head, "camera_tokens", None) if rec_head is not None else None
head_ok = rec_head is not None and cam is not None and float(cam.detach().float().abs().sum()) > 0
save_status(phase="model loaded", camera_head_loaded=bool(head_ok),
            camera_tokens_shape=list(cam.shape) if cam is not None else None,
            vocab_size=int(m.model.config.vocab_size), gpu=torch.cuda.get_device_name(0),
            torch=torch.__version__, cuda=torch.version.cuda)
if not head_ok:
    raise SystemExit("Camera tokens / trained head did not load: refusing to run (see the model log above).")


def ask(pixels, image_size, video, question):
    conv = conv_templates["qwen_1_5"].copy()
    conv.append_message(conv.roles[0], DEFAULT_IMAGE_TOKEN + "\n" + question)
    conv.append_message(conv.roles[1], None)
    rendered = conv.get_prompt()
    ids = tokenizer_image_token(rendered, m.tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt").unsqueeze(0).to(m.device)
    return m._run_inference_step(ids, ids.ne(pad_id), [pixels], image_size, "video", dict(GEN), video)[0], rendered


# ---------------------------------------------------------------------------------------- frames with provenance
def decode_exact(path, k):
    """Decode the protocol's fixed-budget video. It must hold exactly k frames: the model then sees every frame
    once (no second resampling); anything else is a protocol violation, never silently resampled."""
    from decord import VideoReader, cpu
    vr = VideoReader(path, ctx=cpu(0), num_threads=0)
    total = len(vr)
    if total != k:
        raise ValueError(f"protocol violation: input video has {total} frames, the protocol needs exactly {k}")
    idx = L.sample_indices(total, k)
    if idx != list(range(k)):
        raise ValueError("protocol violation: the wrapper sampler would not take every frame exactly once")
    return vr.get_batch(idx).asnumpy(), idx, "decord"


def preprocess(raw):
    bg = tuple(int(x * 255) for x in m._image_processor.image_mean)
    frames = [expand2square(Image.fromarray(f, mode="RGB"), bg) for f in raw]
    pixels = m._image_processor.preprocess(frames, return_tensors="pt")["pixel_values"].half().cuda()
    desc = (f"pad to square with mean color {bg} (expand2square), then the SigLIP image processor "
            f"(resize to {tuple(pixels.shape[-2:])}, normalize); fp16")
    return pixels, desc


def sha1s(frames):
    return [hashlib.sha1(np.ascontiguousarray(f).tobytes()).hexdigest() for f in frames]


# ---------------------------------------------------------------------------------------- main loop
batch = json.load(open(args.batch))
answered = 0
for vk, video in enumerate(batch["videos"], 1):
    rows = []
    save_status(phase=f"video {vk}/{len(batch['videos'])}", current=video["video_key"])
    t0 = time.time()
    try:
        raw, positions, reader = decode_exact(video["local_path"], args.num_frames)
    except Exception:
        err = traceback.format_exc().strip().splitlines()[-1]
        for j in video["jobs"]:
            e = {"kind": "answer", "job_id": j["job_id"], "qa_uid": j["qa_uid"], "condition": j["condition"],
                 "cache_key": j["cache_key"], "video_key": video["video_key"], "status": "error",
                 "error": f"decode: {err}", "attempt": 1, "config_hash": args.config_hash,
                 "answered_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
            errors.append(e); rows.append(e)
        L.write_shard(args.shard_dir, video["safe_key"] + f"-{time.time_ns()}", rows, args.config_hash)
        status["errors"] += len(video["jobs"])
        continue
    # the exact model inputs, saved as evidence (normal condition)
    fdir = os.path.join(args.frames_dir, video["safe_key"])
    os.makedirs(fdir, exist_ok=True)
    files = []
    for p, f in zip(positions, raw):
        name = (f"in{p:02d}_src{video['official_frame_indices'][p]:06d}_t{video['timestamps_s'][p]:09.3f}s.jpg")
        Image.fromarray(f).save(os.path.join(fdir, name), quality=92)
        files.append(name)
    conditions = {}
    for cond in sorted({j["condition"] for j in video["jobs"]}):
        frames = raw if cond == "normal" else np.zeros_like(raw)   # black frames: same count, size and timestamps
        pixels, prep = preprocess(frames)
        meta = {"kind": "video_meta", "video_key": video["video_key"], "safe_key": video["safe_key"], "condition": cond,
                "config_hash": args.config_hash, "positions": positions,
                "official_frame_indices": video["official_frame_indices"], "timestamps_s": video["timestamps_s"],
                "input_frame_count": len(frames), "reader": reader, "frame_size_hw": list(frames.shape[1:3]),
                "frame_sha1": sha1s(frames), "preprocessing": prep, "evidence_files": files if cond == "normal" else [],
                "control_construction": None if cond == "normal" else
                "every decoded input frame replaced by an all-zero RGB frame of the same size before preprocessing; "
                "camera tokens and geometry are computed by the model from these images (no real-image features kept)",
                "decode_seconds": round(time.time() - t0, 2)}
        rows.append(meta)
        journal.append(meta)
        conditions[cond] = (pixels, list(frames.shape[1:3]))
    m.model.rec_head_preds = None          # poses are cached per generate(); cleared per video
    for j in video["jobs"]:
        pixels, image_size = conditions[j["condition"]]
        m.model.rec_head_preds = None
        for attempt in range(1, args.max_retries + 2):
            t1 = time.time()
            # Only the model call is caught and retried; a bug in saving stops the worker with its traceback.
            try:
                answer, rendered = ask(pixels, image_size, video["local_path"], j["prompt"])
            except torch.cuda.OutOfMemoryError:
                e = {"kind": "answer", "job_id": j["job_id"], "qa_uid": j["qa_uid"], "condition": j["condition"],
                     "cache_key": j["cache_key"], "video_key": video["video_key"], "status": "error",
                     "error": "CUDA out of memory", "attempt": attempt, "config_hash": args.config_hash,
                     "answered_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
                errors.append(e); rows.append(e)
                L.write_shard(args.shard_dir, video["safe_key"] + f"-{time.time_ns()}", rows, args.config_hash)
                save_status(phase="stopped: out of GPU memory")
                raise SystemExit(f"Out of GPU memory at {args.num_frames} frames on {video['video_key']}. Frames, model "
                                 "and camera tokens are never changed automatically: use a GPU with more memory, or "
                                 "start a separately identified experiment (new RUN_ID) with different settings.")
            except Exception:
                last_err = traceback.format_exc().strip().splitlines()[-1]
                e = {"kind": "answer", "job_id": j["job_id"], "qa_uid": j["qa_uid"], "condition": j["condition"],
                     "cache_key": j["cache_key"], "video_key": video["video_key"], "status": "error",
                     "error": last_err, "attempt": attempt, "config_hash": args.config_hash,
                     "answered_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
                errors.append(e)
                if attempt == args.max_retries + 1:
                    rows.append(e)
                    status["errors"] += 1
                continue
            r = {"kind": "answer", "job_id": j["job_id"], "qa_uid": j["qa_uid"], "native_id": j["native_id"],
                 "condition": j["condition"], "cache_key": j["cache_key"], "frame_manifest_hash": j["frame_manifest_hash"],
                 "video_key": video["video_key"], "status": "ok", "raw_answer": answer, "rendered_prompt": rendered,
                 "attempt": attempt, "elapsed_s": round(time.time() - t1, 2), "config_hash": args.config_hash,
                 "answered_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
            journal.append(r); rows.append(r)
            answered += 1
            status["answers"] += 1
            break
        if args.stop_after and answered >= args.stop_after:
            save_status(phase=f"stopped on purpose after {answered} answers (resume test)")
            sys.exit(75)
    L.write_shard(args.shard_dir, video["safe_key"] + f"-{time.time_ns()}", rows, args.config_hash)
    status["videos"] += 1
    torch.cuda.empty_cache()
_peak = getattr(torch.cuda, "max_memory_allocated", None)
save_status(phase="done", finished=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            peak_vram_gb=round(_peak() / 1e9, 2) if _peak else None)

### Helper code: `revsi_selftest.py`

In [ ]:
%%writefile /content/revsi/revsi_selftest.py
"""Acceptance checks for the ReVSI audit (spec section 5, checks 3-12) on fixtures with real tiny videos.
No GPU, no dataset download. Exits nonzero on any failure.

  python revsi_selftest.py --official_utils <utils.py> --official_yaml <_default_template_yaml>
                           [--revsi_parquet <32_frame parquet> --frame_idx <json> --visibility <json>]
"""
import argparse
import copy
import csv
import json
import os
import random
import shutil
import subprocess
import sys
import tempfile

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
import numpy as np  # noqa: E402
import revsi_lib as L  # noqa: E402
import revsi_pipeline as P  # noqa: E402
import revsi_review as R  # noqa: E402
import revsi_frames as F  # noqa: E402

ap = argparse.ArgumentParser()
ap.add_argument("--official_utils", required=True)
ap.add_argument("--official_yaml", required=True)
ap.add_argument("--revsi_parquet")
ap.add_argument("--frame_idx")
ap.add_argument("--visibility")
args = ap.parse_args()
O = L.load_official(args.official_utils, args.official_yaml)
RESULTS = []


def check(name, cond, detail=""):
    RESULTS.append((name, bool(cond)))
    print(("PASS " if cond else "FAIL ") + name + ("" if cond else f"   [{str(detail)[:1200]}]"), flush=True)


def raises(fn, exc=Exception):
    try:
        fn()
    except exc:
        return True
    return False


def make_video(path, frames, fps=10):
    import av
    os.makedirs(os.path.dirname(path), exist_ok=True)
    c = av.open(path, "w")
    s = c.add_stream("mpeg4", rate=fps)
    s.width, s.height, s.pix_fmt = frames[0].shape[1], frames[0].shape[0], "yuv420p"
    s.bit_rate = 4_000_000
    for a in frames:
        for pk in s.encode(av.VideoFrame.from_ndarray(a, format="rgb24")):
            c.mux(pk)
    for pk in s.encode():
        c.mux(pk)
    c.close()


def synth_frames(n, w=64, h=48, seed=0):
    rng = np.random.RandomState(seed)
    base = rng.randint(0, 255, (h, w, 3), np.uint8)
    out = []
    for i in range(n):
        a = base.copy()
        a[:, :, 0] = (i * 37) % 256
        a[(i * 3) % h, :, :] = 255
        out.append(a)
    return out


def synth_rows(n_videos=40, per=5, config="32_frame"):
    rows, nid = [], 0
    for v in range(n_videos):
        for k in range(per):
            mc = k % 2 == 0
            rows.append({"id": nid, "dataset": ("ARKitScenes", "ScanNetv2", "3RScan")[v % 3], "scene_id": f"s{v:03d}",
                         "question_type": "route_planning" if mc else "object_counting_single",
                         "question": f"Q{nid}?", "ground_truth": "B" if mc else str(2 + k),
                         "options": ["A. left", "B. right", "C. back"] if mc else None,
                         "num_frames": config.split("_")[0], "queried_object_ids": [1]})
            nid += 1
    return rows


VIS = {f"s{v:03d}": [{"object_id": 1, "object_name": "chair", "visibility_16": v % 2 == 0, "visibility_32": True,
                      "visibility_64": True}] for v in range(40)}
SETTINGS = {"DATASET_REPO": "3dlg-hcvc/ReVSI", "DATASET_REVISION": "rev", "REVSI_CONFIG": "32_frame",
            "REVSI_SPLIT": "test", "VIDEO_FRACTION": 0.10, "SEED": 42}
tmp = tempfile.mkdtemp(prefix="revsi_test_")
try:
    raw = synth_rows()
    qa_all = [P.normalize_row(r, "32_frame", O, VIS) for r in raw]

    # ------------------------------------------------------------------ 3. sample
    keys = L.select_videos([q["video_key"] for q in qa_all], 0.10, 42)
    check("3 sample = ceil(10%) of unique videos (40 -> 4), each video once", len(keys) == 4 and len(set(keys)) == 4)
    shuffled_answers = [dict(q, reference_answer=str(random.Random(q["native_id"]).randint(0, 9))) for q in qa_all]
    check("3 selection uses no answers (changing every reference gives the same videos)",
          L.select_videos([q["video_key"] for q in shuffled_answers], 0.10, 42) == keys)
    d = os.path.join(tmp, "s3")
    man, qa, new = P.freeze_sample(d, qa_all, SETTINGS, {"f": "h"})
    man2, qa2, new2 = P.freeze_sample(d, list(reversed(qa_all)), SETTINGS, {"f": "h"})
    check("3 an interrupted run reuses the frozen sample unchanged (verified, not redrawn)", new and not new2 and qa == qa2)
    check("3 a changed revision or config cannot reuse the sample",
          raises(lambda: P.freeze_sample(d, qa_all, dict(SETTINGS, DATASET_REVISION="other"), {"f": "h"}), RuntimeError))
    check("3 the manifest keeps population, keys, seed, denominator, algorithm, per-task/source counts",
          all(k in man for k in ("population_keys", "selected_keys", "seed", "population_videos", "algorithm", "per_task",
                                 "per_source", "selected_questions")) and man["selected_questions"] == len(qa))

    # ------------------------------------------------------------------ 4. frames
    allf = synth_frames(120)
    idx = [int(i) for i in np.linspace(0, 119, 32).astype(int)]
    make_video(os.path.join(tmp, "v", "all.mp4"), allf)
    make_video(os.path.join(tmp, "v", "in32.mp4"), [allf[i] for i in idx])
    make_video(os.path.join(tmp, "v", "in33.mp4"), [allf[i] for i in idx] + [allf[5]])
    v_ok = F.verify_input_frames(os.path.join(tmp, "v", "in32.mp4"), os.path.join(tmp, "v", "all.mp4"), idx)
    check("4 released 32-frame input verified against all-frame at the official indices; timestamps from source fps",
          v_ok["status"] == "match" and v_ok["timestamps_s"][1] == round(idx[1] / 10, 3), v_ok.get("reason"))
    shifted = [min(i + 2, 119) for i in idx]
    check("4 a wrong index map (shifted) is caught", F.verify_input_frames(
        os.path.join(tmp, "v", "in32.mp4"), os.path.join(tmp, "v", "all.mp4"), shifted)["status"] == "mismatch")
    resampled = [int(i) for i in np.linspace(0, 119, 32 + 5).astype(int)][:32]
    check("4 independently resampled indices (linspace over a different budget) are caught", F.verify_input_frames(
        os.path.join(tmp, "v", "in32.mp4"), os.path.join(tmp, "v", "all.mp4"), resampled)["status"] == "mismatch")
    check("4 a fixed-budget video with the wrong frame count is refused", F.verify_input_frames(
        os.path.join(tmp, "v", "in33.mp4"), os.path.join(tmp, "v", "all.mp4"), idx)["status"] == "mismatch")
    bad = [dict(r, num_frames="16") for r in raw[:3]] + raw[3:]
    check("4 a config/num_frames mismatch is a schema error",
          any("num_frames" in p for p in P.schema_problems(bad, O, "32_frame", {r["scene_id"]: {"16-frame": list(range(16)),
              "32-frame": list(range(32)), "64-frame": list(range(64))} for r in raw}, VIS)))
    q16 = P.normalize_row(raw[1], "16_frame", O, VIS)
    q32 = P.normalize_row(raw[1], "32_frame", O, VIS)
    check("4 visibility uses the budget's own flag (16 vs 32 differ for this scene), never another budget's",
          q16["visibility"] == "all_visible" and P.normalize_row(raw[6], "16_frame", O, VIS)["visibility"] == "none_visible"
          and q32["visibility"] == "all_visible")

    # ------------------------------------------------------------------ 5. ids
    check("5 the same native id in two configs gives two distinct question ids",
          q16["qa_uid"] != q32["qa_uid"] and q16["qa_uid"].startswith("revsi:16_frame:"))
    check("5 normal and control jobs of one question have distinct job ids",
          L.job_id(q32["qa_uid"], "normal") != L.job_id(q32["qa_uid"], "black_frames"))

    # ------------------------------------------------------------------ 6. scorer parity
    mc = next(q for q in qa_all if q["task_family"] == "multiple_choice")
    na = next(q for q in qa_all if q["task_family"] == "numeric")
    cases = [(mc, "B", 1.0, "correct"), (mc, "b.", 1.0, "correct"), (mc, "C", 0.0, "wrong_candidate"),
             (mc, "The answer is B", 0.0, "format_error"), (mc, "", 0.0, "format_error"),
             (na, na["reference_answer"], 1.0, "correct"), (na, "100", 0.0, "wrong_candidate"),
             (na, "lots", 0.0, "format_error")]
    bad = []
    for q, raw_ans, want, st in cases:
        s = L.score_answer(q, raw_ans, O)
        direct = O["ns"]["revsi_process_results"]({"question_type": q["question_type"], "ground_truth": q["reference_answer"]},
                                                  [raw_ans])["overall_acc"]["acc"]
        if not (s["official_score"] == float(direct) == want and s["automatic_status"] == st):
            bad.append((raw_ans, s["official_score"], s["automatic_status"]))
    check("6 official parity: MC correct/wrong, malformed, numeric exact/zero (score == official function)", not bad, bad)
    part = L.score_answer(dict(na, reference_answer="10"), "8", O)
    check("6 partial numeric credit is a graded score ('imperfect'), not a failure",
          part["automatic_status"] == "imperfect" and 0 < part["official_score"] < 1)
    check("6 missing responses and inference errors are not scored",
          L.score_answer(na, None, O, status="missing")["official_score"] is None
          and L.score_answer(na, None, O, status="error")["automatic_status"] == "execution_error")
    z = L.score_answer(dict(na, reference_answer="0"), "0", O)
    check("6 a zero target gets a separate exact-match diagnostic (official MRA divides by the reference)",
          z["zero_target_exact_match"] == 1.0 and z["official_score"] == 0.0)
    sub = [L.score_answer(q, q["reference_answer"], O) for q in qa]
    check("6 subset scoring uses only the scored subset (no unsampled rows turned into zeros)",
          L.official_aggregate(sub, O)["overall_acc"] == 1.0)
    agg = L.official_aggregate([L.score_answer(q, "1", O) for q in qa_all if q["task_family"] == "numeric"][:6]
                               + [L.score_answer(mc, "B", O)], O)
    check("6 the composite hierarchy and macro overall come from the official aggregator",
          set(agg) == {"object_counting_acc", "route_planning_acc", "overall_acc"}
          and abs(agg["overall_acc"] - (agg["object_counting_acc"] + agg["route_planning_acc"]) / 2) < 1e-9)

    # ------------------------------------------------------------------ 7. resume + cache keys
    out = os.path.join(tmp, "run")
    os.makedirs(out)
    man, qa, _ = P.freeze_sample(out, qa_all, SETTINGS, {"f": "h"})
    ctrl = P.control_subset(qa, 4)
    fh = {v: f"fh-{v}" for v in man["selected_keys"]}
    jobs = P.build_jobs(qa, ctrl, fh, "CH", True)       # identity hash = the run config hash
    shard_dir, journal = os.path.join(out, "predictions"), os.path.join(out, "journal.jsonl")
    calls = {"n": 0}

    def fake_worker(batch_path, extra):
        b = json.load(open(batch_path))
        for v in b["videos"]:
            rows = [{"kind": "video_meta", "video_key": v["video_key"], "safe_key": v["safe_key"], "condition": c,
                     "config_hash": "CH", "positions": list(range(32)), "official_frame_indices": v["official_frame_indices"],
                     "timestamps_s": v["timestamps_s"], "frame_sha1": [f"h{i}" for i in range(32)], "preprocessing": "p",
                     "evidence_files": [f"in{i:02d}.jpg" for i in range(32)] if c == "normal" else []}
                    for c in sorted({j["condition"] for j in v["jobs"]})]
            for j in v["jobs"]:
                assert "reference" not in json.dumps(j)
                calls["n"] += 1
                rows.append({"kind": "answer", "job_id": j["job_id"], "qa_uid": j["qa_uid"], "condition": j["condition"],
                             "cache_key": j["cache_key"], "status": "ok", "config_hash": "CH", "video_key": v["video_key"],
                             "raw_answer": ("C" if j["qa_uid"].endswith(("0", "4")) else "B") if "Options" in j["prompt"]
                             else ("40" if j["qa_uid"].endswith(("1", "7")) else "3"),
                             "answered_at": L.time.strftime("%Y-%m-%dT%H:%M:%S.") + str(L.time.time_ns())})
            L.write_shard(shard_dir, v["safe_key"] + f"-{L.time.time_ns()}", rows, "CH")
        return 0

    media = {v: {"safe_key": L.safe_key(v)} for v in man["selected_keys"]}
    finfo = {v: {"official_frame_indices": idx, "timestamps_s": [i / 10 for i in idx]} for v in man["selected_keys"]}
    vid = os.path.join(tmp, "v", "in32.mp4")
    ctx = {"shard_dir": shard_dir, "journal": journal, "config_hash": "CH", "work_dir": out, "batch_bytes": 10 ** 9,
           "run_worker": fake_worker, "fetch_video": lambda v: vid, "release": lambda v, p: None,
           "log_missing": lambda v, why: None, "blocked": set()}
    by_uid = {q["qa_uid"]: q for q in qa}
    P.run_jobs(jobs, by_uid, media, finfo, ctx, log=lambda *a: None)
    n1 = calls["n"]
    s2 = P.run_jobs(jobs, by_uid, media, finfo, ctx, log=lambda *a: None)
    check("7 completed inference is skipped on resume", n1 == len(jobs) and calls["n"] == n1 and s2["pending_jobs"] == 0)
    for change, j2 in (("frame manifest", P.build_jobs(qa, ctrl, {v: "fh-NEW" for v in fh}, "CH", True)),
                       ("model/prompt/decoding identity", P.build_jobs(qa, ctrl, fh, "CH-OTHER", True))):
        ok2, _, _, rep = P.gather(shard_dir, journal, "CH", j2)
        check(f"7 a changed {change} hash invalidates cached results (reported as incompatible, rerun)",
              not ok2 and rep["incompatible_cached_rows"] == len(j2), rep)
    q_changed = [dict(q, content_hash="changed") if q["qa_uid"] == qa[0]["qa_uid"] else q for q in qa]
    ok3, _, _, _ = P.gather(shard_dir, journal, "CH", P.build_jobs(q_changed, ctrl, fh, "CH", True))
    check("7 a changed question content invalidates only that question's cached result",
          len(ok3) == len(jobs) - 1 - sum(1 for c in ctrl if c["qa_uid"] == qa[0]["qa_uid"]))

    # ------------------------------------------------------------------ review fixtures (8-12)
    run_cfg = {"identity": {}, "config_hash": "CH", "settings": {"RUN_VISUAL_CONTROLS": True}, "revisions": {"dataset": "rev"}}
    L.write_json_atomic(os.path.join(out, "run_config.json"), run_cfg)
    L.write_jsonl_atomic(os.path.join(out, "frame_manifest.jsonl"),
                         [{"video_key": v, "frame_manifest_hash": fh[v], "timestamps_s": finfo[v]["timestamps_s"],
                           "verification": {"status": "match"}} for v in man["selected_keys"]])
    L.write_json_atomic(os.path.join(out, "control_subset.json"), [c["qa_uid"] for c in ctrl])
    os.makedirs(os.path.join(out, "input_frames"), exist_ok=True)
    for v in man["selected_keys"]:
        fd = os.path.join(out, "input_frames", L.safe_key(v))
        os.makedirs(fd, exist_ok=True)
        for i in range(32):
            from PIL import Image
            Image.fromarray(allf[idx[i]]).save(os.path.join(fd, f"in{i:02d}.jpg"))
    # make some answers wrong so there are candidates: rewrite one shard row set by appending later wrong answers
    ctxr = R.load_context(out, O, journal, fetch_input=lambda v: vid)
    for v in man["selected_keys"]:
        R.export_candidate(v, ctxr, [q["qa_uid"] for q in qa if q["video_key"] == v])
    ctxr = R.load_context(out, O, journal, fetch_input=lambda v: vid)
    uids = [q["qa_uid"] for q in qa]
    v0 = qa[0]["video_key"]
    same_video = [q["qa_uid"] for q in qa if q["video_key"] == v0][:2]
    other = [q["qa_uid"] for q in qa if q["video_key"] != v0]

    def form(uid, **kw):
        base = {"qa_uid": uid, "annotation_status": "supported", "input_evidence_status": "sufficient",
                "model_verdict": "wrong", "review_stage": "final", "reviewer": "tester", "notes": "seen at #3",
                "citations": "#3", "evidence_source": "model_inputs"}
        base.update(kw)
        return base

    def save(uid, **kw):
        clean, errs = R.validate_review_row(form(uid, **kw), ctxr, "notebook_form")
        if not errs:
            R.append_review(out, clean)
        return clean, errs

    _, latest = R.load_reviews(out)
    queue_rows = R.build_queue(qa, ctxr["scores"], 42, 6, 20)
    cls = R.classify_all(queue_rows, latest, ctxr)
    check("8 zero reviews give zero verified failures", not any(c["verified_model_failure"] for c in cls.values()))
    _, e = save(other[0], notes="", citations="")
    check("8 a final review with empty notes and no citation is rejected (nothing saved)", e and not os.path.exists(
        os.path.join(out, "reviews.jsonl")), e)
    save(same_video[0])
    save(same_video[1])
    save(other[0], annotation_status="wrong", proposed_answer="C", input_evidence_status="sufficient")
    save(other[1], input_evidence_status="partial")
    save(other[2], review_stage="draft")
    _, latest = R.load_reviews(out)
    cls = R.classify_all(queue_rows, latest, ctxr)
    check("8 two verified failures on one video count once",
          len({c["video_key"] for c in cls.values() if c["verified_model_failure"]}) == 1
          and sum(c["verified_model_failure"] for c in cls.values()) == 2)
    check("8 an annotation error is a benchmark issue, never a verified model failure",
          cls[other[0]]["benchmark_issue"] and not cls[other[0]]["verified_model_failure"])
    check("8 partial input evidence and drafts never count",
          not cls[other[1]]["verified_model_failure"] and cls[other[1]]["insufficient_evidence"]
          and not cls[other[2]]["verified_model_failure"] and cls[other[2]]["draft"])
    stale_ctx = copy.deepcopy({k: ctxr[k] for k in ("preds",)})
    ctxr2 = dict(ctxr, preds={**stale_ctx["preds"], L.job_id(same_video[0], "normal"):
                              dict(ctxr["preds"][L.job_id(same_video[0], "normal")], raw_answer="changed")})
    c_stale = R.classify(same_video[0], latest[same_video[0]], ctxr2)
    check("8 a review becomes stale (and stops counting) when the prediction changes",
          c_stale["stale"] and not c_stale["verified_model_failure"])
    _, e = save(other[3], citations="#99")
    check("8 a citation outside the model inputs is rejected", e and any("#99" in x for x in e), e)
    _, e = R.validate_review_row(form("revsi:16_frame:0"), ctxr, "notebook_form")
    check("5 a review for a question id from another config is refused (cannot attach to the wrong QA)",
          e and "not a question of this run" in e[0], e)

    # ------------------------------------------------------------------ 9. 18 candidates, 0 reviews
    c18 = {"sample_videos": 29, "sample_questions": 453, "candidate_videos": 18, "candidate_exports": 18,
           "final_reviewed_videos": 0, "verified_model_failure_videos": 0, "annotation_issue_videos": 0,
           "insufficient_evidence_videos": 0, "uncertain_videos": 0, "inference_complete": True, "minimum": 10,
           "verified_failure_minimum_met": False}
    msg = R.status_message(c18, {"succeeded": 453, "expected": 453, "pending": 0, "failed": 0})
    check("9 18 candidates / 0 reviews: the exact message, no request to rerun inference",
          msg == "18 candidate videos exported; 0 reviewed; verified failures remain unestablished. Import or complete "
                 "reviews, then regenerate reports. Inference need not be rerun.", msg)
    m_empty = R.status_message(dict(c18, candidate_videos=0, candidate_exports=0), {"pending": 0, "failed": 0},
                               R.empty_queue_reasons([{"automatic_status": "imperfect"}] * 3 + [{"automatic_status": "correct"}],
                                                     {"pending": 0, "failed": 0}, {}))
    check("9 an empty queue explains the cause (here: only partial credit), never 'run full mode'",
          "partial-credit" in m_empty and "full mode" not in m_empty, m_empty)

    # ------------------------------------------------------------------ 10. review-only rebuild + CSV import
    code = ("import sys, json; sys.path.insert(0, %r); import revsi_lib as L, revsi_review as R; "
            "O = L.load_official(%r, %r); ctx = R.load_context(%r, O, %r); _, latest = R.load_reviews(%r); "
            "q = R.build_queue(ctx['qa'], ctx['scores'], 42, 6, 20); cls = R.classify_all(q, latest, ctx); "
            "print(json.dumps(sorted(u for u, c in cls.items() if c['verified_model_failure'])))"
            % (HERE, args.official_utils, args.official_yaml, out, journal, out))
    r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
    fresh = json.loads(r.stdout.strip().splitlines()[-1]) if r.returncode == 0 else r.stderr
    check("10 a fresh process rebuilds the same verified set from disk alone (no model, no globals)",
          fresh == sorted(u for u, c in cls.items() if c["verified_model_failure"]) and "torch" not in sys.modules, fresh)
    tpl = os.path.join(out, "review_template.csv")
    n_tpl = R.write_template(tpl, queue_rows, ctxr)
    rows = list(csv.DictReader(open(tpl)))
    target = next(r_ for r_ in rows if r_["qa_uid"] not in latest)
    target.update(annotation_status="supported", input_evidence_status="sufficient", model_verdict="wrong",
                  review_stage="final", reviewer="ext", notes="external: visible at #2", citations="#2")
    bad_row = dict(rows[-1] if rows[-1]["qa_uid"] != target["qa_uid"] else rows[0], review_stage="final",
                   model_verdict="maybe", notes="x", reviewer="ext")
    stale_row = dict(target, qa_uid=target["qa_uid"], prediction_hash="0000")
    with open(os.path.join(tmp, "edited.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=R.TEMPLATE_COLS)
        w.writeheader()
        w.writerows([target, bad_row])
    acc, rej = R.import_reviews(os.path.join(tmp, "edited.csv"), ctxr)
    _, latest2 = R.load_reviews(out)
    c_ext = R.classify(target["qa_uid"], latest2[target["qa_uid"]], ctxr)
    check("10 external CSV import reaches the same validator: valid row accepted and counted, bad row rejected with reasons",
          n_tpl > 0 and len(acc) == 1 and len(rej) == 1 and "model_verdict" in rej[0]["reasons"]
          and c_ext["final"] and os.path.exists(os.path.join(out, "review_import_errors.csv")), (acc, rej, c_ext))
    with open(os.path.join(tmp, "stale.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=R.TEMPLATE_COLS)
        w.writeheader()
        w.writerow(stale_row)
    R.import_reviews(os.path.join(tmp, "stale.csv"), ctxr)
    _, latest3 = R.load_reviews(out)
    check("10 an imported row with outdated hashes is kept but flagged stale and not counted",
          R.classify(target["qa_uid"], latest3[target["qa_uid"]], ctxr)["stale"])

    # ------------------------------------------------------------------ 11. no leakage in controls / baseline
    bl = P.frequency_baseline(qa_all, O)
    folds = {s["qa_uid"]: s["fold"] for s in bl}
    held = [q for q in qa_all if folds[q["qa_uid"]] == 0]
    poisoned = [dict(q, reference_answer="A" if q["task_family"] == "multiple_choice" else "999") if folds[q["qa_uid"]] == 0
                else q for q in qa_all]
    bl2 = {s["qa_uid"]: s["raw_answer"] for s in P.frequency_baseline(poisoned, O)}
    check("11 the frequency baseline never fits on held-out answers (changing fold-0 answers leaves fold-0 predictions)",
          held and all(bl2[q["qa_uid"]] == next(s["raw_answer"] for s in bl if s["qa_uid"] == q["qa_uid"]) for q in held))
    check("11 baseline folds are scene-disjoint", all(
        len({folds[q["qa_uid"]] for q in qa_all if q["video_key"] == v}) == 1 for v in {q["video_key"] for q in qa_all}))
    check("11 MC baseline answers are valid options", all(
        s["parsed_answer"] in L.option_letters(next(q for q in qa_all if q["qa_uid"] == s["qa_uid"])["options"])
        for s in bl if s["task_family"] == "multiple_choice"))
    check("11 control questions are predeclared from ids only (same set whatever the answers)",
          [q["qa_uid"] for q in P.control_subset(qa, 4)] ==
          [q["qa_uid"] for q in P.control_subset([dict(q, reference_answer="Z") for q in qa], 4)])

    # ------------------------------------------------------------------ 12. exports
    _, latest = R.load_reviews(out)
    cls = R.classify_all(queue_rows, latest, ctxr)
    fin = R.export_final(ctxr, cls, latest)
    vf, bi = fin["verified_failures"], fin["benchmark_issues"]
    check("12 every exported case has valid video/case/contact-sheet paths",
          vf["distinct_videos"] >= 1 and all(not c["missing_files"] for c in vf["cases"] + bi["cases"]), vf["cases"])
    check("12 benchmark issues are exported separately from model failures",
          bi["distinct_videos"] >= 1 and not set(c["video_key"] for c in bi["cases"]) & set() and
          all(cls[u]["benchmark_issue"] for c in bi["cases"] for u in c["qa_uids"]))
    cnt = R.counts(ctxr, queue_rows, cls, ctxr["coverage"]["normal"], 10, 20)
    st = R.status_message(cnt, ctxr["coverage"]["normal"])
    check("12 fewer than the minimum is reported honestly as not met, without expanding the sample",
          not cnt["verified_failure_minimum_met"] and "not met" in st and "not expanded" in st, st)
    # ------------------------------------------------------------------ required review set (REVIEW_VIDEOS)
    qn = R.build_queue(qa, ctxr["scores"], 42, 6, 20, review_n=2, audit_required=False)
    req = [r for r in qn if r["required"]]
    prim = [r for r in qn if r["queue"] == "diagnostic" and r["role"] == "primary"]
    check("review set: exactly the main question of the first N candidate videos (fixed queue order); audit optional",
          len(req) == min(2, len(prim)) and [r["rank"] for r in req] == list(range(1, len(req) + 1))
          and all(r["role"] == "primary" for r in req) and not any(r["required"] for r in qn if r["queue"] == "audit"))
    check("review set: AUDIT_REQUIRED adds the audit questions",
          all(r["required"] for r in R.build_queue(qa, ctxr["scores"], 42, 6, 20, 2, True) if r["queue"] == "audit"))
    d2 = os.path.join(tmp, "req")
    shutil.copytree(out, d2, ignore=shutil.ignore_patterns("reviews.jsonl"))
    ctx2 = dict(ctxr, out_root=d2)
    for r in req[:-1]:
        clean, e_ = R.validate_review_row(form(r["qa_uid"]), ctx2, "notebook_form")
        R.append_review(d2, clean)
    _, lt = R.load_reviews(d2)
    c_part = R.counts(ctx2, qn, R.classify_all(qn, lt, ctx2), ctx2["coverage"]["normal"], 10, 20)
    clean, e_ = R.validate_review_row(form(req[-1]["qa_uid"], review_stage="draft"), ctx2, "notebook_form")
    R.append_review(d2, clean)
    _, lt = R.load_reviews(d2)
    c_draft = R.counts(ctx2, qn, R.classify_all(qn, lt, ctx2), ctx2["coverage"]["normal"], 10, 20)
    clean, e_ = R.validate_review_row(form(req[-1]["qa_uid"]), ctx2, "notebook_form")
    R.append_review(d2, clean)
    _, lt = R.load_reviews(d2)
    c_full = R.counts(ctx2, qn, R.classify_all(qn, lt, ctx2), ctx2["coverage"]["normal"], 10, 20)
    check("review set: complete only when all N required reviews are final (a draft does not count; audit not needed)",
          not c_part["review_complete"] and not c_draft["review_complete"] and c_full["review_complete"]
          and c_full["required_reviews_final"] == c_full["required_reviews"] == len(req)
          and "Required reviews: " in R.status_message(c_full, ctx2["coverage"]["normal"]), (c_part, c_full))
finally:
    shutil.rmtree(tmp, ignore_errors=True)

bad = [n for n, ok in RESULTS if not ok]
print(f"\n{len(RESULTS) - len(bad)}/{len(RESULTS)} checks passed")
sys.exit(1 if bad else 0)

### Helper code: `revsi_worker_selftest.py`

In [ ]:
%%writefile /content/revsi/revsi_worker_selftest.py
"""Runs the REAL revsi_worker.py on a real 32-frame video with only the model replaced by a stand-in (tiny fake
torch / cambrianp / lmms_eval packages first on the worker's PYTHONPATH). Checks the worker's own code: exact-budget
decoding (no resampling; a wrong frame count is refused), frame provenance and evidence files, the black-frame
condition, the prompt and generation settings, saved rows with cache keys, retries, out-of-memory, deliberate stop,
and that a bookkeeping bug stops the worker. Seconds; no GPU. Run it with the worker's interpreter (needs decord).

  python revsi_worker_selftest.py --video <32-frame mp4> [--bad_video <mp4 with another frame count>]
"""
import argparse
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tempfile

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
import revsi_lib as L  # noqa: E402
import revsi_pipeline as P  # noqa: E402

ap = argparse.ArgumentParser()
ap.add_argument("--video", required=True)
ap.add_argument("--bad_video")
ap.add_argument("--worker", default=os.path.join(HERE, "revsi_worker.py"))
ap.add_argument("--num_frames", type=int, default=32)
args = ap.parse_args()
args.video = os.path.abspath(args.video)
RESULTS = []


def check(name, cond, detail=""):
    RESULTS.append((name, bool(cond)))
    print(("PASS " if cond else "FAIL ") + name + ("" if cond else f"   [{str(detail)[-1500:]}]"), flush=True)


STUBS = {
    "torch/__init__.py": '''
__version__ = "stub"
class version:
    cuda = "stub"
class _OOM(RuntimeError):
    pass
class cuda:
    OutOfMemoryError = _OOM
    @staticmethod
    def get_device_name(i):
        return "stand-in GPU"
    @staticmethod
    def empty_cache():
        pass
''',
    "cambrianp/__init__.py": "",
    "cambrianp/utils.py": "def cast_cambrian_config_to_llava_ov_style(config, model_path):\n    return config\n",
    "cambrianp/constants.py": 'DEFAULT_IMAGE_TOKEN = "<image>"\nIMAGE_TOKEN_INDEX = -200\n',
    "cambrianp/conversation.py": '''
class _Conv:
    roles = ("<|im_start|>user", "<|im_start|>assistant")
    def __init__(self):
        self.messages = []
    def copy(self):
        return _Conv()
    def append_message(self, role, msg):
        self.messages.append((role, msg))
    def get_prompt(self):
        return "".join(f"{r}\\n{m}<|im_end|>\\n" if m is not None else f"{r}\\n" for r, m in self.messages)
conv_templates = {"qwen_1_5": _Conv()}
''',
    "cambrianp/mm_utils.py": '''
class Ids:
    def __init__(self, prompt):
        self.prompt = prompt
    def unsqueeze(self, i):
        return self
    def to(self, device):
        return self
    def ne(self, pad):
        return ("mask", self.prompt)
def tokenizer_image_token(prompt, tokenizer, index, return_tensors=None):
    return Ids(prompt)
''',
    "lmms_eval/__init__.py": "", "lmms_eval/models/__init__.py": "", "lmms_eval/models/simple/__init__.py": "",
    "lmms_eval/models/simple/cambrian_p.py": '''
import json, os, torch
def expand2square(img, bg):
    return img
class _T:
    shape = (8, 3, 384, 384)
    def half(self):
        return self
    def cuda(self):
        return self
class _Proc:
    image_mean = [0.5, 0.5, 0.5]
    def preprocess(self, frames, return_tensors=None):
        assert len(frames) > 0
        return {"pixel_values": _T()}
class _Cam:
    shape = (2, 3584)
    def detach(self):
        return self
    def float(self):
        return self
    def abs(self):
        return self
    def sum(self):
        return 5.0
class _Inner:
    rec_head = type("H", (), {"camera_tokens": _Cam()})()
class _Model:
    config = type("C", (), {"vocab_size": 151672})()
    def get_model(self):
        return _Inner()
class _Tok:
    pad_token_id = 0
    eos_token_id = 1
class CambrianP:
    calls = {}
    def __init__(self, **kw):
        self.tokenizer, self.model, self._image_processor, self.device = _Tok(), _Model(), _Proc(), "cuda"
    def _run_inference_step(self, ids, mask, images, image_size, task, gen, video):
        prompt = ids.prompt
        log = os.environ["STUB_LOG"]
        n = sum(1 for l in open(log)) if os.path.exists(log) else 0
        with open(log, "a") as f:
            f.write(json.dumps({"prompt": prompt, "gen": gen, "task": task, "image_size": image_size}) + "\\n")
        q = prompt.split("<image>\\n", 1)[1].split("<|im_end|>")[0]
        if os.environ.get("STUB_OOM") and os.environ["STUB_OOM"] in q:
            raise torch.cuda.OutOfMemoryError("stand-in OOM")
        flaky = os.environ.get("STUB_FAIL_ONCE")
        if flaky and flaky in q and not os.path.exists(log + ".failed"):
            open(log + ".failed", "w").close()
            raise RuntimeError("stand-in transient failure")
        return ["B" if "Options:" in q else "3"]
''',
}


def make_stubs(root):
    for rel, text in STUBS.items():
        p = os.path.join(root, rel)
        os.makedirs(os.path.dirname(p), exist_ok=True)
        open(p, "w").write(text)



def jobs_for(vk):
    mk_job = lambda uid, cond, prompt: {"job_id": L.job_id(uid, cond), "qa_uid": uid, "condition": cond, "native_id": 1,
                                        "cache_key": "ck-" + uid + cond, "frame_manifest_hash": "fh", "prompt": prompt}
    mc = ("These are frames of a video.\nWhich way? MARK-MC\nOptions:\nA. left\nB. right\n"
          "Answer with the option's letter from the given choices directly.")
    na = "These are frames of a video.\nHow many chairs are in the scene? MARK-NA\nAnswer the question using a single integer or decimal number."
    return [mk_job("revsi:32_frame:1", "normal", mc), mk_job("revsi:32_frame:2", "normal", na),
            mk_job("revsi:32_frame:2", "black_frames", na)]


def run(tmp, name, env_extra, extra_args=(), video=None):
    d = os.path.join(tmp, name)
    os.makedirs(d)
    vk = "ScanNetv2/scene_test"
    n = args.num_frames
    batch = {"videos": [{"video_key": vk, "safe_key": L.safe_key(vk), "local_path": video or args.video,
                         "official_frame_indices": [i * 7 for i in range(n)], "timestamps_s": [i * 0.7 for i in range(n)],
                         "jobs": jobs_for(vk)}]}
    L.write_json_atomic(os.path.join(d, "batch.json"), batch)
    gen = {"max_new_tokens": 16, "temperature": 0, "do_sample": False}
    env = dict(os.environ, PYTHONPATH=os.path.join(tmp, "stubs") + os.pathsep + os.environ.get("PYTHONPATH", ""),
               STUB_LOG=os.path.join(d, "model_calls.jsonl"), MPLBACKEND="Agg", **env_extra)
    cmd = [sys.executable, args.worker, "--batch", os.path.join(d, "batch.json"), "--model_path", "stub",
           "--num_frames", str(n), "--config_hash", "H", "--identity", json.dumps({"GENERATION": gen}),
           "--journal", os.path.join(d, "journal.jsonl"), "--errors", os.path.join(d, "errors.jsonl"),
           "--shard_dir", os.path.join(d, "shards"), "--status", os.path.join(d, "status.json"),
           "--lib_dir", HERE, "--frames_dir", os.path.join(d, "frames"), "--max_retries", "2", *extra_args]
    r = subprocess.run(cmd, env=env, capture_output=True, text=True, cwd=d)
    calls = [json.loads(l) for l in open(env["STUB_LOG"])] if os.path.exists(env["STUB_LOG"]) else []
    status = json.load(open(os.path.join(d, "status.json"))) if os.path.exists(os.path.join(d, "status.json")) else {}
    jobs = [dict(j, video_key=vk) for j in jobs_for(vk)]
    ok, failed, meta, rep = P.gather(os.path.join(d, "shards"), os.path.join(d, "journal.jsonl"), "H", jobs)
    return r, calls, status, ok, failed, meta, d, gen


tmp = tempfile.mkdtemp(prefix="revsi_worker_test_")
try:
    make_stubs(os.path.join(tmp, "stubs"))
    r, calls, status, ok, failed, meta, d, gen = run(tmp, "normal", {})
    check("worker completes (exit 0, phase done, camera head check passed)",
          r.returncode == 0 and status.get("phase") == "done" and status.get("camera_head_loaded"), r.stdout + r.stderr)
    check("every job is saved once with its cache key, condition and the full rendered prompt",
          len(ok) == 3 and not failed and all(v.get("cache_key") and v.get("condition") and v.get("rendered_prompt")
                                             and isinstance(v.get("raw_answer"), str) for v in ok.values()), (ok, failed))
    check("each job calls the model exactly once", len(calls) == 3, len(calls))
    check("the model gets '<image>\\n' + the official prompt in the qwen_1_5 template, with the run's generation settings",
          calls and calls[0]["prompt"].startswith("<|im_start|>user\n<image>\nThese are frames of a video.\nWhich way? MARK-MC")
          and all(c["gen"] == gen and c["task"] == "video" for c in calls), calls[:1])
    from decord import VideoReader, cpu
    import numpy as np
    vr = VideoReader(args.video, ctx=cpu(0), num_threads=0)
    want = [hashlib.sha1(np.ascontiguousarray(f).tobytes()).hexdigest() for f in vr.get_batch(list(range(len(vr)))).asnumpy()]
    mn, mb = meta.get(("ScanNetv2/scene_test", "normal"), {}), meta.get(("ScanNetv2/scene_test", "black_frames"), {})
    check("normal input = every frame of the fixed-budget video once, in order (no resampling), hashes recorded",
          mn.get("positions") == list(range(args.num_frames)) and mn.get("input_frame_count") == args.num_frames
          and mn.get("frame_sha1") == want, {k: mn.get(k) for k in ("positions", "input_frame_count")})
    check("frame provenance keeps official source indices and timestamps per input position",
          mn.get("official_frame_indices") == [i * 7 for i in range(args.num_frames)] and len(mn.get("timestamps_s", [])) == args.num_frames)
    files = sorted(os.listdir(os.path.join(d, "frames", L.safe_key("ScanNetv2/scene_test"))))
    check("the exact input frames are saved as evidence (one file per input position)",
          len(files) == args.num_frames and files == sorted(mn.get("evidence_files", [])) and files[0].startswith("in00_src000000"))
    zero = hashlib.sha1(np.zeros(tuple(vr[0].shape), np.uint8).tobytes()).hexdigest()
    check("black-frame control: same frame count and timestamps, every input frame all-zero, construction recorded",
          mb.get("input_frame_count") == args.num_frames and set(mb.get("frame_sha1", [])) == {zero}
          and mb.get("timestamps_s") == mn.get("timestamps_s") and "all-zero" in (mb.get("control_construction") or ""))
    check("no reference answer is sent to the worker or the model",
          not any("reference" in json.dumps(j) for j in json.load(open(os.path.join(d, "batch.json")))["videos"][0]["jobs"]))
    if args.bad_video:
        r, calls, status, ok, failed, meta, d, _ = run(tmp, "bad_count", {}, video=os.path.abspath(args.bad_video))
        check("a video with another frame count is refused as a protocol violation (no resampling, no model call)",
              not ok and len(failed) == 3 and all("protocol violation" in v[-1]["error"] for v in failed.values()) and not calls,
              (failed, len(calls)))
    r, calls, status, ok, failed, meta, d, _ = run(tmp, "retry", {"STUB_FAIL_ONCE": "MARK-MC"})
    check("a transient model failure is retried; the answer is saved with attempt 2",
          r.returncode == 0 and ok.get(L.job_id("revsi:32_frame:1", "normal"), {}).get("attempt") == 2, r.stderr[-400:])
    r, calls, status, ok, failed, meta, d, _ = run(tmp, "oom", {"STUB_OOM": "MARK-NA"})
    check("out of GPU memory stops the worker and keeps the earlier answer",
          r.returncode != 0 and status.get("phase") == "stopped: out of GPU memory" and len(ok) == 1, (r.returncode, status))
    r, calls, status, ok, failed, meta, d, _ = run(tmp, "stop", {}, ["--stop_after", "1"])
    check("--stop_after 1 exits 75 with one answer kept", r.returncode == 75 and len(ok) == 1, (r.returncode, list(ok)))
    shutil.copy(args.worker, os.path.join(tmp, "worker_broken.py"))
    src = open(os.path.join(tmp, "worker_broken.py")).read()
    marker = '"native_id": j["native_id"]'
    if marker in src:
        open(os.path.join(tmp, "worker_broken.py"), "w").write(src.replace(marker, '"native_id": j["no_such_field"]'))
        real, args.worker = args.worker, os.path.join(tmp, "worker_broken.py")
        r, calls, status, ok, failed, meta, d, _ = run(tmp, "bookkeeping_bug", {})
        args.worker = real
        check("a bug in saving an answer stops the worker (no retries, no fake model errors)",
              r.returncode != 0 and "KeyError" in r.stderr and len(calls) == 1 and not failed, (r.returncode, len(calls)))
    else:
        check("the worker saves native_id (needed by the bookkeeping-bug test)", False)
finally:
    shutil.rmtree(tmp, ignore_errors=True)

bad = [n for n, ok_ in RESULTS if not ok_]
print(f"\n{len(RESULTS) - len(bad)}/{len(RESULTS)} worker checks passed")
sys.exit(1 if bad else 0)

In [ ]:
# @title A3. Pin dataset / model / code / scorer revisions; load the official scorer; validate the run configuration
_helpers = ['revsi_lib.py', 'revsi_pipeline.py', 'revsi_frames.py', 'revsi_review.py', 'revsi_worker.py', 'revsi_selftest.py', 'revsi_worker_selftest.py']
_absent = [n for n in _helpers if not os.path.exists(os.path.join("/content/revsi", n))]
if _absent:
    raise SystemExit(f"The helper files are not in this runtime ({_absent[0]} ...). Run the 'Helper code' cells "
                     "below A2 (Runtime > Run all, or Run after from A1), then rerun A3.")
sys.path.insert(0, "/content/revsi")
import importlib, urllib.request
import revsi_lib as L, revsi_pipeline as P, revsi_review as R, revsi_frames as F
for _m in (L, P, R, F):
    importlib.reload(_m)
from huggingface_hub import HfApi, hf_hub_url
api = HfApi(token=HF_TOKEN)
AUTH = {"Authorization": f"Bearer {HF_TOKEN}"} if HF_TOKEN else {}
pins_path = os.path.join(OUTPUT_ROOT, "pins.json")
pins = json.load(open(pins_path)) if os.path.exists(pins_path) else {}

def resolve(name, given, fetch):
    """Explicit value > value saved for this run > resolved now. On resume the saved value is used."""
    value = given or pins.get(name) or fetch()
    if pins.get(name) and pins[name] != value:
        raise RuntimeError(f"{name} {value} differs from {pins[name]} saved for this run; use a new RUN_ID.")
    pins[name] = value
    return value
ls_remote = lambda url: subprocess.run(["git", "ls-remote", url, "HEAD"], capture_output=True, text=True,
                                       check=True).stdout.split()[0]
DATASET_REVISION = resolve("DATASET_REVISION", DATASET_REVISION, lambda: api.dataset_info(DATASET_REPO).sha)
MODEL_REVISION = resolve("MODEL_REVISION", MODEL_REVISION, lambda: api.model_info(MODEL_REPO).sha)
CODE_COMMIT = resolve("CODE_COMMIT", CODE_COMMIT, lambda: ls_remote(REPO_URL))
SCORER_COMMIT = resolve("SCORER_COMMIT", SCORER_COMMIT, lambda: ls_remote(f"https://github.com/{SCORER_REPO}"))
L.write_json_atomic(pins_path, pins)

SCORER_DIR = os.path.join(CACHE_ROOT, "scorers", f"lmms-eval-revsi@{SCORER_COMMIT}")
os.makedirs(SCORER_DIR, exist_ok=True)
for _f in ("utils.py", "_default_template_yaml"):
    _p = os.path.join(SCORER_DIR, _f)
    if not os.path.exists(_p):
        _u = f"https://raw.githubusercontent.com/{SCORER_REPO}/{SCORER_COMMIT}/lmms_eval/tasks/revsi/{_f}"
        with urllib.request.urlopen(_u, timeout=120) as r_, open(_p + ".part", "wb") as o_:
            o_.write(r_.read())
        os.replace(_p + ".part", _p)
OFFICIAL = L.load_official(os.path.join(SCORER_DIR, "utils.py"), os.path.join(SCORER_DIR, "_default_template_yaml"))
for _k, _v in (("SCORER_SHA256", OFFICIAL["sha256"]), ("SCORER_YAML_SHA256", OFFICIAL["yaml_sha256"])):
    resolve(_k, None, lambda v=_v: v)
L.write_json_atomic(pins_path, pins)
assert OFFICIAL["dataset_path"] == DATASET_REPO and OFFICIAL["test_split"] == REVSI_SPLIT
GENERATION = dict(OFFICIAL["generation_kwargs"])           # max_new_tokens 16, greedy (from the pinned YAML)
SCORER = {"repo": SCORER_REPO, "commit": SCORER_COMMIT, "file": "lmms_eval/tasks/revsi/utils.py",
          "sha256": OFFICIAL["sha256"], "yaml_sha256": OFFICIAL["yaml_sha256"], "prompt_kwargs": OFFICIAL["prompt_kwargs"]}
IDENTITY = {"BENCHMARK": L.BENCHMARK, "DATASET_REPO": DATASET_REPO, "DATASET_REVISION": DATASET_REVISION,
            "REVSI_CONFIG": REVSI_CONFIG, "REVSI_SPLIT": REVSI_SPLIT, "SAMPLING_UNIT": "video (dataset, scene_id)",
            "VIDEO_FRACTION": VIDEO_FRACTION, "SEED": SEED, "MODEL_REPO": MODEL_REPO, "MODEL_REVISION": MODEL_REVISION,
            "CODE_COMMIT": CODE_COMMIT, "NUM_FRAMES": NUM_FRAMES, "FRAME_PROTOCOL": FRAME_PROTOCOL,
            "PROMPT_POLICY": PROMPT_POLICY, "GENERATION": GENERATION, "PREPROCESSING": PREPROCESSING, "SCORER": SCORER,
            "RUN_KIND": RUN_KIND}
RUN_CONFIG_PATH = os.path.join(OUTPUT_ROOT, "run_config.json")
run_config = L.check_run_config(RUN_CONFIG_PATH, IDENTITY, extra={
    "revisions": {"dataset": DATASET_REVISION, "model": MODEL_REVISION, "code": CODE_COMMIT, "scorer": SCORER_COMMIT,
                  "vsi_dataset": VSI_DATASET_REVISION},
    "settings": {"RUN_VISUAL_CONTROLS": RUN_VISUAL_CONTROLS, "AUDIT_QA": AUDIT_QA, "CONTROL_QA": CONTROL_QA,
                 "MIN_REVIEW_VIDEOS": MIN_REVIEW_VIDEOS, "TARGET_REVIEW_VIDEOS": TARGET_REVIEW_VIDEOS,
                 "REVIEW_VIDEOS": REVIEW_VIDEOS, "AUDIT_REQUIRED": AUDIT_REQUIRED},
    "analysis_packages": ANALYSIS_VERSIONS, "updated": _t.strftime("%Y-%m-%dT%H:%M:%SZ")})
CONFIG_HASH = run_config["config_hash"]
SHARD_DIR = os.path.join(OUTPUT_ROOT, "predictions")
JOURNAL = os.path.join(LOCAL_ROOT, "journal", f"answers-{CONFIG_HASH}.jsonl")
ERRORS_JOURNAL = os.path.join(LOCAL_ROOT, "journal", f"errors-{CONFIG_HASH}.jsonl")
for _d in ("predictions", "input_frames", "candidate_cases", "verified_failures", "benchmark_issues", "logs"):
    os.makedirs(os.path.join(OUTPUT_ROOT, _d), exist_ok=True)
for k in ("DATASET_REVISION", "MODEL_REVISION", "CODE_COMMIT", "SCORER_COMMIT"):
    print(f"{k:17s} {pins[k]}")
print(f"Official scorer sha256 {OFFICIAL['sha256'][:16]}...; generation {GENERATION}; config hash {CONFIG_HASH}")
print("Lightweight analysis can now run without model weights.")

In [ ]:
# @title A4. Model environment (restored from the Drive cache when possible; weights load only when needed)
REPO, VENV_PY = "/content/cambrian-p", "/content/venv/bin/python"
if RUN_MODE == "review_only":
    def ensure_model():
        raise SystemExit("review_only: the model is never loaded; set RUN_MODE to full to run inference.")
    print("review_only: model environment skipped (nothing installed or restored, no weights).")
else:
    import hashlib
    REPO = "/content/cambrian-p"
    VENV_PY = "/content/venv/bin/python"
    MODEL_DIR = f"{REPO}/ckpts/Cambrian-P-7B"
    CHECKOUT = r"""
set -e
cd /content
if [ ! -d cambrian-p/.git ]; then git clone -q "$REPO_URL" cambrian-p; fi
cd cambrian-p
if [ "$(git rev-parse HEAD)" != "$CODE_COMMIT" ]; then
    git fetch -q origin "$CODE_COMMIT" 2>/dev/null || git fetch -q origin
    git checkout -q "$CODE_COMMIT"
fi
echo "cambrian-p at $(git rev-parse HEAD)"
"""
    INSTALL = r"""
set -e
cd /content/cambrian-p
echo "Installing (about 10-15 minutes) ..."
pip install -q uv
uv venv --seed --python 3.11 /content/venv
source /content/venv/bin/activate
pip install -q -e ./vggt
pip install -q hydra-core tensorboard iopath wcmatch fvcore
pip install -q --upgrade pip
pip install -q -e ".[train]"
pip install -q torch==2.4.1+cu121 torchvision==0.19.1+cu121 torchaudio==2.4.1+cu121 --index-url https://download.pytorch.org/whl/cu121
pip install -q accelerate==0.29.3 easydict matplotlib roma evo imageio OpenEXR
pip install -q loguru sqlitedict sacrebleu evaluate tenacity pytz
pip install -q pyarrow==15.0.2
"""
    # The cache name pins the code commit and the exact install steps: change either and the old copy is never used.
    ENV_TAG = f"env-{CODE_COMMIT[:12]}-{hashlib.sha256(INSTALL.encode()).hexdigest()[:12]}"
    ENV_TAR = os.path.join(CACHE_ROOT, "envs", ENV_TAG + ".tar")
    MODEL_CACHE = os.path.join(CACHE_ROOT, "models", f"Cambrian-P-7B@{MODEL_REVISION}")
    env = dict(os.environ, REPO_URL=REPO_URL, CODE_COMMIT=CODE_COMMIT, MODEL_REPO=MODEL_REPO, MODEL_REVISION=MODEL_REVISION,
               MODEL_DIR=MODEL_DIR, UV_PYTHON_INSTALL_DIR="/content/python", HF_HUB_ENABLE_HF_TRANSFER="1")
    if HF_TOKEN:
        env["HF_TOKEN"] = HF_TOKEN

    def sh(script, what):
        if subprocess.run(["bash", "-c", script], env=env).returncode:
            raise SystemExit(f"{what} failed (see the log above).")

    def env_ok():
        """The same checks whether the environment was just installed or restored from the cache."""
        if not os.path.exists(VENV_PY) or not os.path.isdir(os.path.join(REPO, ".git")):
            return False                       # nothing installed yet in this runtime
        imports = subprocess.run([VENV_PY, "-c", "import torch, decord"], capture_output=True, text=True)
        head = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
        return imports.returncode == 0 and head == CODE_COMMIT

    def drive_free():
        try:
            return shutil.disk_usage("/content/drive/MyDrive").free
        except OSError:
            return 0

    def tree_sizes(root):
        out = {}
        for d, dirs, files in os.walk(root):
            dirs[:] = [x for x in dirs if x != ".cache"]
            for f in files:
                if not f.startswith(".revision-"):
                    p = os.path.join(d, f)
                    out[os.path.relpath(p, root)] = os.path.getsize(p)
        return out

    # --- environment ---
    t0 = L.time.time()
    if env_ok():
        print("Environment already set up in this session.")
    else:
        if CACHE_ENV and os.path.exists(ENV_TAR):
            print(f"Restoring the environment from the Drive cache ({os.path.getsize(ENV_TAR) / 1e9:.1f} GB, {ENV_TAG}) ...", flush=True)
            for d in ("venv", "python", "cambrian-p"):
                shutil.rmtree(os.path.join("/content", d), ignore_errors=True)
            r = subprocess.run(["tar", "-xf", ENV_TAR, "-C", "/content"])
            if r.returncode or not env_ok():
                print("The cached environment failed its checks; ignoring it and installing fresh.")
                for d in ("venv", "python", "cambrian-p"):
                    shutil.rmtree(os.path.join("/content", d), ignore_errors=True)
        sh(CHECKOUT, "Checkout")
        if not env_ok():
            sh(INSTALL, "Install")
            if not env_ok():
                raise SystemExit("The environment failed its checks after installing (torch/decord import or commit).")
            if CACHE_ENV:
                print("Saving the environment to the Drive cache for later sessions ...", flush=True)
                os.makedirs(os.path.dirname(ENV_TAR), exist_ok=True)
                r = subprocess.run(["tar", "-cf", ENV_TAR + ".partial", "-C", "/content", "--exclude=cambrian-p/ckpts",
                                    "venv", "python", "cambrian-p"])
                if r.returncode:
                    print("  could not save it (Drive full?); the next session will reinstall.")
                    if os.path.exists(ENV_TAR + ".partial"):
                        os.remove(ENV_TAR + ".partial")
                else:
                    os.replace(ENV_TAR + ".partial", ENV_TAR)
                    print(f"  saved {os.path.getsize(ENV_TAR) / 1e9:.1f} GB as {ENV_TAR}")
    sh(CHECKOUT, "Checkout")
    print(f"Environment ready in {(L.time.time() - t0) / 60:.1f} min.")

    # --- model weights: loaded only when 6b has questions to answer (scoring and review never need them) ---
    def ensure_model():
        t0 = L.time.time()
        local_marker = os.path.join(MODEL_DIR, f".revision-{MODEL_REVISION}")
        cache_marker = os.path.join(MODEL_CACHE, ".complete.json")
        if os.path.exists(local_marker):
            print("Model already on local disk.")
        else:
            if CACHE_MODEL and os.path.exists(cache_marker):
                print(f"Copying the model from the Drive cache ({MODEL_CACHE}) ...", flush=True)
                expected = json.load(open(cache_marker))["files"]
                shutil.rmtree(MODEL_DIR, ignore_errors=True)
                shutil.copytree(MODEL_CACHE, MODEL_DIR, ignore=shutil.ignore_patterns(".complete.json"))
                if tree_sizes(MODEL_DIR) != expected:
                    print("  the cached copy is incomplete; downloading instead.")
                    shutil.rmtree(MODEL_DIR, ignore_errors=True)
            if not os.path.exists(os.path.join(MODEL_DIR, "config.json")):
                sh("""cd /content/cambrian-p && /content/venv/bin/huggingface-cli download "$MODEL_REPO" --revision "$MODEL_REVISION" \\
                  --local-dir "$MODEL_DIR" """, "Model download")
                if CACHE_MODEL:
                    files = tree_sizes(MODEL_DIR)
                    need = sum(files.values())
                    if drive_free() < need + 1e9:
                        print(f"  not cached: Drive has {drive_free() / 1e9:.0f} GB free, the model needs {need / 1e9:.1f} GB.")
                    else:
                        print(f"Saving the model ({need / 1e9:.1f} GB) to the Drive cache for later sessions ...", flush=True)
                        shutil.rmtree(MODEL_CACHE + ".partial", ignore_errors=True)
                        shutil.copytree(MODEL_DIR, MODEL_CACHE + ".partial",
                                        ignore=shutil.ignore_patterns(".cache", ".revision-*"))
                        L.write_json_atomic(os.path.join(MODEL_CACHE + ".partial", ".complete.json"),
                                            {"model_repo": MODEL_REPO, "revision": MODEL_REVISION, "files": files})
                        shutil.rmtree(MODEL_CACHE, ignore_errors=True)
                        os.replace(MODEL_CACHE + ".partial", MODEL_CACHE)
            open(local_marker, "w").close()
        print(f"Model ready in {(L.time.time() - t0) / 60:.1f} min.")
    print("Model weights: not loaded now; D loads them only if jobs still need answers.")


WORKER_ENV = dict(os.environ, PYTHONPATH=f"{REPO}/lmms-eval:{REPO}", PATH=f"/content/venv/bin:{os.environ['PATH']}",
                  MPLBACKEND="Agg")

## B. Load metadata, freeze the sample, inspect overlap

In [ ]:
# @title B1. ReVSI file tree, card, config files and metadata (pinned revision); schema checks; freeze the 10% video sample
DATA_CACHE = os.path.join(CACHE_ROOT, "datasets", f"ReVSI@{DATASET_REVISION}")
os.makedirs(DATA_CACHE, exist_ok=True)
_tree = os.path.join(DATA_CACHE, "files.json")
if os.path.exists(_tree):
    FILES = json.load(open(_tree))
else:
    FILES = {f.path: {"size": getattr(f, "size", None), "sha256": getattr(getattr(f, "lfs", None), "sha256", None)}
             for f in api.list_repo_tree(DATASET_REPO, repo_type="dataset", revision=DATASET_REVISION, recursive=True)}
    L.write_json_atomic(_tree, FILES)

def cached_file(name, repo=DATASET_REPO, rev=None, cache=None, files=None):
    """A small dataset file at the pinned revision, downloaded once over HTTPS, size/LFS-hash checked, kept on Drive."""
    files, cache = files or FILES, cache or DATA_CACHE
    path = os.path.join(cache, "files", name)
    want = files.get(name) or {}
    if not (os.path.exists(path) and (want.get("size") is None or os.path.getsize(path) == want["size"])):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        url = hf_hub_url(repo, name, repo_type="dataset", revision=rev or DATASET_REVISION)
        with urllib.request.urlopen(urllib.request.Request(url, headers=AUTH), timeout=300) as r_, open(path + ".part", "wb") as o_:
            shutil.copyfileobj(r_, o_, 1 << 20)
        if want.get("sha256") and L.sha256_file(path + ".part") != want["sha256"]:
            raise SystemExit(f"{name}: SHA-256 differs from the repository's LFS hash")
        os.replace(path + ".part", path)
    return path

card = open(cached_file("README.md")).read()
CONFIG_FILES = P.card_config_files(card, REVSI_CONFIG, REVSI_SPLIT, list(FILES))
META_FILES = ["metadata/sampled_video_frame_idx.json", "metadata/obj_visibility.json", "metadata/3d_annotation.json"]
missing_meta = [m for m in META_FILES if m not in FILES]
if not CONFIG_FILES or missing_meta:
    raise SystemExit(f"Release layout changed: config files {CONFIG_FILES}, missing metadata {missing_meta}")
FRAME_IDX = json.load(open(cached_file(META_FILES[0])))
VISIBILITY = json.load(open(cached_file(META_FILES[1])))
OBJECTS = {(a["dataset"], a["scene_id"]): {o["id"]: o["name"] for o in a["objects"]}
           for a in json.load(open(cached_file(META_FILES[2])))}
RAW = P.read_revsi_parquet([cached_file(f) for f in CONFIG_FILES])
FILE_HASHES = {f: L.sha256_file(cached_file(f)) for f in CONFIG_FILES + META_FILES}
problems = P.schema_problems(RAW, OFFICIAL, REVSI_CONFIG, FRAME_IDX, VISIBILITY)
if problems:
    raise SystemExit("Schema checks failed (nothing sampled):\n  " + "\n  ".join(problems))
QA_ALL = [P.normalize_row(r, REVSI_CONFIG, OFFICIAL, VISIBILITY) for r in sorted(RAW, key=lambda r: r["id"])]
assert all(P.prompt_independent_of_reference(q, OFFICIAL) for q in QA_ALL), "a prompt depends on the reference answer"
SAMPLE, QA, NEW = P.freeze_sample(OUTPUT_ROOT, QA_ALL, IDENTITY, FILE_HASHES)
QA_BY_UID = {q["qa_uid"]: q for q in QA}
CONTROL_QA = P.control_subset(QA, CONTROL_QA) if RUN_VISUAL_CONTROLS else []
if not os.path.exists(os.path.join(OUTPUT_ROOT, "control_subset.json")):
    L.write_json_atomic(os.path.join(OUTPUT_ROOT, "control_subset.json"), [q["qa_uid"] for q in CONTROL_QA])
assert json.load(open(os.path.join(OUTPUT_ROOT, "control_subset.json"))) == [q["qa_uid"] for q in CONTROL_QA], \
    "the predeclared control subset changed"
print(f"Config {REVSI_CONFIG}/{REVSI_SPLIT}: {len(RAW):,} questions on {SAMPLE['population_videos']} unique videos "
      f"(sources {sorted({r['dataset'] for r in RAW})}); schema, nesting and prompt checks passed.")
print(("New sample frozen." if NEW else "Saved sample verified (not redrawn).") +
      f" {SAMPLE['selected_videos']} videos = ceil({VIDEO_FRACTION} x {SAMPLE['population_videos']}); "
      f"{SAMPLE['selected_questions']} questions (all of theirs; not 10% of questions).")
print("Per task:", SAMPLE["per_task"]); print("Per source (videos):", SAMPLE["per_source_videos"])
print(f"Black-frame control: {len(CONTROL_QA)} predeclared questions (balanced by type, chosen from ids only).")

In [ ]:
# @title B2. Historical VSI-Bench cohort (separate, failure-selected) and the scene / QA overlap tables
HIST = P.import_historical_vsi(EXISTING_VSI_RUN)
VSI_CACHE = os.path.join(CACHE_ROOT, "datasets", f"VSI-Bench@{VSI_DATASET_REVISION}")
try:
    VSI_ROWS = [json.loads(l) for l in open(cached_file("test.jsonl", VSI_DATASET_REPO, VSI_DATASET_REVISION, VSI_CACHE,
                                                       {"test.jsonl": {}}))]
except Exception as exc:
    VSI_ROWS = []
    print(f"VSI-Bench annotations unavailable ({type(exc).__name__}); overlap is computed against the historical cohort only.")
OVERLAP = P.scene_overlap(RAW, SAMPLE["selected_keys"], VSI_ROWS, HIST)
P.write_csv(os.path.join(OUTPUT_ROOT, "scene_overlap.csv"), OVERLAP)
QA_MATCHES = P.qa_matches(QA, VSI_ROWS)
P.write_csv(os.path.join(OUTPUT_ROOT, "qa_matches.csv"), QA_MATCHES)
print("Historical cohort:", "found" if HIST["available"] else HIST.get("note"))
if HIST["available"]:
    print("  provenance:", HIST["provenance_checks"])
    print(f"  {len(HIST['selected_videos'])} videos, {len(HIST['candidate_videos'])} failure-selected candidate videos "
          "(a case study, not an unbiased sample; its predictions are never reused for ReVSI questions)")
both = sum(1 for o in OVERLAP if o["revsi_questions"] and o["in_vsibench"])
print(f"Scenes in ReVSI {REVSI_CONFIG}: {sum(1 for o in OVERLAP if o['revsi_questions'])}; also in VSI-Bench: {both}; "
      f"sample scenes also in VSI-Bench: {sum(1 for o in OVERLAP if o['in_revsi_sample'] and o['in_vsibench'])}; "
      f"sample scenes in the historical cohort: {sum(1 for o in OVERLAP if o['in_revsi_sample'] and o['in_historical_vsi_cohort'])}.")
print("QA matches (sample):", {s: sum(1 for m in QA_MATCHES if m["match_status"] == s) for s in {m["match_status"] for m in QA_MATCHES}},
      "- same scene does not mean same video encoding, question, object definition or answer.")

## C. Download selected media and construct exact inputs

In [ ]:
# @title C1. Read video.zip's directory (range reads), plan transfers, fetch, and verify every input against the official indices
import zipfile
ARCHIVE = "video.zip"
ARCHIVE_SIZE = FILES[ARCHIVE]["size"]
_lp = os.path.join(DATA_CACHE, "video_zip_listing.json")
_ZIP = {}

def open_archive():
    """ZipFile over HTTP range reads (only needed bytes); full download only if ranges are refused and allowed."""
    if "z" in _ZIP:
        return _ZIP["z"]
    url = hf_hub_url(DATASET_REPO, ARCHIVE, repo_type="dataset", revision=DATASET_REVISION)
    try:
        _ZIP["z"] = L.open_zip(L.HttpRangeFile(url, ARCHIVE_SIZE, AUTH))
    except L.RangeNotSupported:
        if ARCHIVE_SIZE / 1e9 > ARCHIVE_DOWNLOAD_MAX_GB:
            raise SystemExit(f"The server refused range reads; the whole {ARCHIVE} is {ARCHIVE_SIZE / 1e9:.1f} GB, above "
                             f"ARCHIVE_DOWNLOAD_MAX_GB={ARCHIVE_DOWNLOAD_MAX_GB}. Raise it to download once (cached on Drive), "
                             f"or place {ARCHIVE} at {DATA_CACHE}/{ARCHIVE}.")
        local = os.path.join(DATA_CACHE, ARCHIVE)
        if not (os.path.exists(local) and os.path.getsize(local) == ARCHIVE_SIZE):
            print(f"Downloading the whole {ARCHIVE} ({ARCHIVE_SIZE / 1e9:.1f} GB) once ...", flush=True)
            with urllib.request.urlopen(urllib.request.Request(url, headers=AUTH), timeout=600) as r_, open(local + ".part", "wb") as o_:
                shutil.copyfileobj(r_, o_, 1 << 22)
            os.replace(local + ".part", local)
        _ZIP["z"] = zipfile.ZipFile(local)
    return _ZIP["z"]

LISTING = json.load(open(_lp)) if os.path.exists(_lp) else None
if LISTING is None:
    LISTING = L.zip_listing(open_archive())
    L.write_json_atomic(_lp, LISTING)
MEDIA = P.media_manifest(SAMPLE["selected_keys"], LISTING, REVSI_CONFIG, VERIFY_FRAMES)
MEDIA_BY_VIDEO = {m["video_key"]: m for m in MEDIA}
in_b = sum(m["input_bytes"] or 0 for m in MEDIA)
ver_b = sum(m["verify_bytes"] or 0 for m in MEDIA)
print(f"Transfer: inputs {in_b / 1e6:.1f} MB + verification {ver_b / 1e6:.1f} MB by range reads "
      f"(10% of videos is not 10% of bytes; the whole archive is {ARCHIVE_SIZE / 1e9:.2f} GB and is not downloaded).")
VIDEO_CACHE = os.path.join(DATA_CACHE, "videos")
MISSING_LOG = os.path.join(OUTPUT_ROOT, "logs", "media_missing.jsonl")

def fetch_member(member, size, crc):
    path = L.safe_extract_path(VIDEO_CACHE, member)
    if os.path.exists(path) and os.path.getsize(path) == size and L.file_crc32(path) == crc:
        return path
    if member not in LISTING:
        raise KeyError(f"{member} is not in {ARCHIVE}")
    return L.extract_zip_member(open_archive(), member, VIDEO_CACHE, size, crc)

fetch_input = lambda v: fetch_member(MEDIA_BY_VIDEO[v]["input_member"], MEDIA_BY_VIDEO[v]["input_bytes"], MEDIA_BY_VIDEO[v]["input_crc32"])
fetch_source = lambda v: (fetch_member(MEDIA_BY_VIDEO[v]["verify_member"], MEDIA_BY_VIDEO[v]["verify_bytes"],
                                       MEDIA_BY_VIDEO[v]["verify_crc32"]) if MEDIA_BY_VIDEO[v].get("verify_member") else None)
FPS_FROM_CARD = {"3RScan": 4.0}                     # README: 3RScan 4 FPS, all other sources 10 FPS
fm_path = os.path.join(OUTPUT_ROOT, "frame_manifest.jsonl")
FRAMES = {r["video_key"]: r for r in L.read_jsonl(fm_path)[0]} if os.path.exists(fm_path) else {}
budget = REVSI_CONFIG.split("_")[0]
ver_used = 0
for m in MEDIA:
    v = m["video_key"]
    if v in FRAMES:
        continue
    idx = FRAME_IDX[m["scene_id"]][f"{budget}-frame"]
    row = {"video_key": v, "scene_id": m["scene_id"], "protocol": FRAME_PROTOCOL, "input_member": m["input_member"],
           "input_crc32": m["input_crc32"], "official_frame_indices": idx, "num_frames": NUM_FRAMES,
           "frame_manifest_hash": P.frame_manifest_hash(m, idx, NUM_FRAMES, FRAME_PROTOCOL)}
    if RUN_MODE == "review_only":
        row["verification"] = {"status": "not run (review_only)"}
    else:
        try:
            src = fetch_input(v)
            row["input_sha256"] = L.sha256_file(src)
            if VERIFY_FRAMES and ver_used + (m["verify_bytes"] or 0) <= VERIFY_MAX_GB * 1e9:
                ver_used += m["verify_bytes"] or 0
                row["verification"] = F.verify_input_frames(src, fetch_source(v), idx)
            else:
                row["verification"] = {"status": "skipped", "reason": "VERIFY_FRAMES off or VERIFY_MAX_GB reached"}
        except (KeyError, zipfile.BadZipFile) as exc:
            row["verification"] = {"status": "media_unavailable", "reason": f"{type(exc).__name__}: {exc}"}
    fps = (row["verification"] or {}).get("fps_all") or FPS_FROM_CARD.get(v.split("/")[0], 10.0)
    row["timestamps_s"] = (row["verification"] or {}).get("timestamps_s") or [round(i / fps, 3) for i in idx]
    row["timestamp_source"] = "all-frame video fps" if (row["verification"] or {}).get("fps_all") else "dataset card fps"
    FRAMES[v] = row
    L.write_jsonl_atomic(fm_path, list(FRAMES.values()))
BLOCKED = {r["video_key"]: r.get("reason") for r in L.read_jsonl(MISSING_LOG)[0]}
for v, f in FRAMES.items():
    st = (f.get("verification") or {}).get("status")
    if st in ("mismatch", "media_unavailable") and v not in BLOCKED:
        P_reason = f"input frames {st}: {(f['verification'] or {}).get('reason')}"
        L.Journal(MISSING_LOG).append({"video_key": v, "reason": P_reason, "logged": _t.strftime("%Y-%m-%dT%H:%M:%SZ")})
        BLOCKED[v] = P_reason
from collections import Counter
print("Frame verification:", dict(Counter((f.get("verification") or {}).get("status") for f in FRAMES.values())))
print(f"Blocked videos (kept in coverage, never replaced): {len(BLOCKED)}" + (f" {list(BLOCKED)[:5]}" if BLOCKED else ""))

## D. Run and resume Cambrian-P

In [ ]:
# @title D1. Runner, worker pre-flight (stand-in model) and the job list with cache keys
WORK_DIR = os.path.join(LOCAL_ROOT, "batches")
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(os.path.dirname(JOURNAL), exist_ok=True)
JOBS = P.build_jobs(QA, CONTROL_QA, {v: f["frame_manifest_hash"] for v, f in FRAMES.items()}, CONFIG_HASH, RUN_VISUAL_CONTROLS)
if RUN_MODE == "smoke":
    _sv = sorted({q["video_key"] for q in QA if q["video_key"] not in BLOCKED},
                 key=lambda v: MEDIA_BY_VIDEO[v]["input_bytes"] or 0)[:SMOKE_VIDEOS]
    _keep = {q["qa_uid"] for q in QA if q["video_key"] in _sv}
    _keep = set(sorted(_keep)[:SMOKE_QA])
    JOBS = [j for j in JOBS if j["qa_uid"] in _keep]
FRAME_INFO = {v: {"official_frame_indices": f["official_frame_indices"], "timestamps_s": f["timestamps_s"]} for v, f in FRAMES.items()}

def run_worker(batch_path, extra, status_name="worker_status.json"):
    cmd = [VENV_PY, "/content/revsi/revsi_worker.py", "--batch", batch_path, "--model_path", "ckpts/Cambrian-P-7B",
           "--num_frames", str(NUM_FRAMES), "--config_hash", CONFIG_HASH, "--identity", json.dumps(IDENTITY),
           "--journal", JOURNAL, "--errors", ERRORS_JOURNAL, "--shard_dir", SHARD_DIR,
           "--status", os.path.join(OUTPUT_ROOT, "logs", status_name), "--lib_dir", "/content/revsi",
           "--frames_dir", os.path.join(OUTPUT_ROOT, "input_frames"), "--max_retries", str(MAX_RETRIES)] + extra
    log_path = os.path.join(OUTPUT_ROOT, "logs", f"worker-{_t.strftime('%Y%m%d-%H%M%S')}-{_t.time_ns() % 10**6:06d}.log")
    with open(log_path, "w") as logf:
        p = subprocess.Popen(cmd, cwd=REPO, env=WORKER_ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            logf.write(line)
            if any(k in line for k in ("[cam-load]", "Error", "error", "Traceback", "protocol violation")):
                print("   " + line.rstrip()[:220])
        code_ = p.wait()
    sp = os.path.join(OUTPUT_ROOT, "logs", status_name)
    st = json.load(open(sp)) if os.path.exists(sp) else {}
    print(f"  worker exit {code_}: {st.get('phase')} | answers {st.get('answers')} | camera head loaded "
          f"{st.get('camera_head_loaded')} | peak VRAM {st.get('peak_vram_gb')} GB | log {log_path}")
    return code_

def context():
    return {"shard_dir": SHARD_DIR, "journal": JOURNAL, "config_hash": CONFIG_HASH, "work_dir": WORK_DIR,
            "batch_bytes": int(MEDIA_BATCH_GB * 1e9), "run_worker": run_worker, "fetch_video": fetch_input,
            "release": lambda v, p: None, "blocked": set(BLOCKED),
            "log_missing": lambda v, why: L.Journal(MISSING_LOG).append({"video_key": v, "reason": why})}

OK, FAILED, META, GATHER = P.gather(SHARD_DIR, JOURNAL, CONFIG_HASH, JOBS)
COVERAGE_BEFORE = P.coverage_counts(JOBS, OK, FAILED, BLOCKED)
print("Coverage before inference:", COVERAGE_BEFORE, "| incompatible cached rows:", GATHER["incompatible_cached_rows"])
if RUN_MODE != "review_only":
    _pv = next((fetch_input(v) for v in sorted(FRAMES) if v not in BLOCKED), None)
    _pf = subprocess.run([VENV_PY, "/content/revsi/revsi_worker_selftest.py", "--video", _pv, "--num_frames", str(NUM_FRAMES)],
                         capture_output=True, text=True, cwd="/content/revsi")
    print(_pf.stdout.strip().splitlines()[-1] if _pf.stdout.strip() else "worker pre-flight produced no output")
    if _pf.returncode:
        print(_pf.stdout[-3000:] + _pf.stderr[-1500:])
        raise SystemExit("Worker pre-flight failed: nothing was sent to the GPU.")

In [ ]:
# @title D2. Run only missing jobs (first one job as a timed probe with the real model), then report coverage
if RUN_MODE == "review_only":
    print("review_only: no installs, no model; saved predictions are reloaded in E.")
else:
    pending = [j for j in JOBS if j["job_id"] not in OK and j["video_key"] not in BLOCKED]
    print(f"Jobs: {len(JOBS)} scheduled; {len(pending)} still need answers.")
    if pending:
        if not gpu:
            raise SystemExit(f"{len(pending)} job(s) need the model, which needs a large GPU (A100-class). "
                             "Review, export and analysis run on CPU or T4.")
        ensure_model()
        probe = [pending[0]]
        t_probe = _t.time()
        s_probe = P.run_jobs(probe, QA_BY_UID, MEDIA_BY_VIDEO, FRAME_INFO, context())
        el = _t.time() - t_probe
        st = json.load(open(os.path.join(OUTPUT_ROOT, "logs", "worker_status.json")))
        pre = {"probe_job": probe[0]["job_id"], "seconds_including_model_load": round(el, 1),
               "peak_vram_gb": st.get("peak_vram_gb"), "gpu": gpu, "exit": s_probe.get("stopped")}
        L.write_json_atomic(os.path.join(OUTPUT_ROOT, "logs", "preflight.json"), pre)
        if s_probe.get("stopped"):
            raise SystemExit("The probe failed (see the worker log; saved in logs/preflight.json). Options: a larger GPU; "
                             "or a separately identified experiment (new RUN_ID) with another frame budget or quantization. "
                             "Frames, camera tokens and the checkpoint are never changed silently.")
        print(f"Probe: {el:.0f} s including model load; peak VRAM {pre['peak_vram_gb']} GB. Rough estimate for the rest: "
              f"{len(pending) - 1} jobs x 2-6 s = {(len(pending) - 1) * 2 / 60:.0f}-{(len(pending) - 1) * 6 / 60:.0f} min "
              "(uncertain: the probe includes loading the model once).")
        summary = P.run_jobs(JOBS, QA_BY_UID, MEDIA_BY_VIDEO, FRAME_INFO, context())
        if summary.get("stopped"):
            print(f"Stopped with worker exit {summary['stopped']}; rerun D2 to resume (only missing jobs run).")
OK, FAILED, META, GATHER = P.gather(SHARD_DIR, JOURNAL, CONFIG_HASH, JOBS)
COVERAGE_AFTER = P.coverage_counts(JOBS, OK, FAILED, BLOCKED)
L.write_json_atomic(os.path.join(OUTPUT_ROOT, "coverage.json"), {"before": COVERAGE_BEFORE, "after": COVERAGE_AFTER,
                    "gather": GATHER, "blocked_media": BLOCKED, "run_mode": RUN_MODE,
                    "inference_complete": (COVERAGE_AFTER.get("normal") or {}).get("pending") == 0
                    and (COVERAGE_AFTER.get("normal") or {}).get("failed") == 0})
print("Coverage after:", COVERAGE_AFTER)

## E. Score and compare

In [ ]:
# @title E1. Official subset scores (normal input), scene-cluster bootstrap, coverage; historical comparison (descriptive)
CTX = R.load_context(OUTPUT_ROOT, OFFICIAL, JOURNAL, fetch_input=fetch_input, fetch_source=fetch_source)
SCORES = CTX["scores"]
P.write_csv(os.path.join(OUTPUT_ROOT, "scores.csv"), SCORES)
L.write_jsonl_atomic(os.path.join(OUTPUT_ROOT, "scores.jsonl"), SCORES)
normal = [s for s in SCORES if s["condition"] == "normal"]
METRICS = {f"ReVSI {REVSI_CONFIG} / Cambrian-P-7B / normal input ({NUM_FRAMES} frames)": P.metrics_block(normal, OFFICIAL)}
BASELINE = P.frequency_baseline(QA, OFFICIAL)
METRICS["ReVSI scene-disjoint frequency baseline (local diagnostic)"] = P.metrics_block(BASELINE, OFFICIAL, n_boot=200)
HIST_SUMMARY = "No historical VSI-Bench run was found."
if HIST.get("available"):
    hs = [s for s in HIST["scores"] if s.get("official_score") is not None]
    by_task = {}
    for s in hs:
        t = P.VSI_FAMILY.get(s["question_type"])
        if t:
            by_task.setdefault(t, []).append(s["official_score"])
    HIST_SUMMARY = ("Historical VSI-Bench cohort (128 frames, VSI-Bench references, different questions and videos): "
                    "QA-weighted mean per common task: " + ", ".join(f"{t} {100 * sum(v) / len(v):.1f} (n={len(v)})"
                    for t, v in sorted(by_task.items())) + ". This is a descriptive, confounded comparison (frames 128 vs "
                    f"{NUM_FRAMES}, different references and samples); its 18-style candidate list is failure-selected.")
L.write_json_atomic(os.path.join(OUTPUT_ROOT, "metrics.json"), {"metrics": METRICS, "historical": HIST_SUMMARY,
                    "coverage": CTX["coverage"], "gather": CTX["gather_report"], "scorer": SCORER})
for name, m in METRICS.items():
    agg = m["official_subset_macro"] or {}
    ci = (m["bootstrap_scene_cluster"].get("overall_acc") or {}).get("ci95")
    print(f"{name}: overall {100 * (agg.get('overall_acc') or 0):.1f}" + (f" (95% CI {100 * ci[0]:.1f}-{100 * ci[1]:.1f}, "
          f"{m['scored_videos']} scenes)" if ci else "") + f"; scored {m['scored_questions']}; excluded {m['excluded']}")
print(HIST_SUMMARY)

## F. Test dependence on visual evidence

In [ ]:
# @title F1. Paired normal vs black-frame results (outside benchmark accuracy and verified-failure totals)
CONTROL_ROWS = P.control_results(SCORES, CONTROL_QA) if RUN_VISUAL_CONTROLS else []
P.write_csv(os.path.join(OUTPUT_ROOT, "control_results.csv"), CONTROL_ROWS or [{"status": "controls disabled"}])
paired = [r for r in CONTROL_ROWS if r["pair_available"]]
if paired:
    d_ = sum(r["score_change"] for r in paired) / len(paired)
    agree = sum(bool(r["same_parsed_answer"]) for r in paired) / len(paired)
    bl = (METRICS["ReVSI scene-disjoint frequency baseline (local diagnostic)"]["official_subset_macro"] or {}).get("overall_acc")
    CONTROLS_SUMMARY = (f"Black-frame control: {len(paired)} of {len(CONTROL_ROWS)} predeclared questions paired. Mean official "
                        f"score change (black - normal): {d_:+.3f}; same parsed answer in {agree:.0%} of pairs. Retaining "
                        "accuracy with black frames suggests a strong prior or shortcut; it does not prove memorization. "
                        f"Frequency baseline (scene-disjoint): official macro {100 * (bl or 0):.1f}.")
else:
    CONTROLS_SUMMARY = ("Black-frame control: no paired results yet (" + ("controls disabled" if not RUN_VISUAL_CONTROLS else
                        f"{sum(1 for r in CONTROL_ROWS if not r['pair_available'])} of {len(CONTROL_ROWS)} unpaired: "
                        "run D2 on a GPU") + "). No verdict is drawn.")
print(CONTROLS_SUMMARY)
print("Object-removal controls: skipped - the release gives per-budget object visibility, not per-frame target "
      "visibility, which is needed to identify every target-containing frame.")

## G/H. Review queues, candidate export, manual review, final exports
**Frame-evidence review is manual.** Exported frames do not establish that the answer is visible; a reviewer decides, through the same validator for the form and the CSV.

In [ ]:
# @title G1. Review queues (seeded audit + diagnostic candidates) and the external review template
assert 1 <= REVIEW_VIDEOS <= TARGET_REVIEW_VIDEOS, "REVIEW_VIDEOS must be between 1 and TARGET_REVIEW_VIDEOS"
QUEUE = R.build_queue(QA, SCORES, SEED, AUDIT_QA, TARGET_REVIEW_VIDEOS, REVIEW_VIDEOS, AUDIT_REQUIRED)
P.write_csv(os.path.join(OUTPUT_ROOT, "review_queue.csv"), QUEUE)
n_tpl = R.write_template(os.path.join(OUTPUT_ROOT, "review_template.csv"), QUEUE, CTX)
diag = [r for r in QUEUE if r["queue"] == "diagnostic"]
print(f"Audit queue: {sum(r['queue'] == 'audit' for r in QUEUE)} questions (seeded, independent of correctness). "
      f"Diagnostic: {len({r['video_key'] for r in diag})} candidate videos, {len(diag)} candidate questions "
      f"(exported: top {TARGET_REVIEW_VIDEOS} videos). review_template.csv: {n_tpl} rows.")
_req = [r for r in QUEUE if r["required"]]
print(f"Required reviews: {len(_req)} (the main question of the first {len([r for r in _req if r['queue'] == 'diagnostic'])} "
      f"candidate videos in queue order; REVIEW_VIDEOS = {REVIEW_VIDEOS}"
      + ("; plus the audit questions" if AUDIT_REQUIRED else "; audit questions optional") + "):")
for r in _req:
    print(f"  #{r['rank']:2d} {r['video_key']:45s} {r['qa_uid']}  ({r['automatic_status']})")
if not diag:
    print(R.empty_queue_reasons([s for s in SCORES if s["condition"] == "normal"], CTX["coverage"].get("normal"),
                                CTX["gather_report"]))

In [ ]:
# @title H1. Export up to 20 candidate videos BEFORE review (video, exact input frames, contact sheet, case.json)
_exp = sorted({r["video_key"] for r in QUEUE if r["queue"] == "diagnostic" and r["export"]})
_aud = sorted({r["video_key"] for r in QUEUE if r["queue"] == "audit"})
for v in _exp + [a for a in _aud if a not in _exp]:
    pk = sorted({r["qa_uid"] for r in QUEUE if r["video_key"] == v} | {q["qa_uid"] for q in QA if q["video_key"] == v})
    probs = R.export_candidate(v, CTX, pk)
    print(f"  {v}: {'ok' if not probs else probs}")
print(f"Exported {len(_exp)} diagnostic candidate videos and {len([a for a in _aud if a not in _exp])} audit videos to "
      "candidate_cases/ (all QA records stay in the tables).")

In [ ]:
# @title G2. Review form (saves to reviews.jsonl; answers can be hidden until you reveal them)
import ipywidgets as W
from IPython.display import display, clear_output
CTX = R.load_context(OUTPUT_ROOT, OFFICIAL, JOURNAL, fetch_input=fetch_input, fetch_source=fetch_source)
MATCH_BY_UID = {m["qa_uid"]: m for m in QA_MATCHES}
items = [r for r in QUEUE if r["role"] in ("primary", "audit", "supporting")]
items.sort(key=lambda r: (not r["required"], r["queue"] != "diagnostic", r["rank"]))   # required items first
pick = W.Dropdown(options=[(f"{'[required]' if r['required'] else '[optional]'} {r['queue']}#{r['rank']} {r['role']} "
                            f"{r['video_key']} {r['qa_uid']}", k) for k, r in enumerate(items)], layout=W.Layout(width="95%"))
body, status_line = W.Output(), W.HTML()
UI = {}

def refresh_status():
    _, latest = R.load_reviews(OUTPUT_ROOT)
    cls = R.classify_all(QUEUE, latest, CTX)
    c = R.counts(CTX, QUEUE, cls, CTX["coverage"].get("normal"), MIN_REVIEW_VIDEOS, TARGET_REVIEW_VIDEOS)
    status_line.value = "<b>" + L.esc(R.status_message(c, CTX["coverage"].get("normal"))) + "</b>"
    return latest, cls

def show(k):
    r = items[k]
    q, s = QA_BY_UID[r["qa_uid"]], CTX["scores_by_uid"].get(r["qa_uid"]) or {}
    cdir = os.path.join(OUTPUT_ROOT, "candidate_cases", R.video_dir_name(q["video_key"]))
    latest, cls = refresh_status()
    cur = latest.get(r["qa_uid"]) or {}
    names = OBJECTS.get((q["dataset"], q["scene_id"]), {})
    head = W.HTML(f"<h3>{L.esc(q['video_key'])} · {L.esc(q['qa_uid'])}</h3><p>{L.esc(q['question_type'])}: "
                  f"<b>{L.esc(q['question'])}</b></p><p>{'<br>'.join(L.esc(o) for o in q['options'] or [])}</p>"
                  f"<p>Release metadata: queried objects {L.esc([names.get(i, i) for i in q['queried_object_ids'] or []])}; "
                  f"visibility in {NUM_FRAMES} frames: {L.esc(q['visibility'])} (an aggregate flag, not proof). Frame "
                  f"verification: {L.esc((CTX['verification'].get(q['video_key']) or {}).get('status'))}. Cross-benchmark: "
                  f"{L.esc((MATCH_BY_UID.get(q['qa_uid']) or {}).get('match_status'))}</p>")
    reveal = W.ToggleButton(value=False, description="Reveal model + reference answers")
    ans = W.HTML()
    def _rv(ch):
        ans.value = (f"<p><b>Model:</b> {L.esc(s.get('raw_answer'))} → {L.esc(s.get('parsed_answer'))} · <b>Reference:</b> "
                     f"{L.esc(q['reference_answer'])} · official score {L.esc(s.get('official_score'))} "
                     f"({L.esc(s.get('automatic_status'))})</p>") if ch["new"] else ""
    reveal.observe(_rv, names="value")
    parts = [head, reveal, ans]
    if os.path.exists(os.path.join(cdir, "video.mp4")):
        parts.append(W.Video(value=open(os.path.join(cdir, "video.mp4"), "rb").read(), format="mp4", width=480))
    if os.path.exists(os.path.join(cdir, "contact_sheet.jpg")):
        parts.append(W.HTML("<b>Model inputs</b> (#position, source frame index, seconds):"))
        parts.append(W.Image(value=open(os.path.join(cdir, "contact_sheet.jpg"), "rb").read(), format="jpg", layout=W.Layout(width="100%")))
    xw, xb, xm, xbox = W.Text(description="Extra frames", placeholder="fps, or start,end,fps (seconds)"), W.Button(description="Extract"), W.HTML(), W.VBox()
    def _x(_):
        src = os.path.join(cdir, "source_video.mp4")
        if not os.path.exists(src):
            xm.value = "No full source video exported for this case."
            return
        try:
            vals = [float(x) for x in (xw.value or "1").split(",")]
            res = F.extra_frames(src, os.path.join(cdir, "extra_frames"), fps=vals[0] if len(vals) == 1 else 0,
                                 windows=[tuple(vals)] if len(vals) == 3 else ())
        except Exception as exc:
            xm.value = L.esc(f"{type(exc).__name__}: {exc}")
            return
        L.write_json_atomic(os.path.join(cdir, "extra_frames", "index.json"), res)
        CTX["extra_index"][q["video_key"]] = {e["frame_index"]: e for e in res["files"]}
        xbox.children = [W.Image(value=open(res["sheet"], "rb").read(), format="jpg", layout=W.Layout(width="100%"))]
        xm.value = f"{len(res['files'])} extra frames (NOT model inputs; cite as x&lt;index&gt;)."
    xb.on_click(_x)
    dd = lambda d_, opts, v: W.Dropdown(description=d_, options=opts, value=v if v in opts else opts[0], style={"description_width": "160px"})
    fa = dd("Reference", R.ANNOTATION, cur.get("annotation_status", "unreviewed"))
    fe = dd("Input evidence", R.EVIDENCE, cur.get("input_evidence_status", "unreviewed"))
    fv = dd("Model verdict", R.VERDICT, cur.get("model_verdict", "unreviewed"))
    fs = dd("Stage", R.STAGE, cur.get("review_stage", "draft"))
    fsrc = dd("Evidence source", R.EVIDENCE_SOURCE, cur.get("evidence_source", "model_inputs"))
    frev = W.Text(description="Reviewer", value=cur.get("reviewer", ""))
    fc = W.Text(description="Citations", placeholder="#12, #40 (inputs) · x340 (extra) · t=41.2", layout=W.Layout(width="95%"),
                value=" ".join(f"#{c['position']}" if c["frame_set"] == "input" else f"x{c['source_frame_index']}" for c in cur.get("citations") or []))
    fp = W.Text(description="Proposed answer", value=cur.get("proposed_answer", ""))
    fn = W.Textarea(description="Notes", value=cur.get("notes", ""), layout=W.Layout(width="95%", height="70px"))
    save, msg = W.Button(description="Save review", button_style="success"), W.HTML()
    def _save(_):
        clean, errs = R.validate_review_row({"qa_uid": r["qa_uid"], "annotation_status": fa.value, "input_evidence_status": fe.value,
                                             "model_verdict": fv.value, "review_stage": fs.value, "reviewer": frev.value,
                                             "notes": fn.value, "citations": fc.value, "evidence_source": fsrc.value,
                                             "proposed_answer": fp.value}, CTX, "notebook_form")
        if errs:
            msg.value = "<span style='color:#b00'>Not saved: " + L.esc("; ".join(errs)) + "</span>"
            return
        R.append_review(OUTPUT_ROOT, clean)
        _, cls2 = refresh_status()
        c2 = cls2.get(r["qa_uid"], {})
        msg.value = f"Saved ({clean['review_stage']}). Verified model failure: {c2.get('verified_model_failure')}; reasons: {L.esc(c2.get('reasons'))}"
    save.on_click(_save)
    UI.update(item=r, reveal=reveal, answers=ans, extra=xw, extract=xb, extra_msg=xm, annotation=fa, evidence=fe,
              verdict=fv, stage=fs, reviewer=frev, citations=fc, proposed=fp, notes=fn, save=save, msg=msg)
    parts += [W.HBox([xw, xb]), xm, xbox, fa, fe, fv, fsrc, fs, frev, fc, fp, fn, save, msg]
    with body:
        clear_output()
        display(W.VBox(parts))

pick.observe(lambda ch: show(ch["new"]) if ch["new"] is not None else None, names="value")
refresh_status()
if items:
    display(W.VBox([status_line, pick, body]))
    show(0)
else:
    print(R.status_message(R.counts(CTX, QUEUE, {}, CTX["coverage"].get("normal"), MIN_REVIEW_VIDEOS, TARGET_REVIEW_VIDEOS),
                           CTX["coverage"].get("normal"),
                           R.empty_queue_reasons(normal, CTX["coverage"].get("normal"), CTX["gather_report"])))

In [ ]:
# @title G3. Import an edited review CSV (download review_template.csv, edit, upload to review_import/ in the run folder)
IMPORT_DIR = os.path.join(OUTPUT_ROOT, "review_import")
os.makedirs(IMPORT_DIR, exist_ok=True)
CTX = R.load_context(OUTPUT_ROOT, OFFICIAL, JOURNAL, fetch_input=fetch_input, fetch_source=fetch_source)
files_ = sorted(f for f in os.listdir(IMPORT_DIR) if f.endswith(".csv"))
if not files_:
    print(f"No CSV in {IMPORT_DIR}. Viewing files in Drive does not save a review: edit review_template.csv and place it there.")
for f in files_:
    acc, rej = R.import_reviews(os.path.join(IMPORT_DIR, f), CTX)
    print(f"{f}: {len(acc)} accepted, {len(rej)} rejected (reasons in review_import_errors.csv)")
    for x in rej[:10]:
        print(f"  line {x['line']} {x['qa_uid']}: {x['reasons']}")
    os.replace(os.path.join(IMPORT_DIR, f), os.path.join(IMPORT_DIR, f + f".imported-{_t.strftime('%Y%m%d-%H%M%S')}"))

In [ ]:
# @title H2. Final exports, manifests, summary and report (all from saved files; rerun after reviews)
CTX = R.load_context(OUTPUT_ROOT, OFFICIAL, JOURNAL, fetch_input=fetch_input, fetch_source=fetch_source)
_, LATEST = R.load_reviews(OUTPUT_ROOT)
CLS = R.classify_all(QUEUE, LATEST, CTX)
FINAL = R.export_final(CTX, CLS, LATEST)
R.write_summary(os.path.join(OUTPUT_ROOT, "summary.csv"), QUEUE, CLS, LATEST, CTX)
COUNTS = R.counts(CTX, QUEUE, CLS, CTX["coverage"].get("normal"), MIN_REVIEW_VIDEOS, TARGET_REVIEW_VIDEOS)
STATUS = R.status_message(COUNTS, CTX["coverage"].get("normal"),
                          R.empty_queue_reasons(normal, CTX["coverage"].get("normal"), CTX["gather_report"]))
PUBLISHED = ["ReVSI's authors report object/geometry annotation problems in VSI-Bench and problems applying full-scene "
             "answers to sparse input frames (arXiv 2604.24300, sections 3-4).",
             "They report reduced gains for several spatially fine-tuned models under ReVSI and evidence-absent controls "
             "suggesting reliance on priors (sections 5-6); their Cambrian experiments concern Cambrian-S, not Cambrian-P."]
ov = f"Scene overlap: see scene_overlap.csv; QA matches: see qa_matches.csv ({sum(1 for m in QA_MATCHES if m['match_status'] != 'unmatched')} template matches with unverified object identity)."
R.report({"out_root": OUTPUT_ROOT, "config": REVSI_CONFIG, "fraction": VIDEO_FRACTION, "seed": SEED,
          "population_videos": SAMPLE["population_videos"]}, COUNTS, STATUS, METRICS, CONTROLS_SUMMARY,
         METRICS["ReVSI scene-disjoint frequency baseline (local diagnostic)"]["official_subset_macro"], HIST_SUMMARY, ov,
         CLS, LATEST, PUBLISHED)
cov = json.load(open(os.path.join(OUTPUT_ROOT, "coverage.json")))
cov.update(review=COUNTS)
L.write_json_atomic(os.path.join(OUTPUT_ROOT, "coverage.json"), cov)
print(STATUS)
for k, v in COUNTS.items():
    print(f"  {k}: {v}")
print(f"Wrote verified_failure_manifest.json ({FINAL['verified_failures']['distinct_videos']} videos), "
      f"benchmark_issue_manifest.json ({FINAL['benchmark_issues']['distinct_videos']}), summary.csv, report.md, report.html")
print(f"Drive folder: {OUTPUT_ROOT}")
print("Recovery after a disconnect: A1, A2, helper cells, A3, (A4 unless review_only), B1, B2, C1, D1, D2 (only missing "
      "jobs), E1, F1, G1, H1, G2/G3, H2. In review_only nothing is installed and no model is loaded.")

In [ ]:
# @title H3. (Optional) Zip the final deliverable (manifest-listed cases, tables and reports only)
MAKE_ZIP = False  # @param {type:"boolean"}
if MAKE_ZIP:
    zpath = os.path.join(os.path.dirname(OUTPUT_ROOT), RUN_ID + "_deliverable.zip")
    keep = ["report.html", "report.md", "summary.csv", "scores.csv", "coverage.json", "metrics.json", "control_results.csv",
            "scene_overlap.csv", "qa_matches.csv", "sample_manifest.json", "verified_failure_manifest.json",
            "benchmark_issue_manifest.json", "reviews.jsonl", "review_queue.csv", "run_config.json"]
    with zipfile.ZipFile(zpath + ".part", "w") as z:
        for k in keep:
            if os.path.exists(os.path.join(OUTPUT_ROOT, k)):
                z.write(os.path.join(OUTPUT_ROOT, k), k)
        for kind in ("verified_failures", "benchmark_issues"):
            for d_, _, fs in os.walk(os.path.join(OUTPUT_ROOT, kind)):
                for f in fs:
                    z.write(os.path.join(d_, f), os.path.relpath(os.path.join(d_, f), OUTPUT_ROOT))
    os.replace(zpath + ".part", zpath)
    print(f"Saved {zpath} ({os.path.getsize(zpath) / 1e6:.1f} MB). Unzip and open report.html in a browser.")
else:
    print("Set MAKE_ZIP = True for a zip of the reports, tables and manifest-listed cases.")

## V. Self-tests

In [ ]:
# @title V1. Acceptance self-tests (no GPU). A failure stops this cell with an error.
res = {}
r = subprocess.run([sys.executable, "/content/revsi/revsi_selftest.py", "--official_utils", os.path.join(SCORER_DIR, "utils.py"),
                    "--official_yaml", os.path.join(SCORER_DIR, "_default_template_yaml")], capture_output=True, text=True,
                   cwd="/content/revsi")
print(r.stdout[-6000:] + r.stderr[-2000:])
res["revsi_selftest.py"] = r.returncode
if RUN_MODE != "review_only":
    _pv = next((fetch_input(v) for v in sorted(FRAMES) if v not in BLOCKED), None)
    r = subprocess.run([VENV_PY, "/content/revsi/revsi_worker_selftest.py", "--video", _pv, "--num_frames", str(NUM_FRAMES)],
                       capture_output=True, text=True, cwd="/content/revsi")
    print(r.stdout[-3000:] + r.stderr[-1500:])
    res["revsi_worker_selftest.py"] = r.returncode
if any(res.values()):
    raise RuntimeError(f"Self-tests failed: {res}")
print("All self-tests passed.")